## 🔧 Correcciones aplicadas (2026-08-05)

### P1 — Escenarios B/C no registrados en logs
**Causa**: `EXECUTION_ID` era variable LOCAL dentro de `ejecutar_escenario_X()`. `nodo_metricas()` leía la global vieja de celda 10.

**Fix**: Añadido `global EXECUTION_ID, ESCENARIO` + re-set de ESCENARIO en:
- Celda 45 (`ejecutar_escenario_a`)
- Celda 47 (`ejecutar_escenario_b`)
- Celda 50 (`ejecutar_escenario_c`)

---

### P2 — 20 errores AEBE PDFs
**Causa**: `nodo_lectura` fallaba en PDFs con `leer_archivo()` → seteaba `error_lectura`. Aunque `nodo_transformacion` procesaba el PDF con pdfplumber, el error persistía en state y `nodo_metricas` marcaba status=ERROR.

**Fix (Celda 39)**:
1. Ampliada condición AEBE: detecta también PDFs por extensión + ruta `/aebe/` (para Escenario B donde se renombra a `BANANOTAS_REPORTE_*`)
2. Return incluye `"error_lectura": None` para limpiar el error cuando transformación fue exitosa
3. Return incluye `"total_filas": total_filas_crudo` (faltaba → M6 usaba 0)

---

### P3 — M3 recuperación en 0.93 en vez de 2
**Causa**: M3 heredaba `recuperacion_extraccion` del LOG_EXTRACCION (valores 0-1) incluso cuando el ETL no tenía errores. Además, `error_lectura` de PDFs activaba `tiene_errores=True` → M3=0.

**Fix (Celda 43 — nodo_metricas)**:
```
tiene_errores = error_mapeo OR error_transform OR error_carga
# error_lectura solo cuenta si TAMBIÉN error_transform
M3 = 0 si tiene_errores
M3 = 1 si reintentos > 0 (pero sin errores)
M3 = 2 si todo limpio
```

#  ETL Inteligente con LlamaIndex — Sector Bananero Ecuador
### Databricks · Llama 4 Maverick · LlamaIndex Workflows

---

**Trabajo de Titulación — Universidad Técnica de Machala**

Este notebook replica el pipeline ETL del experimento LangGraph usando el framework de agentes **LlamaIndex Workflows**.  
El LLM orquestador es **Llama 4 Maverick** (Databricks Foundation Model).  
Se mantiene el mismo marco de métricas M1–M5 y la estructura de tablas Delta Lake.

| Fase | Tabla de métricas | Qué mide |
|------|-------------------|---------|
| 📥 Extracción | `metricas_extraccion_li` | Tiempo, archivos nuevos, errores de descarga |
| 🔧 Transformación | `metricas_transformacion_li` | Calidad, duplicados, casteos, DPMO parcial |
| 💾 Carga | `metricas_carga_li` | Registros insertados, errores, tiempo de escritura |
| 📊 General ETL | `control_logs_etl` | M1–M5 completas del proceso end-to-end |

### Diferencias clave vs LangGraph
| Aspecto | LangGraph | LlamaIndex Workflows |
|---------|-----------|---------------------|
| Primitiva de orquestación | `StateGraph` con nodos y aristas | `Workflow` con `Events` y `@step` |
| Estado compartido | `TypedDict` mutable | Paso de eventos inmutables |
| Enrutamiento condicional | `add_conditional_edges` | `send_event` / retorno de `Event` tipado |
| LLM | `ChatDatabricks` (LangChain) | `DatabricksLLM` (LlamaIndex) |
| Memoria de agente | `StateGraph` state | `Context` del Workflow |

### Orden de ejecución
1. **Bloque 1** — Instalar librerías (reiniciar kernel después)
2. **Bloque 2** — Imports y configuración global
3. **Bloque 3** — Extracción con LlamaIndex Workflow + métricas
4. **Bloque 4** — Diccionario de conocimiento
5. **Bloque 5** — Funciones utilitarias compartidas
6. **Bloque 6** — Workflow ETL principal (mapeo, transformación, carga, métricas)
7. **Bloque 7** — Orquestador principal
8. **Bloque 8** — Métricas de transformación
9. **Bloque 9** — Métricas de carga
10. **Bloque 10** — Métricas generales M1–M5
11. **Bloque 11** — Utilidades (reset, consultas)


## 🗑️ Bloque 0 — Reset completo (opcional)

> ⚠️ **CUIDADO — estas celdas son destructivas.**  
> Úsalas solo si quieres empezar el experimento desde cero.

### Bloque 0A — Borrar tablas Delta del catálogo


## 📦 Bloque 1 — Instalación de librerías

⚠️ **Ejecuta SOLO esta celda primero.** Espera el reinicio del kernel antes de continuar.


In [ ]:
# PASO 1: Ejecuta las celdas 3 y 4 manualmente para borrar BD + volúmenes
print("✅ Ejecuta celdas 3 y 4 primero")

In [ ]:
%pip install llama-index llama-index-core langchain-databricks \
             google-api-python-client google-auth-httplib2 google-auth-oauthlib \
             openpyxl xlrd beautifulsoup4 numpy faostat scikit-learn \
             pdfplumber
dbutils.library.restartPython()

## ⚙️ Bloque 2 — Imports y Configuración Global

✏️ **Edita las credenciales** en esta celda antes de continuar.

**LlamaIndex vs LangGraph:** en lugar de `StateGraph` se usa `Workflow` con eventos tipados y decorador `@step`.


In [ ]:
import io, os, re, json, time, zipfile, hashlib, unicodedata
import numpy as np
import pandas as pd
import requests
from datetime import datetime
from typing import TypedDict, Optional, List
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# LangGraph
# LlamaIndex core
from llama_index.core.workflow import (
    Workflow, StartEvent, StopEvent, step, Context, Event,
)
from llama_index.core.llms import ChatMessage, MessageRole
from langchain_databricks import ChatDatabricks
from langchain_core.messages import HumanMessage, SystemMessage

# Google Drive
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaInMemoryUpload

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import lower, trim, col, when
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, IntegerType, DoubleType, FloatType
)

# ── CREDENCIALES — edita estos valores ────────────────────────────────────
# Ya no se necesita GEMINI_API_KEY — usaremos Databricks Foundation Models

SERVICE_ACCOUNT_INFO = {
  "type": "service_account",
  "project_id": "secret",
  "private_key_id": "secret",
  "private_key": "secret",
  "client_email": "secret",
  "client_id": "secret",
  "auth_uri": "secret",
  "token_uri": "secret",
  "auth_provider_x509_cert_url": "secret",
  "client_x509_cert_url": "secret",
  "universe_domain": "googleapis.com"
}

FOLDER_OUTPUT_ID = "secret" # output_tableau

# ── Rutas de volúmenes Databricks ─────────────────────────────────────────
RAW_PATH_ESPAC   = "/Volumes/databricksbanano/default/bronce/espac"
RAW_PATH_SIPA    = "/Volumes/databricksbanano/default/bronce/sipa"
RAW_PATH_FAOSTAT = "/Volumes/databricksbanano/default/bronce/faostat"
RAW_PATH_AEBE    = "/Volumes/databricksbanano/default/bronce/aebe"

# ── CONFIGURACIÓN DE TABLAS (TODO CENTRALIZADO EN BD_BANANO_EC) ─────────────
DB_NAME        = "bd_banano_ec"
FRAMEWORK_NAME = "LlamaIndex"  # Identificador del framework

# 🆔 CORRECCIÓN #21: ID único de ejecución (para historial de métricas)
from datetime import datetime
EXECUTION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")  # Ej: 20260619_041556

# ── CONFIGURACIÓN DE ESCENARIO EXPERIMENTAL ──────────────────────────────
ESCENARIO = "A"  # "A" (normal), "B" (variaciones schema), "C" (fallos controlados)

LOG_TABLE      = f"{DB_NAME}.control_logs_etl"  # Tabla unificada LangGraph + LlamaIndex
CONTROL_TABLE  = f"{DB_NAME}.control_descargas_fuentes"  # Compartida con LangGraph
LOG_EXTRACCION = f"{DB_NAME}.metricas_extraccion"
LOG_TRANSFORM  = f"{DB_NAME}.metricas_transformacion"
LOG_CARGA      = f"{DB_NAME}.metricas_carga"

HEADERS_HTTP = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Referer": "https://www.banana-traders.com/"
}

# ── Credenciales FAOSTAT (auto-renovación de token) ───────────────────────
FAOSTAT_USERNAME = "secret@gmail.com"
FAOSTAT_PASSWORD = "secret" # ⚠️ CAMBIAR POR TU PASSWORD REAL
FAOSTAT_TOKEN = None  # Se obtiene automáticamente
FAOSTAT_TOKEN_EXPIRY = None  # Timestamp de expiración

def get_faostat_token():
    """
    Obtiene o renueva el token de FAOSTAT automáticamente.
    El token dura 60 minutos - esta función lo renueva cuando expira.
    """
    global FAOSTAT_TOKEN, FAOSTAT_TOKEN_EXPIRY
    
    # Verificar si el token sigue válido (con margen de 5 min)
    if FAOSTAT_TOKEN and FAOSTAT_TOKEN_EXPIRY:
        if time.time() < (FAOSTAT_TOKEN_EXPIRY - 300):  # 5 min antes
            return FAOSTAT_TOKEN
    
    # Obtener nuevo token
    print("  🔄 Renovando token FAOSTAT...")
    try:
        resp = requests.post(
            "https://faostatservices.fao.org/api/v1/auth/login",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={"username": FAOSTAT_USERNAME, "password": FAOSTAT_PASSWORD},
            timeout=30
        )
        resp.raise_for_status()
        token_data = resp.json()
        
        # FAOSTAT usa AWS Cognito - extraer de AuthenticationResult
        if 'AuthenticationResult' in token_data:
            auth_result = token_data['AuthenticationResult']
            # Preferir IdToken, luego AccessToken
            FAOSTAT_TOKEN = auth_result.get('IdToken') or auth_result.get('AccessToken')
        else:
            # Fallback por si cambia el formato
            FAOSTAT_TOKEN = token_data.get("access_token") or token_data.get("token")
        
        FAOSTAT_TOKEN_EXPIRY = time.time() + 3600  # 60 min
        print("  ✅ Token renovado (válido por 60 min)")
        return FAOSTAT_TOKEN
    except Exception as e:
        print(f"  ❌ Error renovando token: {e}")
        raise

# ── Inicializar servicios ──────────────────────────────────────────────────
creds = service_account.Credentials.from_service_account_info(
    SERVICE_ACCOUNT_INFO,
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive_service = build("drive", "v3", credentials=creds)

# Usar modelo de Databricks Foundation Models (sin límites de API gratuita)
llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",  # Llama 4 Maverick - 400B MoE, 128K context
    temperature=0.0,
    max_tokens=2000,
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
print("✅ Configuración cargada correctamente.")
print(f"   Base de datos      : {DB_NAME}")
print(f"   Log general ETL    : {LOG_TABLE}")
print(f"   Log extracción     : {LOG_EXTRACCION}")
print(f"   Log transformación : {LOG_TRANSFORM}")
print(f"   Log carga          : {LOG_CARGA}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FUNCIÓN HELPER: GUARDAR ARCHIVO Y REGISTRAR EN TABLA DE CONTROL
# ══════════════════════════════════════════════════════════════════════════

def _guardar_y_registrar(fuente_nombre: str, nombre_archivo: str, url: str,
                         contenido: bytes, volumen_destino: str, anio: int = 0) -> str:
    """
    Guarda el archivo descargado en el volumen y registra en la tabla de control.
    
    Returns:
        "nuevo" si el archivo fue guardado (nuevo hash)
        "omitido" si el archivo ya existía (mismo hash)
    """
    try:
        # Calcular hash MD5 del contenido
        hash_md5 = hashlib.md5(contenido).hexdigest()
        
        # Verificar si ya existe el mismo archivo (mismo hash)
        if spark.catalog.tableExists(CONTROL_TABLE):
            df_existente = spark.table(CONTROL_TABLE).filter(
                (F.col("fuente") == fuente_nombre) &
                (F.col("nombre_archivo") == nombre_archivo) &
                (F.col("hash_md5") == hash_md5)
            )
            if df_existente.count() > 0:
                return "omitido"
        
        # Guardar archivo en volumen (sin prefijo /dbfs en Databricks modernos)
        ruta_destino = f"{volumen_destino}/{nombre_archivo}"
        with open(ruta_destino, "wb") as f:
            f.write(contenido)
        
        # Registrar en tabla de control
        schema_control = StructType([
            StructField("fuente", StringType(), True),
            StructField("framework", StringType(), True),
            StructField("anio", IntegerType(), True),
            StructField("nombre_archivo", StringType(), True),
            StructField("url_archivo", StringType(), True),
            StructField("hash_md5", StringType(), True),
            StructField("fecha_descarga", TimestampType(), True),
        ])
        
        df_registro = spark.createDataFrame([(
            fuente_nombre,
            FRAMEWORK_NAME,
            anio if anio > 0 else None,
            nombre_archivo,
            url[:500],  # Truncar URL si es muy larga
            hash_md5,
            datetime.now()
        )], schema_control)
        
        df_registro.write.format("delta").mode("append").saveAsTable(CONTROL_TABLE)
        
        return "nuevo"
        
    except Exception as e:
        print(f"    ⚠️ Error guardando {nombre_archivo}: {e}")
        raise

print("✅ Función _guardar_y_registrar() definida.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FUNCIONES AUXILIARES PARA ESCENARIOS EXPERIMENTALES
# ══════════════════════════════════════════════════════════════════════════

def aplicar_variacion_esquema(archivo_local, nombre_archivo, fuente):
    """
    ESCENARIO B: Aplica transformaciones que rompen el patrón esperado.
    Retorna la ruta del archivo modificado (o la original si no aplica).
    """
    if ESCENARIO != "B":
        return archivo_local

    import shutil
    print(f"  🔬 [ESCENARIO B] Aplicando variación de esquema a {fuente}...")

    ruta_modificada = archivo_local.replace(".xls", "_VAR.xls").replace(".csv", "_VAR.csv")

    if fuente == "ESPAC":
        try:
            df = pd.read_excel(archivo_local)
            if 'PROVINCIA' in df.columns:
                df = df.rename(columns={'PROVINCIA': 'PROVINCIA_NOMBRE_EC'})
                df.to_excel(ruta_modificada, index=False)
                print(f"    ✅ ESPAC: Columna PROVINCIA renombrada")
                return ruta_modificada
        except Exception as e:
            print(f"    ⚠️ ESPAC variación falló ({e}), usando archivo original")

    elif fuente == "SIPA":
        try:
            df = pd.read_excel(archivo_local)
            df_extra = pd.concat([pd.DataFrame([[''] * len(df.columns)], columns=df.columns), df], ignore_index=True)
            df_extra.to_excel(ruta_modificada, index=False)
            print(f"    ✅ SIPA: Fila vacía insertada antes del header")
            return ruta_modificada
        except Exception as e:
            print(f"    ⚠️ SIPA variación falló ({e}), usando archivo original")

    elif fuente == "FAOSTAT":
        try:
            with open(archivo_local, 'r', encoding='utf-8') as f:
                datos = json.load(f)
            if isinstance(datos, list):
                for item in datos:
                    item['metadata_experimental'] = 'col_extra_no_esperada'
            ruta_modificada = archivo_local.replace(".json", "_VAR.json")
            with open(ruta_modificada, 'w', encoding='utf-8') as f:
                json.dump(datos, f, ensure_ascii=False, indent=2)
            print(f"    ✅ FAOSTAT: Columna metadata_experimental agregada")
            return ruta_modificada
        except Exception as e:
            print(f"    ⚠️ FAOSTAT variación falló ({e}), usando archivo original")

    elif fuente == "AEBE":
        ruta_modificada = archivo_local.replace("AEBE_BANANOTAS_", "BANANOTAS_REPORTE_")
        try:
            shutil.copy2(archivo_local, ruta_modificada)
            print(f"    ✅ AEBE: PDF renombrado rompiendo patrón")
            return ruta_modificada
        except Exception as e:
            print(f"    ⚠️ AEBE variación falló ({e}), usando archivo original")

    return archivo_local


def aplicar_fallo_controlado(contenido_bytes, nombre_archivo, fuente, intento=1):
    """
    ESCENARIO C: Simula fallos controlados en descarga/lectura.
    Retorna (contenido_modificado, fallo_aplicado: bool)
    """
    if ESCENARIO != "C":
        return contenido_bytes, False

    import random
    print(f"  🔬 [ESCENARIO C] Evaluando fallo controlado en {fuente} (intento {intento})...")

    if intento == 1 and random.random() < 0.5:
        print(f"    💥 Truncando archivo a la mitad de bytes")
        return contenido_bytes[:len(contenido_bytes) // 2], True

    print(f"    ✅ Intento {intento}: sin fallo (recuperación)")
    return contenido_bytes, False


print("✅ Funciones auxiliares para Escenarios B y C definidas")

## 📥 Bloque 3 — Extracción de Fuentes con LlamaIndex Workflow

Replica el Agente de Extracción de LangGraph usando `Workflow` + eventos tipados.

| # | Fuente | Método |
|---|--------|--------|
| 1 | ESPAC (INEC) | URLs hardcodeadas + HEAD request |
| 2 | SIPA (MAG) | Descarga directa |
| 3 | FAOSTAT | API REST con token |
| 4 | Banana-Traders | Scraping XLSX |

### Diferencia arquitectónica LangGraph → LlamaIndex

| LangGraph | LlamaIndex Workflows |
|-----------|---------------------|
| Nodos conectados con `add_conditional_edges` | `@step` que devuelve `Event` tipado |
| Estado `TypedDict` mutable | Eventos inmutables pasados entre pasos |
| `graph.invoke(estado)` | `await workflow.run(StartEvent(...))` |


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLA DE CONTROL DE DESCARGAS
# ══════════════════════════════════════════════════════════════════════════
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    fuente          STRING,
    framework       STRING,
    anio            INT,
    nombre_archivo  STRING,
    url_archivo     STRING,
    hash_md5        STRING,
    fecha_descarga  TIMESTAMP
) USING DELTA
""")

# ── Tabla de métricas de extracción ───────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_EXTRACCION} (
    fuente                  STRING,
    archivos_nuevos         INT,
    archivos_omitidos       INT,
    archivos_error          INT,
    kb_descargados          DOUBLE,
    tiempo_segundos         DOUBLE,
    timestamp_inicio        STRING,
    timestamp_fin           STRING,
    execution_id            STRING,
    framework               STRING,
    llamadas_llm            INT,
    razonamiento_llm        STRING,
    status                  STRING,
    reintentos_realizados   INT,
    intervencion_manual     INT,
    recuperacion            INT,
    escenario               STRING
) USING DELTA
""")


def _hash_md5(contenido: bytes) -> str:
    return hashlib.md5(contenido).hexdigest()

def _historial() -> dict:
    """Devuelve dict {(nombre_archivo, framework): hash_md5} de todo lo ya descargado.
    
    CORRECCIÓN #20: Incluye framework para permitir múltiples pruebas con diferentes agentes.
    """
    try:
        rows = spark.table(CONTROL_TABLE).select("nombre_archivo","framework","hash_md5").collect()
        return {(r["nombre_archivo"], r.get("framework", "LangGraph")): r["hash_md5"] for r in rows}
    except:
        return {}

def _guardar_y_registrar(fuente, nombre, url, contenido, ruta_vol, anio=0):
    """
    Guarda el archivo si su hash es nuevo PARA ESTE FRAMEWORK.
    
    CORRECCIÓN #20: Validación por framework permite:
    - Omitir si MISMO framework Y MISMO hash (evita duplicados)
    - Permitir si DIFERENTE framework (nueva prueba experimental)
    
    Retorna: 'nuevo', 'omitido', o 'error'
    """
    historial   = _historial()
    nuevo_hash  = _hash_md5(contenido)
    
    # ⭐ CORRECCIÓN #20: Verificar por framework
    clave_actual = (nombre, FRAMEWORK_NAME)
    
    if historial.get(clave_actual) == nuevo_hash:
        print(f"  ⏭ Sin cambios ({FRAMEWORK_NAME}) → {nombre}")
        return "omitido"
    
    # Verificar si existe con otro framework (permitir, es nueva prueba)
    otros_frameworks = [fw for (n, fw) in historial.keys() if n == nombre and fw != FRAMEWORK_NAME]
    if otros_frameworks:
        print(f"  🆕 Ya existe en {otros_frameworks[0]}, pero permitido para {FRAMEWORK_NAME}")
    
    ruta_dest = f"{ruta_vol}/{nombre}"
    with open(ruta_dest, "wb") as f:
        f.write(contenido)
    
    schema = StructType([
        StructField("fuente",         StringType(),  True),
        StructField("framework",      StringType(),  True),  # ⭐ CORRECCIÓN #20
        StructField("anio",           IntegerType(), True),
        StructField("nombre_archivo", StringType(),  True),
        StructField("url_archivo",    StringType(),  True),
        StructField("hash_md5",       StringType(),  True),
        StructField("fecha_descarga", TimestampType(),True),
    ])
    spark.createDataFrame(
        [(fuente, FRAMEWORK_NAME, int(anio), nombre, url, nuevo_hash, datetime.now())], schema  # ⭐ CORRECCIÓN #20
    ).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(CONTROL_TABLE)
    
    print(f"  ✅ Nuevo ({FRAMEWORK_NAME}) → {nombre}  ({len(contenido)//1024} KB)")
    return "nuevo"

def _registrar_metrica_extraccion(fuente, nuevos, omitidos, errores, kb, t_inicio, t_fin):
    schema = StructType([
        StructField("fuente",            StringType(),  True),
        StructField("archivos_nuevos",   IntegerType(), True),
        StructField("archivos_omitidos", IntegerType(), True),
        StructField("archivos_error",    IntegerType(), True),
        StructField("kb_descargados",    DoubleType(),  True),
        StructField("tiempo_segundos",   DoubleType(),  True),
        StructField("timestamp_inicio",  StringType(),True),  # ✅ CORRECCIÓN: STRING no TIMESTAMP
        StructField("timestamp_fin",     StringType(),True),  # ✅ CORRECCIÓN: STRING no TIMESTAMP
    ])
    spark.createDataFrame(
        [(fuente, nuevos, omitidos, errores, round(kb,2),
          round((t_fin-t_inicio).total_seconds(),2), 
          t_inicio.isoformat(), t_fin.isoformat())], schema  # ✅ Convertir a ISO string
    ).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_EXTRACCION)

print("✅ Infraestructura de control y métricas lista.")

In [ ]:
\
# ══════════════════════════════════════════════════════════════════════════
# EVENTOS DEL AGENTE 1: EXTRACCIÓN
# En LlamaIndex Workflows el estado viaja como eventos tipados (no TypedDict mutable)
# Equivalencia: cada Event = un "estado parcial" del nodo LangGraph correspondiente
# ==============================================================================

class InicioExtraccionEvent(Event):
    fuente_nombre:       str
    fuente_url:          str
    volumen_destino:     str
    keywords_relevantes: List[str]
    timestamp_inicio:    str

class ArchivosListadosEvent(Event):
    fuente_nombre:       str
    fuente_url:          str
    volumen_destino:     str
    keywords_relevantes: List[str]
    archivos:            List[dict]
    timestamp_inicio:    str

class ArchivosSeleccionadosEvent(Event):
    fuente_nombre:       str
    volumen_destino:     str
    archivos_sel:        List[dict]
    razonamiento_llm:    str
    llamadas_api:        int
    timestamp_inicio:    str

class DescargaCompletaEvent(Event):
    fuente_nombre:       str
    archivos_nuevos:     int
    archivos_omitidos:   int
    archivos_error:      int
    kb_descargados:      float
    llamadas_api:        int
    timestamp_inicio:    str
    reintentos_realizados: int = 0
    recuperacion:           int = 2

class ExtraccionErrorEvent(Event):
    fuente_nombre:    str
    causa:            str
    llamadas_api:     int
    timestamp_inicio: str

print("OK Eventos AGENTE 1 definidos.")


In [ ]:
# ==============================================================================
# ORQUESTADOR AGENTE 1 - ejecutar_agente1_extraccion()
# Equivalente a ejecutar_extraccion_todas_fuentes() de LangGraph
# En lugar de graph.invoke() usa await workflow.run()
# ==============================================================================

import asyncio

async def ejecutar_agente1_extraccion():
    """Ejecuta ExtractionWorkflow para las fuentes en secuencia."""
    fuentes = [
        {
            "fuente_nombre":       "ESPAC",
            "fuente_url":          "https://www.ecuadorencifras.gob.ec/encuesta-de-superficie-y-produccion-agropecuaria-continua-espac/",
            "volumen_destino":     RAW_PATH_ESPAC,
            "keywords_relevantes": ["banano","platano","musaceas","produccion","superficie","tabulados","series"],
        },
        {
            "fuente_nombre":       "SIPA",
            "fuente_url":          "https://sipa.agricultura.gob.ec/estadisticas-agropecuarias",
            "volumen_destino":     RAW_PATH_SIPA,
            "keywords_relevantes": ["temperatura","precipitacion","uso suelo","climatico"],
        },
        {
            "fuente_nombre":       "FAOSTAT",
            "fuente_url":          "https://faostatservices.fao.org/api/v1/en/data/QCL",
            "volumen_destino":     RAW_PATH_FAOSTAT,
            "keywords_relevantes": ["bananas","plantains","production","area harvested","yield","Ecuador"],
        },
        {
            "fuente_nombre":       "AEBE_BANANOTAS",
            "fuente_url":          "https://www.aebe.com.ec/bananotas",
            "volumen_destino":     RAW_PATH_AEBE,
            "keywords_relevantes": ["exportaciones","regiones","estadisticas","cajas","participacion"],
        },
    ]

    resultados = []
    ts_global = datetime.now()

    for fuente in fuentes:
        print(f"\n{'#'*70}")
        print(f"# AGENTE 1 - EXTRACCION: {fuente['fuente_nombre']}")
        print(f"{'#'*70}")
        try:
            wf = ExtractionWorkflow(timeout=600, verbose=False)
            resultado = await wf.run(**fuente)
            resultados.append(resultado)
            status = resultado.get('status','?') if resultado else 'ERROR'
            nuevos  = resultado.get('archivos_nuevos', 0) if resultado else 0
            omit    = resultado.get('archivos_omitidos', 0) if resultado else 0
            kb      = resultado.get('kb_descargados', 0) if resultado else 0
            print(f"  => Status: {status} | Nuevos: {nuevos} | Omitidos: {omit} | KB: {kb:.0f}")
        except Exception as e:
            print(f"  ERROR en {fuente['fuente_nombre']}: {e}")
            resultados.append({"status":"ERROR","fuente_nombre":fuente['fuente_nombre'],"error":str(e)})

    tiempo_total = (datetime.now() - ts_global).total_seconds()
    ok  = sum(1 for r in resultados if r and r.get('status') == 'OK')
    err = sum(1 for r in resultados if r and r.get('status') == 'ERROR')
    nuevos_tot = sum(r.get('archivos_nuevos', 0) for r in resultados if r)
    kb_tot     = sum(r.get('kb_descargados', 0)  for r in resultados if r)
    print(f"\n{'='*70}")
    print(f"EXTRACCION COMPLETADA en {tiempo_total:.1f}s")
    print(f"  Fuentes OK: {ok}/{len(fuentes)} | Errores: {err} | Archivos nuevos: {nuevos_tot} | Total: {kb_tot:.0f} KB")
    print(f"{'='*70}")
    return resultados

# Ejecutar
import nest_asyncio
nest_asyncio.apply()
resultados_extraccion = asyncio.run(ejecutar_agente1_extraccion())


In [ ]:
# ── Métricas de Extracción ─────────────────────────────────────────────────
print("\n📊 MÉTRICAS DE EXTRACCIÓN — por fuente:")
spark.table(LOG_EXTRACCION).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN TOTAL DE EXTRACCIÓN:")
spark.sql(f"""
SELECT
    SUM(archivos_nuevos)    AS total_archivos_nuevos,
    SUM(archivos_omitidos)  AS total_omitidos,
    SUM(archivos_error)     AS total_errores,
    ROUND(SUM(kb_descargados)/1024, 2) AS mb_descargados,
    ROUND(SUM(tiempo_segundos), 1)     AS tiempo_total_s
FROM {LOG_EXTRACCION}
""").display()

## 📐 Bloque 4 — Dimensiones

Crea las tablas dimensionales compartidas: `dim_productos`, `dim_provincias`, `dim_tiempo`.
Identico al LangGraph — estas tablas son independientes del framework de orquestación.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLAS DIMENSIONALES: REGIONES Y PROVINCIAS DE ECUADOR
# Esta estructura maestra normaliza los datos geográficos
# ══════════════════════════════════════════════════════════════════════════

print("📍 Creando/verificando tablas dimensionales...\n")

# ──────────────────────────────────────────────────────────────────────────
# 1️⃣ TABLA DIMENSIONAL: REGIONES
# ──────────────────────────────────────────────────────────────────────────
tabla_dim_regiones = f"{DB_NAME}.dim_regiones"

if spark.catalog.tableExists(tabla_dim_regiones):
    print(f"🌍 1. Tabla {tabla_dim_regiones} ya existe")
    count_regiones = spark.table(tabla_dim_regiones).count()
    print(f"   ℹ️  {count_regiones} regiones registradas (sin cambios)\n")
else:
    print("🌍 1. Creando dim_regiones...")
    
    regiones_data = [
        (1, "Sierra", "Región Interandina"),
        (2, "Costa", "Región Litoral"),
        (3, "Amazonía", "Región Amazónica"),
        (4, "Insular", "Región Insular - Galápagos"),
    ]
    
    schema_regiones = StructType([
        StructField("region_id", IntegerType(), False),
        StructField("region", StringType(), False),
        StructField("descripcion", StringType(), True),
    ])
    
    df_regiones = spark.createDataFrame(regiones_data, schema=schema_regiones)
    df_regiones.write.mode("overwrite").saveAsTable(tabla_dim_regiones)
    
    print(f"   ✅ Tabla creada: {tabla_dim_regiones}")
    print(f"   📊 {df_regiones.count()} regiones registradas\n")

# ──────────────────────────────────────────────────────────────────────────
# 2️⃣ TABLA DIMENSIONAL: PROVINCIAS (con FK a regiones)
# ──────────────────────────────────────────────────────────────────────────
tabla_dim_provincias = f"{DB_NAME}.dim_provincias"

if spark.catalog.tableExists(tabla_dim_provincias):
    print(f"🗺️  2. Tabla {tabla_dim_provincias} ya existe")
    count_provincias = spark.table(tabla_dim_provincias).count()
    print(f"   ℹ️  {count_provincias} provincias registradas (sin cambios)\n")
else:
    print("🗺️  2. Creando dim_provincias...")
    
    provincias_data = [
        (1, "Azuay", 1),
        (2, "Bolívar", 1),
        (3, "Cañar", 1),
        (4, "Carchi", 1),
        (5, "Cotopaxi", 1),
        (6, "Chimborazo", 1),
        (7, "El Oro", 2),
        (8, "Esmeraldas", 2),
        (9, "Guayas", 2),
        (10, "Imbabura", 1),
        (11, "Loja", 1),
        (12, "Los Ríos", 2),
        (13, "Manabí", 2),
        (14, "Morona Santiago", 3),
        (15, "Napo", 3),
        (16, "Pastaza", 3),
        (17, "Pichincha", 1),
        (18, "Tungurahua", 1),
        (19, "Zamora Chinchipe", 3),
        (20, "Galápagos", 4),
        (21, "Sucumbíos", 3),
        (22, "Orellana", 3),
        (23, "Santo Domingo De Los Tsáchilas", 1),
        (24, "Santa Elena", 2),
    ]
    
    schema_provincias = StructType([
        StructField("provincia_id", IntegerType(), False),
        StructField("provincia", StringType(), False),
        StructField("region_id", IntegerType(), False),
    ])
    
    df_provincias = spark.createDataFrame(provincias_data, schema=schema_provincias)
    
    # Agregar variantes normalizadas para el mapeo (sin tildes, minúsculas)
    df_provincias = df_provincias.withColumn(
        "provincia_normalizada",
        F.lower(F.translate(F.col("provincia"), "áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnn"))
    )
    
    df_provincias.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla_dim_provincias)
    
    print(f"   ✅ Tabla creada: {tabla_dim_provincias}")
    print(f"   📊 {df_provincias.count()} provincias registradas\n")

# ──────────────────────────────────────────────────────────────────────────
# VISTA PREVIA
# ──────────────────────────────────────────────────────────────────────────
print("🔍 Vista previa de la estructura dimensional:\n")

print("📊 Consulta unificada (con JOIN):")
spark.table(tabla_dim_provincias).display()

print("\n✅ Estructura dimensional creada exitosamente!")
print("\n📝 Resumen:")
print("   • 4 regiones (Sierra, Costa, Amazonía, Insular)")
print("   • 24 provincias con FK a regiones")
print("   • provincia_normalizada para mapeo automático")

## 🔧 Bloque 5 — Funciones Utilitarias y Transformaciones Especializadas

Funciones compartidas entre LangGraph y LlamaIndex (independientes del framework):

- `normalizar_columna`, `identificar_fuente`
- `_aplanar_excel_openpyxl`, `_score_header`, `leer_archivo`
- `castear_columnas`, `mapear_provincia_a_id`
- `transformar_espac_t13_t26_mejorado_v3`
- `transformar_sipa_temperatura`, `transformar_uso_del_suelo`
- `transformar_faostat`, `transformar_precios_v2`

In [ ]:
def mapear_provincia_a_id(df_pandas: pd.DataFrame, columna_provincia: str = 'provincia') -> pd.DataFrame:
    """
    Mapea nombres de provincias a sus IDs usando la tabla dimensional.
    
    Args:
        df_pandas: DataFrame con datos que incluyen nombres de provincias
        columna_provincia: Nombre de la columna que contiene el nombre de provincia
    
    Returns:
        DataFrame con provincia_id en lugar de nombre de provincia
    """
    if columna_provincia not in df_pandas.columns:
        print(f"  ⚠️  Columna '{columna_provincia}' no encontrada, omitiendo mapeo")
        return df_pandas
    
    print(f"  🗺️  Mapeando provincias a IDs...")
    
    # Leer tabla dimensional como pandas para el mapeo
    dim_prov_pd = spark.table(f"{DB_NAME}.dim_provincias").toPandas()
    
    # Detectar columna de nombre en la tabla dimensional
    if 'provincia_normalizada' in dim_prov_pd.columns:
        col_nombre = 'provincia_normalizada'
    elif 'provincia_nombre' in dim_prov_pd.columns:
        col_nombre = 'provincia_nombre'
    elif 'provincia' in dim_prov_pd.columns:
        col_nombre = 'provincia'
    else:
        print(f"  \u26a0\ufe0f  No se encontr\u00f3 columna de nombre en dim_provincias")
        return df_pandas

    # Normalizar la columna de referencia
    dim_prov_pd['_norm'] = dim_prov_pd[col_nombre].apply(
        lambda x: str(x).strip().lower().translate(str.maketrans('\u00e1\u00e9\u00ed\u00f3\u00fa\u00c1\u00c9\u00cd\u00d3\u00da\u00f1\u00d1', 'aeiouAEIOUnn')) if pd.notna(x) else ''
    )
    mapeo_dict = dict(zip(dim_prov_pd['_norm'], dim_prov_pd['provincia_id']))
    
    # Normalizar la columna de provincia en el DataFrame de datos
    def normalizar_provincia(texto):
        if pd.isna(texto) or texto == '':
            return None
        # Convertir a string, quitar espacios, minusculas
        texto = str(texto).strip().lower()
        # Quitar tildes
        texto = texto.translate(str.maketrans('áéíóúÁÉÍÓÚñÑ', 'aeiouAEIOUnn'))
        return texto
    
    df_pandas['_provincia_norm'] = df_pandas[columna_provincia].apply(normalizar_provincia)
    
    # Mapear a provincia_id
    df_pandas['provincia_id'] = df_pandas['_provincia_norm'].map(mapeo_dict)
    
    # Verificar cuántas NO mapearon
    sin_mapear = df_pandas['provincia_id'].isna().sum()
    if sin_mapear > 0:
        print(f"    ⚠️  {sin_mapear} registros sin mapeo de provincia")
        provincias_no_mapeadas = df_pandas[df_pandas['provincia_id'].isna()][columna_provincia].unique()
        print(f"       Provincias no reconocidas: {list(provincias_no_mapeadas)[:5]}")
    
    # Eliminar columnas auxiliares y original
    df_pandas = df_pandas.drop(columns=['_provincia_norm', columna_provincia])
    
    # Convertir provincia_id a int (donde no sea null)
    df_pandas['provincia_id'] = df_pandas['provincia_id'].astype('Int64')
    
    mapeados = (~df_pandas['provincia_id'].isna()).sum()
    print(f"    ✅ {mapeados} provincias mapeadas correctamente")
    
    return df_pandas

print("✅ Función mapear_provincia_a_id() definida.")

In [ ]:
# Asignar la nueva versión corregida como la función predeterminada
transformar_espac_t13_t26_mejorado = transformar_espac_t13_t26_mejorado_v2

print("✅ Versión corregida activada: transformar_espac_t13_t26_mejorado ahora usa _v2")
print("   - Lee con pandas directamente (sin openpyxl)")
print("   - Valida rangos numéricos razonables")
print("   - Superficie < 500,000 ha")
print("   - Producción/Ventas < 10,000,000 tm")

In [ ]:
def transformar_espac_t13_t26_mejorado_v3(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    CORRECCIÓN #10 FINAL: Mapeo por ÍNDICE de columna fijo (no por nombre).
    Los archivos ESPAC tienen SIEMPRE la misma estructura de columnas:
      - Col 1: Provincia
      - Col 3: Superficie plantada
      - Col 4: Superficie cosechada
      - Col 5: Producción
      - Col 6: Ventas
    """
    import numpy as np
    
    print(f"  📊 Transformación ESPAC Tabulados/Series Históricas (v3 - mapeo por índice): {nombre_archivo}")
    
    # Extraer año del nombre del archivo
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    try:
        # Leer con pandas directamente
        xl_file = pd.ExcelFile(local_path)
        nombres_hojas = xl_file.sheet_names
        
        print(f"    📂 Hojas disponibles: {len(nombres_hojas)} hojas")
        
        # Buscar hojas T13 o T26
        hojas_relevantes = []
        for nombre_hoja in nombres_hojas:
            nombre_upper = nombre_hoja.upper()
            if 'T13' in nombre_upper or 'T26' in nombre_upper:
                hojas_relevantes.append(nombre_hoja)
        
        print(f"    ✅ Hojas relevantes identificadas: {hojas_relevantes if hojas_relevantes else 'Ninguna'}")
        
        if not hojas_relevantes:
            hojas_relevantes = [xl_file.sheet_names[0]]
        
        dfs_procesados = []
        
        for nombre_hoja in hojas_relevantes:
            print(f"    📄 Procesando hoja: {nombre_hoja}")
            
            try:
                # Leer sin header (header=None)
                df_raw = pd.read_excel(local_path, sheet_name=nombre_hoja, header=None)
                
                if df_raw.empty:
                    print(f"       ⚠️  Hoja vacía, saltando...")
                    continue
                
                # Buscar fila de header
                header_idx = 0
                for i in range(min(20, len(df_raw))):
                    fila_text = ' '.join(str(v) for v in df_raw.iloc[i].values if pd.notna(v)).upper()
                    if any(kw in fila_text for kw in ['PROVINCIA', 'SUPERFICIE', 'PRODUCCIÓN', 'PRODUCCION']):
                        header_idx = i
                        print(f"       🎯 Header detectado en fila {header_idx}")
                        break
                
                # Datos empiezan después del header
                df_datos = df_raw.iloc[header_idx+1:].copy().reset_index(drop=True)
                
                # MAPEO POR ÍNDICE DE COLUMNA FIJO (estructura ESPAC estándar)
                # Estructura SIEMPRE es: [0]=índice, [1]=provincia, [2]=categoría?, [3]=sup_plant, [4]=sup_cosech, [5]=prod, [6]=ventas
                # Pero con merge cells, algunas pueden estar en posiciones ligeramente diferentes
                
                # Buscar columna de provincia (primera columna con texto largo)
                col_provincia_idx = None
                for i in range(min(5, df_datos.shape[1])):
                    muestra = df_datos.iloc[:5, i].astype(str)
                    if muestra.str.len().mean() > 5:  # Provincias son texto largo
                        col_provincia_idx = i
                        break
                
                if col_provincia_idx is None:
                    print(f"       ⚠️  No se encontró columna de provincia, saltando...")
                    continue
                
                print(f"       📍 Provincia en col[{col_provincia_idx}]")
                
                # Las columnas numéricas están DESPUÉS de provincia
                # Normalmente: provincia(col 1), luego 4 columnas numéricas
                num_cols_start = col_provincia_idx + 1
                
                # Filtrar filas válidas (que tengan provincia)
                df_datos = df_datos[df_datos.iloc[:, col_provincia_idx].notna()].copy()
                
                # Filtrar palabras excluidas
                palabras_excluir = ['TOTAL', 'REGIÓN', 'REGION', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN']
                patron = '|'.join(palabras_excluir)
                mask = df_datos.iloc[:, col_provincia_idx].astype(str).str.upper().str.contains(patron, na=False, case=False, regex=True)
                df_datos = df_datos[~mask].copy().reset_index(drop=True)
                
                # Crear DataFrame estandarizado
                n_rows = len(df_datos)
                df_std = pd.DataFrame(index=range(n_rows))
                
                df_std['provincia'] = df_datos.iloc[:, col_provincia_idx].values
                df_std['categoria'] = 'Solo'  # No usamos categoría por ahora
                
                # MAPEO NUMÉRICO POR POSICIÓN RELATIVA
                # Después de provincia, las siguientes 4-5 columnas son numéricas
                # Identificar cuáles columnas son numéricas
                cols_numericas = []
                for i in range(num_cols_start, min(num_cols_start + 6, df_datos.shape[1])):
                    muestra = df_datos.iloc[:10, i].astype(str).str.replace(',', '.', regex=False)
                    try:
                        pd.to_numeric(muestra, errors='coerce')
                        es_numerica = muestra.str.match(r'^\d').sum() >= 3  # Al menos 3 valores parecen números
                        if es_numerica:
                            cols_numericas.append(i)
                    except:
                        pass
                
                print(f"       🔢 Columnas numéricas identificadas: {cols_numericas}")
                
                # Asignar en orden: plantada, cosechada, producción, ventas
                if len(cols_numericas) >= 4:
                    df_std['superficie_plantada_ha'] = df_datos.iloc[:, cols_numericas[0]].values
                    df_std['superficie_cosechada_ha'] = df_datos.iloc[:, cols_numericas[1]].values
                    df_std['produccion_tm'] = df_datos.iloc[:, cols_numericas[2]].values
                    df_std['ventas_tm'] = df_datos.iloc[:, cols_numericas[3]].values
                elif len(cols_numericas) >= 3:
                    df_std['superficie_plantada_ha'] = df_datos.iloc[:, cols_numericas[0]].values
                    df_std['superficie_cosechada_ha'] = 0
                    df_std['produccion_tm'] = df_datos.iloc[:, cols_numericas[1]].values
                    df_std['ventas_tm'] = df_datos.iloc[:, cols_numericas[2]].values
                else:
                    print(f"       ⚠️  Insuficientes columnas numéricas ({len(cols_numericas)}), usando ceros")
                    df_std['superficie_plantada_ha'] = 0
                    df_std['superficie_cosechada_ha'] = 0
                    df_std['produccion_tm'] = 0
                    df_std['ventas_tm'] = 0
                
                # Convertir numéricos con validación
                for col in ['superficie_plantada_ha', 'superficie_cosechada_ha', 'produccion_tm', 'ventas_tm']:
                    df_std[col] = df_std[col].astype(str).str.strip().str.replace(',', '.', regex=False)
                    df_std[col] = df_std[col].apply(lambda x: x.replace('.', '') if isinstance(x, str) and x.count('.') > 1 else x)
                    df_std[col] = pd.to_numeric(df_std[col], errors='coerce').fillna(0)
                    
                    # Validación de rangos
                    if 'superficie' in col:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 500000 else x)
                    else:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 10000000 else x)
                
                # Producto y año
                producto = 'BANANO' if 'T13' in nombre_hoja.upper() else 'PLATANO' if 'T26' in nombre_hoja.upper() else 'DESCONOCIDO'
                df_std['producto'] = producto
                df_std['anio'] = anio if anio else 0
                
                # Calcular rendimiento
                df_std['rendimiento'] = np.where(
                    df_std['superficie_cosechada_ha'] > 0,
                    (df_std['produccion_tm'] / df_std['superficie_cosechada_ha']).round(2),
                    0
                )
                
                print(f"       ✅ {len(df_std)} registros procesados de hoja {nombre_hoja}")
                dfs_procesados.append(df_std)
                
            except Exception as e_hoja:
                print(f"       ❌ Error en hoja {nombre_hoja}: {e_hoja}")
                continue
        
        xl_file.close()
        
        if dfs_procesados:
            df_final = pd.concat(dfs_procesados, ignore_index=True)
            df_final = df_final[df_final['provincia'].notna() & (df_final['provincia'].astype(str).str.strip() != '')]
            
            # 🆕 MAPEO DE PROVINCIA A ID (usando tabla dimensional)
            df_final = mapear_provincia_a_id(df_final, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            # 🚫 IMPORTANTE: NO aplicar KNN a provincia_id - los NULL son agregaciones que deben eliminarse
            antes_filtro = len(df_final)
            df_final = df_final.dropna(subset=['provincia_id'])
            despues_filtro = len(df_final)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
            
            # 🚫 KNN DESHABILITADO para ESPAC - provincia_id NULL debe eliminarse, no imputarse
            print(f"    ℹ️  KNN deshabilitado para ESPAC (provincia_id NULL = agregaciones, no errores)")
            
            print(f"    ✅ Transformación completada: {len(df_final)} registros totales, {len(df_final.columns)} columnas")
            return df_final
        else:
            print(f"    ⚠️  No se procesaron hojas, retornando DataFrame vacío")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"    ❌ Error en transformación ESPAC: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Activar la versión v3
transformar_espac_t13_t26_mejorado = transformar_espac_t13_t26_mejorado_v3

print("✅ CORRECCIÓN #10 APLICADA: Mapeo por índice de columna (v3)")
print("   - Identifica columna de provincia por contenido de texto")
print("   - Identifica columnas numéricas por patrón de dígitos")
print("   - Asigna en orden: plantada, cosechada, producción, ventas")

In [ ]:
def transformar_sipa_temperatura(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos SIPA de temperatura/precipitación.
    Aplica mapeo de provincia a ID.
    """
    print(f"  🌡️  Transformación SIPA Temperatura/Precipitación: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes de SIPA
        columnas_mapeo = {
            'ano': 'anio',
            'mes': 'mes',
            'estacion': 'estacion',
            'provincia': 'provincia',
            'canton': 'canton',
            'precipitacion_mm': 'precipitacion_mm',
            'temperatura_promedio_c': 'temperatura_promedio_c'
        }
        
        # Renombrar columnas si existen
        for old_col, new_col in columnas_mapeo.items():
            if old_col in df_clean.columns:
                df_clean = df_clean.rename(columns={old_col: new_col})
        
        # Convertir columnas numéricas - IMPORTANTE: preservar tipo INT
        if 'anio' in df_clean.columns:
            # Convertir a float primero, luego a int para manejar decimales (2000.0 → 2000)
            df_clean['anio'] = pd.to_numeric(df_clean['anio'], errors='coerce')
            # Redondear antes de convertir a int para evitar errores
            df_clean['anio'] = df_clean['anio'].round(0).fillna(0).astype(int)
            # Reemplazar 0 con NaN y eliminar esas filas
            df_clean['anio'] = df_clean['anio'].replace(0, pd.NA)
            # Asegurar que se mantenga como Int64 (nullable int)
            df_clean['anio'] = df_clean['anio'].astype('Int64')
        
        if 'mes' in df_clean.columns:
            # Convertir a int, manejando valores vacíos
            df_clean['mes'] = pd.to_numeric(df_clean['mes'], errors='coerce')
            df_clean['mes'] = df_clean['mes'].fillna(0).astype(int)
            # Si todos son 0, intentar extraer mes de otra columna si existe
            if (df_clean['mes'] == 0).all() and 'anio' in df_clean.columns:
                # Si no hay mes, agregar columna con 0 para indicar dato anual
                print(f"    ℹ️  Columna 'mes' vacía, usando 0 (dato anual)")
        elif 'anio' in df_clean.columns:
            # Si no existe columna mes, crearla con 0
            df_clean['mes'] = 0
            print(f"    ℹ️  Columna 'mes' no existe, creada con 0 (dato anual)")
        
        if 'precipitacion_mm' in df_clean.columns:
            df_clean['precipitacion_mm'] = pd.to_numeric(df_clean['precipitacion_mm'], errors='coerce')
        
        if 'temperatura_promedio_c' in df_clean.columns:
            df_clean['temperatura_promedio_c'] = pd.to_numeric(df_clean['temperatura_promedio_c'], errors='coerce')
        
        # Filtrar filas con valores nulos en campos críticos
        if 'anio' in df_clean.columns:
            antes = len(df_clean)
            df_clean = df_clean.dropna(subset=['anio'])
            despues = len(df_clean)
            if antes > despues:
                print(f"    🧹 Filtradas {antes - despues} filas con anio nulo")
        
        # 🎯 MAPEO DE PROVINCIA A ID (usando tabla dimensional)
        if 'provincia' in df_clean.columns:
            df_clean = mapear_provincia_a_id(df_clean, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            antes_filtro = len(df_clean)
            df_clean = df_clean.dropna(subset=['provincia_id'])
            despues_filtro = len(df_clean)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
                print(f"    ✅ {despues_filtro} registros con provincia válida")
        
        # ✅ APLICAR KNN A COLUMNAS NUMÉRICAS EN SIPA
        print(f"    🔧 Aplicando KNN a columnas numéricas con NULL...")
        
        # Identificar columnas numéricas con NULL (excluir IDs)
        columnas_numericas_con_null = []
        for col in df_clean.columns:
            if col in ['provincia_id', 'anio', 'mes']:  # No aplicar KNN a IDs y fechas
                continue
            try:
                col_numeric = pd.to_numeric(df_clean[col], errors='coerce')
                nulls = col_numeric.isna().sum()
                if nulls > 0 and nulls < len(df_clean):  # Tiene NULL pero no todos
                    columnas_numericas_con_null.append(col)
                    print(f"       {col}: {nulls} NULL ({nulls/len(df_clean)*100:.1f}%)")
            except:
                pass
        
        if columnas_numericas_con_null and len(df_clean) >= 5:
            try:
                from sklearn.impute import KNNImputer
                
                # Preparar datos para KNN
                features_knn = columnas_numericas_con_null.copy()
                if 'anio' in df_clean.columns:
                    features_knn = ['anio'] + features_knn
                if 'mes' in df_clean.columns and df_clean['mes'].nunique() > 1:
                    features_knn.insert(1, 'mes')
                
                # Convertir a numérico
                df_knn = df_clean[features_knn].copy()
                for col in df_knn.columns:
                    df_knn[col] = pd.to_numeric(df_knn[col], errors='coerce')
                
                # Aplicar KNN
                imputer = KNNImputer(n_neighbors=min(5, len(df_clean)), weights='distance')
                df_imputed = imputer.fit_transform(df_knn)
                
                # Reemplazar valores en columnas originales (sin año/mes)
                start_idx = 0
                if 'anio' in features_knn:
                    start_idx += 1
                if 'mes' in features_knn:
                    start_idx += 1
                
                for i, col in enumerate(columnas_numericas_con_null):
                    df_clean[col] = df_imputed[:, start_idx + i]
                
                print(f"    ✅ KNN aplicado a {len(columnas_numericas_con_null)} columnas")
            except Exception as e:
                print(f"    ⚠️  No se pudo aplicar KNN: {e}")
        else:
            print(f"    ℹ️  KNN omitido (no hay suficientes datos o columnas)")
        
        # Eliminar filas completamente vacías
        df_clean = df_clean.dropna(how='all').reset_index(drop=True)
        
        print(f"    ✅ Transformación completada: {len(df_clean)} registros, {len(df_clean.columns)} columnas")
        print(f"    📊 Columnas finales: {list(df_clean.columns)}")
        
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación SIPA: {e}")
        import traceback
        traceback.print_exc()
        return df_pandas

print("✅ Función de transformación SIPA Temperatura cargada (con mapeo de provincia_id y KNN)")

In [ ]:
def transformar_uso_del_suelo(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos de Uso del Suelo.
    La columna 'region_y_provincia' puede contener nombres de provincias O regiones.
    Solo mapeamos las provincias a provincia_id. Las regiones quedan sin mapear (NULL).
    """
    print(f"  🌾 Transformación Uso del Suelo: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes
        columnas_mapeo = {
            'ano': 'anio',
            'region_y_provincia': 'region_y_provincia',
            'categoria_de_uso_del_suelo': 'categoria_de_uso_del_suelo',
            'superficie_ha': 'superficie_ha'
        }
        
        # Renombrar columnas si existen
        for old_col, new_col in columnas_mapeo.items():
            if old_col in df_clean.columns:
                df_clean = df_clean.rename(columns={old_col: new_col})
        
        # Convertir columnas numéricas
        if 'anio' in df_clean.columns:
            df_clean['anio'] = pd.to_numeric(df_clean['anio'], errors='coerce').astype('Int64')
        
        if 'superficie_ha' in df_clean.columns:
            df_clean['superficie_ha'] = pd.to_numeric(df_clean['superficie_ha'], errors='coerce')
        
        # 🎯 SEPARAR region_y_provincia EN SOLO provincia
        # La columna puede tener nombres de provincia directos ("Azuay", "Guayas")
        # o nombres de región ("Centro-Suroriente", "Costa", etc.)
        # Solo mapeamos provincias, las regiones quedan como NULL y se ELIMINAN
        if 'region_y_provincia' in df_clean.columns:
            # Renombrar temporalmente para el mapeo
            df_clean = df_clean.rename(columns={'region_y_provincia': 'provincia'})
            
            print(f"    🗺️  Intentando mapear valores de región_y_provincia a provincia_id...")
            
            # Aplicar mapeo - los valores que NO son provincias quedarán NULL
            df_clean = mapear_provincia_a_id(df_clean, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            antes_filtro = len(df_clean)
            df_clean = df_clean.dropna(subset=['provincia_id'])
            despues_filtro = len(df_clean)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
                print(f"    ✅ {despues_filtro} registros con provincia válida")
        
        # Filtrar filas con valores nulos en campos críticos
        if 'anio' in df_clean.columns:
            antes = len(df_clean)
            df_clean = df_clean.dropna(subset=['anio'])
            despues = len(df_clean)
            if antes > despues:
                print(f"    🧹 Filtradas {antes - despues} filas con anio nulo")
        
        # ✅ APLICAR KNN A COLUMNAS NUMÉRICAS EN USO DEL SUELO
        print(f"    🔧 Aplicando KNN a columnas numéricas con NULL...")
        
        # Identificar columnas numéricas con NULL (excluir IDs)
        columnas_numericas_con_null = []
        for col in df_clean.columns:
            if col in ['provincia_id', 'anio']:  # No aplicar KNN a IDs y fechas
                continue
            try:
                col_numeric = pd.to_numeric(df_clean[col], errors='coerce')
                nulls = col_numeric.isna().sum()
                if nulls > 0 and nulls < len(df_clean):  # Tiene NULL pero no todos
                    columnas_numericas_con_null.append(col)
                    print(f"       {col}: {nulls} NULL ({nulls/len(df_clean)*100:.1f}%)")
            except:
                pass
        
        if columnas_numericas_con_null and len(df_clean) >= 5:
            try:
                from sklearn.impute import KNNImputer
                
                # Preparar datos para KNN
                features_knn = columnas_numericas_con_null.copy()
                if 'anio' in df_clean.columns:
                    features_knn = ['anio'] + features_knn
                if 'provincia_id' in df_clean.columns:
                    features_knn.insert(0 if 'anio' not in features_knn else 1, 'provincia_id')
                
                # Convertir a numérico
                df_knn = df_clean[features_knn].copy()
                for col in df_knn.columns:
                    df_knn[col] = pd.to_numeric(df_knn[col], errors='coerce')
                
                # Aplicar KNN
                imputer = KNNImputer(n_neighbors=min(5, len(df_clean)), weights='distance')
                df_imputed = imputer.fit_transform(df_knn)
                
                # Reemplazar valores en columnas originales (sin provincia_id/año)
                start_idx = 0
                if 'provincia_id' in features_knn:
                    start_idx += 1
                if 'anio' in features_knn:
                    start_idx += 1
                
                for i, col in enumerate(columnas_numericas_con_null):
                    df_clean[col] = df_imputed[:, start_idx + i]
                
                print(f"    ✅ KNN aplicado a {len(columnas_numericas_con_null)} columnas")
            except Exception as e:
                print(f"    ⚠️  No se pudo aplicar KNN: {e}")
        else:
            print(f"    ℹ️  KNN omitido (no hay suficientes datos o columnas)")
        
        # Eliminar filas completamente vacías
        df_clean = df_clean.dropna(how='all').reset_index(drop=True)
        
        print(f"    ✅ Transformación completada: {len(df_clean)} registros, {len(df_clean.columns)} columnas")
        print(f"    📊 Columnas finales: {list(df_clean.columns)}")
        
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación Uso del Suelo: {e}")
        import traceback
        traceback.print_exc()
        return df_pandas

print("✅ Función de transformación Uso del Suelo cargada (con mapeo de provincia_id y KNN)")

In [ ]:
def transformar_faostat(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos FAOSTAT (producción banan/plátano).
    Normaliza columnas y aplica mapeo de nombres.
    """
    print(f"  🌍 Transformación FAOSTAT: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes de FAOSTAT
        columnas_mapeo = {
            'area_code': 'pais_codigo',
            'area': 'pais',
            'item_code': 'producto_codigo',
            'item': 'producto',
            'element_code': 'elemento_codigo',
            'element': 'elemento',
            'year': 'anio',
            'value': 'valor',
            'unit': 'unidad'
        }
        
        df_clean = df_clean.rename(columns=columnas_mapeo)
        
        # Convertir columnas numéricas
        for col in ['pais_codigo', 'producto_codigo', 'elemento_codigo', 'anio']:
            if col in df_clean.columns:
                df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
        if 'valor' in df_clean.columns:
            df_clean['valor'] = pd.to_numeric(df_clean['valor'], errors='coerce')
        
        # Filtrar filas sin datos válidos
        registros_antes = len(df_clean)
        df_clean = df_clean.dropna(subset=['anio', 'valor'], how='any')
        registros_eliminados = registros_antes - len(df_clean)
        
        if registros_eliminados > 0:
            print(f"    🗑️  Filtrados {registros_eliminados} registros sin año o valor")
        
        print(f"    ✅ Transformación completada: {len(df_clean):,} registros, {len(df_clean.columns)} columnas")
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación FAOSTAT: {e}")
        print(f"    🔄 Usando estructura original...")
        return df_pandas

print("✅ Función transformar_faostat() creada")

In [ ]:
def transformar_aebe_bananotas(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma PDFs de AEBE Bananotas extrayendo rankings de:
    - Exportadores (Ubesa, Reybanpac, etc.)
    - Marcas (Dole, Chiquita, etc.)
    - Navieras (MSC, Maersk, etc.)
    - Puertos (San Petersburgo, Rotterdam, etc.)
    - Mercados/países (Unión Europea, Medio Oriente, etc.)

    MOTOR: pdfplumber (puro Python, sin JVM) en lugar de tabula-py.
    tabula-py lanza un subproceso JVM por cada llamada a read_pdf(), y el
    código anterior lo invocaba ~20 veces por PDF (5 tipos x ~4 páginas),
    lo que multiplicaba el overhead de arranque de JVM y explicaba los
    tiempos de horas. pdfplumber abre el PDF una sola vez y recorre todas
    las páginas en una sola pasada en memoria.

    DETECCIÓN DINÁMICA: no se asume un rango fijo de páginas (ej. 65-76,
    que puede variar de edición a edición). En su lugar, se inspecciona el
    texto de CADA página del PDF buscando tablas cuya fila de cabecera
    contenga dos columnas de año consecutivas (ej. "2025" y "2026") como
    valores aislados — patrón característico de las tablas de ranking de
    Bananotas. El tipo (exportador/marca/naviera/puerto/mercado) se infiere
    por palabras clave en el título inmediatamente anterior a la tabla, con
    fallback a la propia fila de cabecera.

    NORMALIZACIÓN A UN SOLO AÑO: cada tabla de ranking trae dos columnas de
    cantidad (año N y año N+1, ej. 2025 y 2026). Se conserva únicamente la
    columna del año MÁS RECIENTE de cada tabla — no se asume que todas las
    tablas del PDF compartan el mismo par de años, por lo que el año
    "más reciente" se calcula tabla por tabla, y el año de la revista
    completa es el máximo visto en todas las tablas.

    Estructura normalizada (UN registro por entidad del ranking):
    tipo | dato | cantidad | medida | anio | framework | fuente | archivo_origen

    Args:
        local_path: Ruta local al PDF descargado
        nombre_archivo: Nombre del archivo

    Returns:
        DataFrame consolidado con todos los rankings del año más reciente
        detectado en cada tabla.
    """
    print(f"  📊 Transformación AEBE Bananotas (pdfplumber): {nombre_archivo}")

    try:
        import pdfplumber
    except ImportError:
        print("  ⚠️  Instalando pdfplumber...")
        import subprocess
        subprocess.run(["pip", "install", "-q", "pdfplumber"], check=True)
        import pdfplumber

    COLUMNAS_VACIAS = ['tipo', 'dato', 'cantidad', 'medida', 'anio', 'framework', 'fuente', 'archivo_origen']

    RANKING_TIPOS = {
        'exportador': ['EXPORTADOR', 'EXPORTADORES'],
        'marca':      ['MARCA', 'MARCAS'],
        'naviera':    ['NAVIERA', 'NAVIERAS'],
        'puerto':     ['PUERTO', 'PUERTOS', 'DESTINO'],
        'mercado':    ['PAIS', 'PAÍS', 'MERCADO', 'MERCADOS'],
    }

    def _detectar_anios_header(tabla_filas):
        """Fila de cabecera real = la que contiene >=2 años de 4 dígitos.
        Busca años dentro de las celdas (no solo celdas que sean exactamente un año),
        para manejar headers con formato 'R MARCAS 2025 2026\n2025 2026...'."""
        for fila in tabla_filas:
            celdas = [str(c).strip() if c else "" for c in fila]
            # Buscar TODOS los años en TODAS las celdas (incluso si hay otros textos)
            anios_encontrados = []
            for celda in celdas:
                # Encuentra todos los años de 4 dígitos en esta celda
                matches = re.findall(r'\b20[12][0-9]\b', celda)
                anios_encontrados.extend(matches)
            
            # Si encontramos al menos 2 años distintos, es el header
            anios_unicos = sorted(set(int(a) for a in anios_encontrados))
            if len(anios_unicos) >= 2:
                return anios_unicos, fila
        return None, None

    def _clasificar_tipo(texto):
        texto_up = texto.upper()
        for tipo, keywords in RANKING_TIPOS.items():
            if any(kw in texto_up for kw in keywords):
                return tipo
        return None

    def _limpiar_numero(val):
        if val is None:
            return None
        s = str(val).strip()
        if s in ("", "-", "—", "–"):
            return None
        s = s.replace("%", "")
        s = re.sub(r"\s+", "", s)
        if "," in s and "." in s:
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
        try:
            return float(s)
        except ValueError:
            return None

    registros = []
    anios_vistos = []

    try:
        with pdfplumber.open(local_path) as pdf:
            print(f"    🔍 Escaneando {len(pdf.pages)} páginas (detección dinámica)...")
            for page_num, page in enumerate(pdf.pages, start=1):
                # 1) intento con tablas de bordes reales (lattice); si no hay,
                #    fallback a deteccion por alineacion de texto (streams)
                tables = page.find_tables(table_settings={
                    "vertical_strategy": "lines", "horizontal_strategy": "lines"
                })
                if not tables:
                    tables = page.find_tables(table_settings={
                        "vertical_strategy": "text", "horizontal_strategy": "text"
                    })
                if not tables:
                    continue

                for tabla_obj in tables:
                    filas = tabla_obj.extract()
                    if not filas or len(filas) < 3:
                        continue

                    anios, header_fila = _detectar_anios_header(filas)
                    if not anios or header_fila is None:
                        continue  # no es tabla de ranking (sin 2 cols de año)

                    anio_reciente = max(anios)

                    # Título: texto de la página por encima del bbox de la tabla
                    top_tabla = tabla_obj.bbox[1]
                    texto_arriba = page.crop((0, 0, page.width, top_tabla)).extract_text() or ""
                    lineas_arriba = [l for l in texto_arriba.split("\n") if l.strip()]
                    titulo_candidato = " ".join(lineas_arriba[-3:]) if lineas_arriba else ""

                    tipo = _clasificar_tipo(titulo_candidato)
                    if tipo is None:
                        tipo = _clasificar_tipo(" ".join(str(c) for c in header_fila if c))
                    if tipo is None:
                        continue

                    anios_vistos.append(anio_reciente)
                    n_antes = len(registros)

                    # CASO ESPECIAL: Tabla extraída como UNA SOLA CELDA
                    # (común en PDFs sin bordes de tabla claros)
                    if len(header_fila) == 1 and len(header_fila[0]) > 50:
                        # Parseo manual de filas con patrón: ranking + nombre + valores
                        for fila in filas:
                            if not fila or not fila[0]:
                                continue
                            fila_texto = str(fila[0]).strip()
                            
                            # Patrón: dígitos (ranking) + texto (nombre) + 2+ números
                            match = re.match(r'^(\d{1,2})\s+(.+?)\s+([\d.]+)\s+([\d.]+)', fila_texto)
                            if not match:
                                continue
                            
                            ranking_txt = match.group(1)
                            nombre = match.group(2).strip()
                            
                            # Saltar filas de totales/otros
                            if nombre.upper() in ("TOTAL", "TOTAL GENERAL", "OTROS"):
                                continue
                            
                            # Los dos primeros valores numéricos son para los 2 años
                            valor_anio1 = _limpiar_numero(match.group(3))
                            valor_anio2 = _limpiar_numero(match.group(4))
                            
                            # Tomar el valor del año más reciente
                            # Si hay 2 años [2025, 2026], el más reciente es 2026 (segundo)
                            cantidad = valor_anio2 if len(anios) >= 2 and anio_reciente == anios[-1] else valor_anio1
                            
                            if cantidad is None or cantidad <= 0:
                                continue
                            
                            registros.append({
                                'tipo': tipo,
                                'dato': nombre,
                                'cantidad': cantidad,
                                'medida': 'cajas (millones) de 18.14kg',
                                'anio': anio_reciente,
                                'framework': 'llamaindex',
                                'fuente': 'AEBE_BANANOTAS',
                                'archivo_origen': nombre_archivo,
                            })
                    
                    else:
                        # CASO NORMAL: Tabla con columnas separadas
                        col_idx = None
                        for i, cell in enumerate(header_fila):
                            if cell and str(anio_reciente) in str(cell):
                                col_idx = i
                                break
                        if col_idx is None:
                            continue

                        for fila in filas:
                            if not fila or fila[0] is None:
                                continue
                            rank_val = str(fila[0]).strip()
                            if not re.match(r"^\d{1,2}$", rank_val):
                                continue

                            nombre = str(fila[1]).strip() if len(fila) > 1 and fila[1] else None
                            if not nombre or nombre.upper() in ("TOTAL", "TOTAL GENERAL", "OTROS"):
                                continue

                            if col_idx >= len(fila):
                                continue
                            cantidad = _limpiar_numero(fila[col_idx])
                            if cantidad is None or cantidad <= 0:
                                continue

                            registros.append({
                                'tipo': tipo,
                                'dato': nombre,
                                'cantidad': cantidad,
                                'medida': 'cajas (millones) de 18.14kg',
                                'anio': anio_reciente,
                                'framework': 'llamaindex',
                                'fuente': 'AEBE_BANANOTAS',
                                'archivo_origen': nombre_archivo,
                            })

                    n_nuevos = len(registros) - n_antes
                    if n_nuevos > 0:
                        print(f"       ✅ p.{page_num} [{tipo}] {n_nuevos} registros (año {anio_reciente})")

        if not registros:
            print(f"    ⚠️  No se encontraron rankings en el PDF - devolviendo DataFrame vacío")
            return pd.DataFrame(columns=COLUMNAS_VACIAS)

        anio_revista = max(anios_vistos)
        df_final = pd.DataFrame(registros)
        print(f"    ✅ Transformación completada: {len(df_final)} registros | año revista: {anio_revista} | tipos: {sorted(df_final['tipo'].unique())}")
        return df_final

    except Exception as e:
        print(f"    ❌ Error en transformación: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame(columns=COLUMNAS_VACIAS)


print("✅ Función transformar_aebe_bananotas (pdfplumber) cargada.")


In [ ]:
# Versión 2: Parseo basado en texto (sin detección de tablas)
def transformar_aebe_bananotas_v2(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """Versión simplificada que parsea texto directamente sin depender de find_tables."""
    print(f"  📊 Transformación AEBE Bananotas v2 (texto): {nombre_archivo}")
    
    try:
        import pdfplumber
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "pdfplumber"], check=True)
        import pdfplumber
    
    COLUMNAS_VACIAS = ['tipo', 'dato', 'cantidad', 'medida', 'anio', 'framework', 'fuente', 'archivo_origen']
    
    RANKING_KEYWORDS = {
        'exportador': ['EXPORTADOR'],
        'marca': ['MARCA'],
        'naviera': ['NAVIERA'],
        'puerto': ['PUERTO'],
        'mercado': ['PAIS', 'PAÍS', 'MERCADO'],
    }
    
    def _limpiar_numero(val):
        if not val:
            return None
        s = str(val).strip().replace("%", "").replace(",", ".")
        s = re.sub(r"\s+", "", s)
        try:
            return float(s)
        except:
            return None
    
    registros = []
    anio_revista = None
    
    try:
        with pdfplumber.open(local_path) as pdf:
            print(f"    🔍 Escaneando {len(pdf.pages)} páginas...")
            
            for page_num, page in enumerate(pdf.pages, start=1):
                texto = page.extract_text()
                if not texto:
                    continue
                
                lineas = texto.split('\n')
                
                # Buscar líneas con header (2 años consecutivos)
                for idx, linea in enumerate(lineas):
                    # Detectar header con años (1 o más)
                    anios = re.findall(r'\b20[12][0-9]\b', linea)
                    if len(set(anios)) < 1:
                        continue
                    
                    anios_unicos = sorted(set(int(a) for a in anios))
                    anio_reciente = max(anios_unicos)
                    
                    if anio_revista is None or anio_reciente > anio_revista:
                        anio_revista = anio_reciente
                    
                    # Determinar tipo de ranking
                    tipo = None
                    contexto = ' '.join(lineas[max(0, idx-3):idx+1]).upper()
                    for t, keywords in RANKING_KEYWORDS.items():
                        if any(kw in contexto for kw in keywords):
                            tipo = t
                            break
                    
                    if not tipo:
                        # Inferir tipo por contenido de datos
                        for linea_test in lineas[idx+1:idx+5]:
                            if re.match(r'^\d{1,2}\s+', linea_test):
                                tipo = 'exportador'  # default
                                break
                        if not tipo:
                            continue
                    
                    # Parsear filas de datos después del header
                    n_antes = len(registros)
                    for linea_datos in lineas[idx+1:idx+25]:  # Máximo 25 líneas después
                        # Patrón: ranking + nombre + números
                        match = re.match(r'^(\d{1,2})\s+(.+?)\s+([\d.,]+)(?:\s+([\d.,]+))?', linea_datos)
                        if not match:
                            continue
                        
                        ranking = match.group(1)
                        nombre = match.group(2).strip()
                        
                        if nombre.upper() in ['TOTAL', 'OTROS', 'TOTAL GENERAL']:
                            continue
                        
                        valor1 = _limpiar_numero(match.group(3))
                        valor2 = _limpiar_numero(match.group(4))
                        
                        # Segundo valor = año más reciente
                        cantidad = valor2 if valor2 else valor1
                        
                        if not cantidad or cantidad <= 0:
                            continue
                        
                        registros.append({
                            'tipo': tipo,
                            'dato': nombre,
                            'cantidad': cantidad,
                            'medida': 'cajas (millones) de 18.14kg',
                            'anio': anio_reciente,
                            'framework': 'llamaindex',
                            'fuente': 'AEBE_BANANOTAS',
                            'archivo_origen': nombre_archivo,
                        })
                    
                    if len(registros) > n_antes:
                        print(f"       ✅ p.{page_num} [{tipo}] {len(registros)-n_antes} registros (año {anio_reciente})")
                        break  # Solo procesar el primer ranking por página
        
        if not registros:
            print(f"    ⚠️  No se encontraron rankings")
            return pd.DataFrame(columns=COLUMNAS_VACIAS)
        
        df_final = pd.DataFrame(registros)
        print(f"    ✅ {len(df_final)} registros | año: {anio_revista} | tipos: {sorted(df_final['tipo'].unique())}")
        return df_final
    
    except Exception as e:
        print(f"    ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame(columns=COLUMNAS_VACIAS)

print("✅ transformar_aebe_bananotas_v2 cargada")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FUNCIONES AUXILIARES PARA AEBE (limpieza de números y porcentajes)
# ══════════════════════════════════════════════════════════════════════

def limpiar_numero(valor) -> float:
    """
    Extrae un número flotante de un string, manejando:
    - Separadores de miles (comas, puntos)
    - Decimales
    - Valores NULL/vacíos
    
    Ejemplos:
        "17,084.5" → 17084.5
        "17.084,5" → 17084.5
        "25.79%" → 25.79
        "" → None
    """
    if pd.isna(valor) or valor == '' or valor is None:
        return None
    
    try:
        # Convertir a string y limpiar
        s = str(valor).strip()
        
        # Eliminar símbolos no numéricos (excepto . , - +)
        s = re.sub(r'[^\d.,-]', '', s)
        
        if not s or s in ['.', ',', '-']:
            return None
        
        # Detectar formato: si tiene coma después de punto, es formato europeo
        if ',' in s and '.' in s:
            if s.rindex(',') > s.rindex('.'):
                # Formato europeo: 1.234,56
                s = s.replace('.', '').replace(',', '.')
            else:
                # Formato americano: 1,234.56
                s = s.replace(',', '')
        elif ',' in s:
            # Solo comas: puede ser separador de miles o decimal
            # Si hay más de una coma, es separador de miles
            if s.count(',') > 1:
                s = s.replace(',', '')
            else:
                # Una sola coma: puede ser decimal (europeo) o miles (americano)
                # Asumimos decimal si hay 1-2 dígitos después
                partes = s.split(',')
                if len(partes) == 2 and len(partes[1]) <= 2:
                    s = s.replace(',', '.')
                else:
                    s = s.replace(',', '')
        
        return float(s)
    
    except (ValueError, AttributeError):
        return None


def limpiar_porcentaje(valor) -> float:
    """
    Extrae un porcentaje y lo convierte a decimal.
    
    Ejemplos:
        "25.79%" → 25.79
        "4.39" → 4.39
        "25,79%" → 25.79
    """
    if pd.isna(valor) or valor == '' or valor is None:
        return None
    
    try:
        s = str(valor).strip()
        # Eliminar símbolo de porcentaje
        s = s.replace('%', '').strip()
        return limpiar_numero(s)
    
    except:
        return None

# ══════════════════════════════════════════════════════════════════════════
# VALIDACION DE CALIDAD M2 (Quality Gate: Silver -> Gold)
# ══════════════════════════════════════════════════════════════════════════

def validar_calidad_m2(df_pandas, tabla_destino):
    """
    Valida cada registro del DataFrame Silver antes de cargarlo a Gold.
    Retorna: (df_validos, total, validos, calidad_pct)
    
    Checks:
    1. Completitud: campos clave no nulos
    2. Validez: valores numericos en rango razonable
    3. Unicidad: sin duplicados exactos
    """
    if df_pandas is None or df_pandas.empty:
        return df_pandas, 0, 0, 0.0
    
    total = len(df_pandas)
    mask_valido = pd.Series([True] * total, index=df_pandas.index)
    
    # 1. Completitud: al menos 50% de columnas no nulas por fila
    min_cols = max(1, len(df_pandas.columns) // 2)
    mask_valido &= df_pandas.notna().sum(axis=1) >= min_cols
    
    # 2. Validez numerica
    for col_name in df_pandas.select_dtypes(include=['float64','int64']).columns:
        if any(k in col_name for k in ['superficie','produccion','ventas','cantidad','valor']):
            mask_valido &= (df_pandas[col_name].isna()) | ((df_pandas[col_name] >= 0) & (df_pandas[col_name] < 50_000_000))
        if 'anio' in col_name:
            mask_valido &= (df_pandas[col_name].isna()) | ((df_pandas[col_name] >= 1900) & (df_pandas[col_name] <= 2030))
    
    # 3. Unicidad
    duplicados = df_pandas.duplicated(keep='first')
    mask_valido &= ~duplicados
    
    df_validos = df_pandas[mask_valido].reset_index(drop=True)
    validos = len(df_validos)
    calidad_pct = round((validos / total) * 100, 1) if total > 0 else 0.0
    
    print(f"  M2 Quality Gate: {validos}/{total} registros pasan ({calidad_pct}%)")
    if total - validos > 0:
        print(f"     Descartados: {total - validos} (completitud/validez/unicidad)")
    
    return df_validos, total, validos, calidad_pct

print("\u2705 Funciones auxiliares cargadas: limpiar_numero(), limpiar_porcentaje(), validar_calidad_m2()")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DICCIONARIO DE CONOCIMIENTO CORREGIDO
# Mapeo archivo → tabla Delta (usado por el LLM para decidir destino)
# ═══════════════════════════════════════════════════════════════════════════

DICCIONARIO_CONOCIMIENTO = """
{
  "mapeo_archivos": [
    {"archivo": "ESPAC_T13",                    "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_T26",                    "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_Tabulados_excel",        "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_Series_historicas",      "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_USO_DEL_SUELO",          "tabla_destino": "espac_uso_del_suelo",            "fuente": "INEC_ESPAC"},
    {"archivo": "SIPA_USO_SUELO",               "tabla_destino": "espac_uso_del_suelo",            "fuente": "SIPA_MAG"},
    {"archivo": "TEMPERATURA_Y_PRECIPITACION",  "tabla_destino": "sipa_temperatura_precipitacion", "fuente": "SIPA_MAG"},
    {"archivo": "SIPA_TEMPERATURA",             "tabla_destino": "sipa_temperatura_precipitacion", "fuente": "SIPA_MAG"},
    {"archivo": "FAOSTAT",                      "tabla_destino": "faostat_produccion_banano_platano", "fuente": "FAOSTAT"},
    {"archivo": "AEBE_BANANOTAS",               "tabla_destino": "aebe_exportaciones_regiones",      "fuente": "AEBE_BANANOTAS",      "descripcion": "Rankings consolidados: exportadores, marcas, navieras, puertos, mercados. Año extraído del contenido (max year en texto). Estructura: tipo|ranking|dato|cantidad_2025|cantidad_2026|participacion_2025|participacion_2026|variacion_abs|variacion_pct|medida|anio|archivo"}
  ]
}
"""

# ═══════════════════════════════════════════════════════════════════════════
# CREAR/VERIFICAR TABLA AEBE CON SCHEMA CORRECTO
# ═══════════════════════════════════════════════════════════════════════════

# Borrar tabla vieja si existe con schema incorrecto
try:
    spark.sql(f"DROP TABLE IF EXISTS {DB_NAME}.aebe_exportaciones_regiones")
    print("♻️  Tabla aebe_exportaciones_regiones eliminada para recrear con schema correcto")
except:
    pass

# Crear tabla con schema CORRECTO (estructura normalizada)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.aebe_exportaciones_regiones (
    tipo              STRING COMMENT 'Tipo de ranking: exportador, marca, naviera, puerto, mercado',
    dato              STRING COMMENT 'Nombre de la entidad (empresa, marca, puerto, país)',
    cantidad          DOUBLE COMMENT 'Valor para el año detectado (cajas en millones)',
    medida            STRING COMMENT 'Unidad de medida (ej: cajas (millones))',
    anio              INT    COMMENT 'Año del dato (extraído del documento, siempre el mayor)',
    framework         STRING COMMENT 'Framework ETL usado: langraph o llamaindex',
    fuente            STRING COMMENT 'Fuente: AEBE_BANANOTAS',
    archivo_origen    STRING COMMENT 'Nombre del archivo PDF procesado',
    fecha_carga       TIMESTAMP COMMENT 'Timestamp de carga a Delta'
) USING DELTA
COMMENT 'Rankings de exportaciones de banano Ecuador desde AEBE Bananotas (estructura normalizada)'
""")

print("✅ Diccionario de conocimiento cargado.")
print("   • ESPAC_Tabulados_excel y ESPAC_Series_historicas → espac_banano_platano_provincia")
print("   • FAOSTAT → faostat_produccion_banano_platano")
print("   • SIPA_TEMPERATURA → sipa_temperatura_precipitacion")
print("✅ Tabla AEBE creada/verificada:")
print("   • AEBE_BANANOTAS → aebe_exportaciones_regiones (tipo|dato|cantidad|medida|año)")

## 🔄 Bloque 6 — ETL Workflow: Transformación y Carga (Agente 2)

Replica el Agente ETL de LangGraph con la misma lógica de transformación:

| Paso | LangGraph (nodo) | LlamaIndex (@step) |
|------|------------------|--------------------|
| 1 | `nodo_detectar_archivo` | `detectar_archivo` |
| 2 | `nodo_leer_archivo` | `leer_archivo_step` |
| 3 | `nodo_consultar_llm_mapeo` | `mapear_columnas` |
| 4 | `nodo_transformar_datos` | `transformar_datos` |
| 5 | `nodo_cargar_delta` | `cargar_delta` |

Las métricas M1–M5 se registran en las mismas tablas Delta con `framework = 'LlamaIndex'`.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLAS DE MÉTRICAS ETL (compatibles con LangGraph)
# ══════════════════════════════════════════════════════════════════════════

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TRANSFORM} (
    execution_id        STRING,
    file_name           STRING,
    fuente              STRING,
    framework           STRING,
    tabla_destino       STRING,
    filas_entrada       INT,
    filas_salida        INT,
    duplicados_elim     INT,
    calidad_pct         DOUBLE,
    dpmo_transformacion DOUBLE,
    tiempo_segundos     DOUBLE,
    timestamp_inicio    TIMESTAMP,
    timestamp_fin       TIMESTAMP,
    analisis_agente     STRING,
    llamadas_llm        INT,
    status              STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_CARGA} (
    execution_id         STRING,
    file_name            STRING,
    fuente               STRING,
    framework            STRING,
    tabla_destino        STRING,
    registros_insertados INT,
    tiempo_escritura_s   DOUBLE,
    timestamp_inicio     TIMESTAMP,
    timestamp_fin        TIMESTAMP,
    analisis_agente      STRING,
    llamadas_llm         INT,
    status               STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    execution_id            STRING,
    file_name               STRING,
    framework               STRING,
    fuente                  STRING,
    tabla_destino           STRING,
    execution_timestamp     TIMESTAMP,
    status                  STRING,
    m1_tiempo_segundos      DOUBLE,
    m2_intervencion_manual  INT,
    m3_recuperacion_errores INT,
    m4_calidad_pct          DOUBLE,
    m5_llamadas_api         INT,
    m6_dpmo                 DOUBLE,
    total_filas             INT,
    registros_validos       INT,
    registros_duplicados    INT,
    llm_response            STRING,
    error_message           STRING,
    escenario               STRING,
    reintentos_realizados   INT,
    recuperacion            INT
) USING DELTA
""")

print("OK Tablas de metricas ETL listas (compatibles con LangGraph).")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EVENTOS AGENTE 2: ETL (Transformación + Carga)
# ══════════════════════════════════════════════════════════════════════════

class ETLStartEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    timestamp_inicio: str

class ArchivoDetectadoEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    extension:        str
    timestamp_inicio: str
    llamadas_llm:     int

class ArchivoLeidoEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    extension:        str
    columnas_raw:     List[str]
    n_registros_raw:  int
    timestamp_inicio: str
    llamadas_llm:     int

class ColumnasMapeadasEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    tabla_destino:    str
    cols_double:      List[str]
    cols_integer:     List[str]
    razonamiento_llm: str
    columnas_raw:     List[str]
    n_registros_raw:  int
    timestamp_inicio: str
    llamadas_llm:     int

class DatosTransformadosEvent(Event):
    nombre_archivo:   str
    fuente:           str
    tabla_destino:    str
    ruta_archivo:     str
    n_registros:      int
    registros_dup:    int
    calidad_pct:      float
    dpmo:             float
    columnas_raw:     List[str]
    temp_view_name:   str
    razonamiento_llm: str
    timestamp_inicio: str
    llamadas_llm:     int

class ETLErrorEvent(Event):
    nombre_archivo:   str
    fuente:           str
    causa:            str
    timestamp_inicio: str
    llamadas_llm:     int


def _mapeo_por_reglas(file_name: str, columnas: List[str]) -> dict:
    """Fallback identico al de LangGraph cuando el LLM falla."""
    fn   = file_name.upper()
    cols = " ".join(columnas).lower()
    if any(k in fn for k in ["TABULADOS_EXCEL","SERIES_HISTORICAS","SERIES_HIST","TABULADOS_ESPAC"]):
        return {"tabla":"espac_banano_platano_provincia",
                "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"],
                "int":["anio"]}
    if "T13" in fn or "T26" in fn:
        return {"tabla":"espac_banano_platano_provincia",
                "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"],
                "int":["anio"]}
    if "SIPA_USO" in fn or ("uso" in cols and "suelo" in cols):
        return {"tabla":"espac_uso_del_suelo","double":["superficie_ha"],"int":["anio"]}
    if "TEMPERATURA" in fn or ("temperatura" in cols and "precipitacion" in cols):
        return {"tabla":"sipa_temperatura_precipitacion",
                "double":["precipitacion_mm","temperatura_promedio_c"],"int":["anio"]}
    if "FAOSTAT" in fn:
        return {"tabla":"faostat_produccion_banano_platano","double":["value"],"int":["year","area_code","item_code"]}
    if "BANANO_PREC" in fn or "precios" in fn.lower():
        return {"tabla":"banano_precios_semanales","double":[],"int":["anio"]}
    return {"tabla":"tabla_temporal","double":[],"int":[]}


print("OK Eventos AGENTE 2 (ETL) y helper de mapeo definidos.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ESTADO ETL Y FUNCIONES DE TRANSFORMACIÓN
# Funciones compartidas con LangGraph (patrón bridge)
# ══════════════════════════════════════════════════════════════════════════
# ── DEFINICIÓN DEL ESTADO ──────────────────────────────────────────────────
class ETLState(TypedDict):
    # ── Input ─────────────────────────────────────────────────────────────
    file_name:              str
    local_path:             str
    # ── Nodo 1: Detección ─────────────────────────────────────────────────
    fuente:                 str
    inicio_ts:              str          # timestamp ISO inicio general
    # ── Nodo 2: Lectura ───────────────────────────────────────────────────
    df_columnas:            List[str]
    total_filas:            int
    ts_fin_lectura:         str          # timestamp ISO fin lectura
    error_lectura:          Optional[str]
    # ── Nodo 3: Mapeo Híbrido (LLM + Reglas) ──────────────────────────────────────
    tabla_destino:          str
    cols_double:            List[str]
    cols_integer:           List[str]
    llm_response:           str
    llamadas_api:           int
    error_mapeo:            Optional[str]
    # ── Nodo 4: Transformación ────────────────────────────────────────────
    registros_validos:      int
    registros_duplicados:   int
    ts_fin_transformacion:  str          # timestamp ISO fin transformación
    error_transform:        Optional[str]
    temp_view_name:         str          # nombre único temp view Spark
    # ── Nodo 5: Carga ─────────────────────────────────────────────────────
    tabla_completa:         str
    ts_fin_carga:           str          # timestamp ISO fin carga
    error_carga:            Optional[str]
    # ── Nodo 6: Métricas generales ────────────────────────────────────────
    tiempo_segundos:        float
    dpmo:                   float
    calidad_pct:            float
    status:                 str
    error_final:            Optional[str]
    # ⚡ MÉTRICAS DE EXTRACCIÓN (pasadas desde el orquestador)
    reintentos_extraccion:  int          # reintentos HTTP de descarga
    recuperacion_extraccion: int         # 0=falló, 1=recuperó, 2=sin error

# ── NODO 1: DETECCIÓN ─────────────────────────────────────────────────────
def nodo_deteccion(state: ETLState) -> dict:
    """
    Primer nodo del grafo. Identifica el organismo de origen del archivo
    y registra el timestamp de inicio para medir el tiempo total (M1).
    """
    print(f"\n{'='*55}")
    print(f"[NODO 1 — DETECCIÓN] {state['file_name']}")
    fuente = identificar_fuente(state["file_name"])
    print(f"  Fuente: {fuente}")
    return {"fuente": fuente, "inicio_ts": datetime.now().isoformat(), "llamadas_api": 0}

# ── NODO 2: LECTURA ───────────────────────────────────────────────────────
def nodo_lectura(state: ETLState) -> dict:
    """
    Lee el archivo físico del volumen y detecta automáticamente el header.
    Registra ts_fin_lectura para calcular la métrica de tiempo de lectura.
    """
    print(f"[NODO 2 — LECTURA]")
    try:
        df = leer_archivo(state["local_path"], state["file_name"])
        n_filas, n_cols = df.shape
        print(f"  {n_filas} filas × {n_cols} columnas")
        print(f"  Primeras columnas: {list(df.columns)[:5]}")
        if n_filas == 0:
            return {"error_lectura":"Archivo vacío","total_filas":0,"df_columnas":[],
                    "ts_fin_lectura": datetime.now().isoformat()}
        return {"df_columnas":list(df.columns), "total_filas":n_filas, "error_lectura":None,
                "ts_fin_lectura": datetime.now().isoformat()}
    except Exception as e:
        print(f"  ❌ {e}")
        return {"error_lectura":str(e),"total_filas":0,"df_columnas":[],
                "ts_fin_lectura": datetime.now().isoformat()}

# ── NODO 3: MAPEO SEMÁNTICO CON LLM + FALLBACK ──────────────────────────
def _mapeo_basado_en_reglas(file_name: str, columnas: List[str]) -> dict:
    """
    Fallback basado en reglas cuando el LLM falla.
    Usa el nombre del archivo y columnas para determinar la tabla destino.
    """
    fn = file_name.upper()
    cols_text = " ".join(columnas).lower()
    
    # Reglas por nombre de archivo
    # PRIORIDAD MÁXIMA: ESPAC Tabulados y Series históricas → TABLA UNIFICADA banano/plátano
    # (debe estar ANTES de otras reglas para evitar falsos positivos)
    if "TABULADOS_EXCEL" in fn or "TABULADOS" in fn or "SERIES_HISTORICAS" in fn or "SERIES_HIST" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    # ESPAC T13 (banano) y T26 (plátano) → TABLA UNIFICADA
    if "T13" in fn or "T26" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    if "USO_DEL_SUELO" in fn or "uso" in cols_text and "suelo" in cols_text:
        return {"tabla":"espac_uso_del_suelo", "conf":0.85, "double":["superficie_ha"], "int":["ano"]}
    if "TEMPERATURA" in fn or ("temperatura" in cols_text and "precipitacion" in cols_text):
        return {"tabla":"sipa_temperatura_precipitacion", "conf":0.95, "double":["precipitacion_mm","temperatura_promedio_c"], "int":["ano"]}
    
    # FAOSTAT bananas y plantains → TABLA UNIFICADA
    if "FAOSTAT" in fn and ("BANANAS" in fn or "PLANTAINS" in fn):
        return {"tabla":"faostat_produccion_banano_platano", "conf":0.95, "double":["value"], "int":["year","area_code","item_code"]}
    
    if "AEBE" in fn or "BANANOTAS" in fn:
        return {"tabla":"aebe_exportaciones_regiones", "conf":0.95, "double":["cantidad"], "int":["anio"]}
    
    if "BANANO_PREC" in fn or "precios" in fn.lower():
        return {"tabla":"banano_precios_semanales", "conf":0.85, "double":["22xu","2080"], "int":["cont"]}
    
    # Fallback genérico
    return {"tabla":"tabla_temporal", "conf":0.3, "double":[], "int":[]}

def nodo_mapeo(state: ETLState) -> dict:
    """
    Mapeo híbrido: intenta LLM primero, fallback a reglas si falla.
    Cada llamada LLM incrementa el contador M5.
    """
    print(f"[NODO 3 — MAPEO HÍBRIDO] columnas: {state['df_columnas'][:4]}...")
    # Estrategia 1: Intentar LLM (solo 1 intento para ser más rápido)
    llamadas = state.get("llamadas_api", 0)
    try:
        print(f"  LLM — intento 1/1...")
        
        # Prompt simplificado para mejor respuesta
        prompt_simple = f"""Tabla destino para: {state['file_name']}
Columnas: {', '.join(state['df_columnas'][:8])}

Opciones:
- espac_banano_platano_provincia (T13 o T26, banano/plátano por provincia UNIFICADO)
- espac_uso_del_suelo (uso del suelo)
- espac_cultivos_permanentes (tabulados generales)
- sipa_temperatura_precipitacion (clima)
- faostat_produccion_banano_platano (FAO bananas/plantains UNIFICADO)
- banano_precios_semanales (precios)

Respuesta JSON:
{{"tabla_destino":"","confianza":0.0,"columnas_double":[],"columnas_integer":[]}}"""
        
        resp = llm.invoke([HumanMessage(content=prompt_simple)])
        llamadas += 1
        
        if resp.content and resp.content.strip():
            texto = re.sub(r"^```json\s*|^```\s*|```$","",resp.content.strip(),flags=re.MULTILINE).strip()
            start = texto.find("{")
            end = texto.rfind("}")
            if start != -1 and end != -1:
                texto = texto[start:end+1]
                mapeo = json.loads(texto)
                tabla = mapeo.get("tabla_destino","").lower().replace(" ","_")
                if tabla and tabla != "tabla_temporal":
                    print(f"  ✅ LLM: '{tabla}' (confianza: {mapeo.get('confianza',0)})")
                    return {"tabla_destino":tabla, "cols_double":mapeo.get("columnas_double",[]),
                            "cols_integer":mapeo.get("columnas_integer",[]),
                            "llm_response":json.dumps(mapeo,ensure_ascii=False),
                            "llamadas_api":llamadas, "error_mapeo":None}
    except Exception as e:
        print(f"  ⚠️ LLM falló: {str(e)[:50]}...")
    
    # Estrategia 2: Fallback basado en reglas
    print(f"  🔧 Usando mapeo basado en reglas...")
    fallback = _mapeo_basado_en_reglas(state["file_name"], state["df_columnas"])
    print(f"  ✅ Regla: '{fallback['tabla']}' (confianza: {fallback['conf']})")
    
    return {"tabla_destino":fallback["tabla"], 
            "cols_double":fallback["double"],
            "cols_integer":fallback["int"],
            "llm_response":json.dumps({"metodo":"fallback_reglas","confianza":fallback["conf"]},ensure_ascii=False),
            "llamadas_api":llamadas, 
            "error_mapeo":None}

# ── FUNCIÓN AUXILIAR: TRANSFORMACIÓN ESPECIALIZADA T13/T26 ───────────────
def transformar_espac_t13_t26(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma archivos T13 (banano) y T26 (plátano) de ESPAC al formato estandarizado.
    Pasos: Elimina sub-headers, renombra columnas, filtra provincias y notas, convierte numéricos, extrae año.
    """
    import numpy as np
    print(f"  📋 Transformación específica T13/T26 para: {nombre_archivo}")
    
    # Extraer año del nombre del archivo con regex flexible
    # Captura: _2021_, 2021.csv, ESPAC2021, T13_2022, etc.
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    df = df_pandas.iloc[1:].copy()
    # ⚡ ESCENARIO B: Detección robusta de columna provincia
    cols_orig = list(df.columns)
    col_provincia = next((c for c in cols_orig if 'PROVINCIA' in str(c).upper()), None)
    if col_provincia is None:
        col_provincia = cols_orig[0]
        print(f"  ⚠️ [ESCENARIO B] Columna PROVINCIA no encontrada, usando '{col_provincia}' como fallback (degradación)")
    expected_cols = ['PROVINCIA','CATEGORIA','SUPERFICIE_PLANTADA_HA','SUPERFICIE_COSECHADA_HA','PRODUCCION_TM','VENTAS_TM']
    rename_map = {cols_orig[idx]: name for idx, name in enumerate(expected_cols) if idx < len(cols_orig)}
    df = df.rename(columns=rename_map)
    for col_needed in expected_cols:
        if col_needed not in df.columns:
            df[col_needed] = None
    
    # Filtrar filas que NO son datos reales (provincias)
    palabras_excluir = ['TOTAL', 'REGIÓN', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN', 'OBSERVACION', 
                        'La tabla', 'INSTITUT', 'INEC', 'HTTP', 'WWW', 'ENCUESTA']
    
    df = df[
        df['PROVINCIA'].notna() &
        (~df['PROVINCIA'].astype(str).str.upper().str.contains('|'.join(palabras_excluir), na=False, case=False, regex=True))
    ].copy()
    
    print(f"    🗑️  Filtradas filas con notas/totales/fuentes")
    print(f"    → {len(df)} provincias válidas después del filtrado")
    
    # Convertir columnas numéricas
    for col in ['SUPERFICIE_PLANTADA_HA', 'SUPERFICIE_COSECHADA_HA', 'PRODUCCION_TM', 'VENTAS_TM']:
        df[col] = df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Agregar columna de año
    df['ANIO'] = anio if anio else 0
    
    # Agregar columnas calculadas
    df['PRODUCTO'] = 'BANANO' if 'T13' in nombre_archivo.upper() else 'PLATANO' if 'T26' in nombre_archivo.upper() else 'DESCONOCIDO'
    df['RENDIMIENTO'] = np.where(df['SUPERFICIE_COSECHADA_HA'] > 0, (df['PRODUCCION_TM'] / df['SUPERFICIE_COSECHADA_HA']).round(2), 0)
    
    # Limpiar strings
    df['PROVINCIA'] = df['PROVINCIA'].astype(str).str.strip()
    df['CATEGORIA'] = df['CATEGORIA'].fillna('Solo').astype(str).str.strip()
    
    df.reset_index(drop=True, inplace=True)
    print(f"    ✅ Transformación completada: {len(df)} registros, {len(df.columns)} columnas (incluye ANIO)")
    return df

# ── NODO 4: TRANSFORMACIÓN ────────────────────────────────────────────────
def nodo_transformacion(state: ETLState) -> dict:
    """
    Transformaciones de calidad de dato en Spark:
    1. Casteo numérico con guía del LLM
    2. Limpieza de strings nulos/vacíos/sin sentido
    3. Filtro banano/plátano si hay columna de producto
    4. Deduplicación — base para calcular DPMO y M4
    Registra ts_fin_transformacion y escribe en LOG_TRANSFORM.
    """
    print(f"[NODO 4 — TRANSFORMACIÓN]")
    ts_ini_t = datetime.now()
    try:
        nombre_upper = state["file_name"].upper()
        
        # ── CORRECCIÓN #6: TRANSFORMACIÓN PRECIOS ────────────────────────
        if "PRECIOS" in nombre_upper or "BANANO_PREC" in nombre_upper:
            print(f"  🎯 Detectado archivo de Precios — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])  # Leer primero
            df_pandas = transformar_precios(df_pandas, state["file_name"])     # Transformar estructura alternada
        
        # ── CORRECCIÓN #7: TRANSFORMACIÓN ESPAC TABULADOS/SERIES HISTÓRICAS ──
        elif "TABULADOS_EXCEL" in nombre_upper or "SERIES_HISTORICAS" in nombre_upper or "SERIES_HIST" in nombre_upper:
            print(f"  🎯 Detectado ESPAC Tabulados/Series Históricas — Aplicando transformación de múltiples hojas")
            df_pandas = transformar_espac_t13_t26_mejorado(state["local_path"], state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        # ── TRANSFORMACIÓN ESPECÍFICA PARA T13/T26 (archivos individuales) ──
        elif "T13" in nombre_upper or "T26" in nombre_upper:
            print(f"  🎯 Detectado archivo T13/T26 — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_espac_t13_t26(df_pandas, state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        # ── TRANSFORMACIÓN FAOSTAT ──────────────────────────────────────
        elif "FAOSTAT" in nombre_upper:
            print(f"  🎯 Detectado archivo FAOSTAT — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_faostat(df_pandas, state["file_name"])
        
        # ── TRANSFORMACIÓN SIPA TEMPERATURA ──────────────────────────────
        elif "TEMPERATURA" in nombre_upper or ("SIPA" in nombre_upper and "USO" not in nombre_upper):
            print(f"  🎯 Detectado SIPA Temperatura — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_sipa_temperatura(df_pandas, state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        # ── TRANSFORMACIÓN USO DEL SUELO ──────────────────────────────────
        elif "USO_DEL_SUELO" in nombre_upper or "USO" in nombre_upper:
            print(f"  🎯 Detectado Uso del Suelo — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_uso_del_suelo(df_pandas, state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL (totales regionales)
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id (totales regionales)")
        
        # ── TRANSFORMACIÓN ESTÁNDAR ──────────────────────────────────────
        else:
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
        
        df_spark  = spark.createDataFrame(df_pandas)
        df_spark  = castear_columnas(df_spark, state["cols_double"], state["cols_integer"])
        df_spark  = (df_spark
            .withColumn("pipeline_source_file", F.lit(state["file_name"]))
            .withColumn("pipeline_framework",   F.lit("LangGraph"))
            .withColumn("pipeline_load_ts",     F.current_timestamp()))

        # CORRECCIÓN SCPAP001 + SCPAP004: Cachear columnas antes del loop y usar transformación batch
        cols_to_clean = df_spark.columns  # Cachear lista de columnas UNA VEZ
        clean_exprs = {}
        for c in cols_to_clean:
            clean_exprs[c] = when(
                trim(col(c).cast("string")).isin("","null","NULL","None","nan","NaN",".","-","0"),
                None).otherwise(trim(col(c).cast("string")))
        df_spark = df_spark.withColumns(clean_exprs)  # Aplicar TODAS las transformaciones de una vez
        df_spark = df_spark.na.drop(how="all")

        # ── EXTRACCIÓN DE AÑO (para todas las tablas) ────────────────────────
        # Extraer año del nombre del archivo si no existe columna de año
        cols_anio = [c for c in df_spark.columns if c.lower() in ['anio', 'ano', 'year', 'fecha']]
        if not cols_anio:
            anio_match = re.search(r'(\d{4})', state["file_name"])
            if anio_match:
                anio_extraido = int(anio_match.group(1))
                df_spark = df_spark.withColumn("anio", F.lit(anio_extraido))
                print(f"  📅 Año extraído del nombre: {anio_extraido}")
        
        # ── LIMPIEZA DE COLUMNAS VACÍAS (>95% nulos) ─────────────────────────
        # Aplicar solo a tablas que lo necesiten (precios semanales tiene muchas columnas vacías)
        if "precios" in state.get("tabla_destino", "").lower():
            total_rows = df_spark.count()
            if total_rows > 0:
                # Columnas de metadatos a preservar
                meta_cols_preserve = {'_input_file_name', '_rescued_data', '_metadata', 'anio', 'pipeline_source_file', 'pipeline_load_ts', 'pipeline_framework'}
                cols_to_drop = []
                for col_name in df_spark.columns:
                    if col_name not in meta_cols_preserve:
                        null_count = df_spark.filter(col(col_name).isNull() | (trim(col(col_name)) == "")).count()
                        null_pct = (null_count / total_rows) * 100
                        if null_pct > 95:
                            cols_to_drop.append(col_name)
                
                if cols_to_drop:
                    df_spark = df_spark.drop(*cols_to_drop)
                    print(f"  🗑️  Eliminadas {len(cols_to_drop)} columnas vacías (>95% nulos)")
        
        # ── NO APLICAR FILTRO DE BANANO/PLÁTANO ──────────────────────────────
        # Los archivos ya vienen filtrados por fuente (ESPAC solo descarga archivos de banano)
        # El filtro anterior estaba eliminando TODOS los registros porque buscaba
        # columnas que no existían. DESHABILITADO.
        print(f"  ✓ Filtro de producto deshabilitado (archivos pre-filtrados por fuente)")

        meta_cols = {"pipeline_source_file","pipeline_load_ts","pipeline_framework"}
        # CORRECCIÓN SCPAP001: Cachear columnas antes del list comprehension
        all_columns = df_spark.columns
        real_cols = [c for c in all_columns if c not in meta_cols]
        
        # CORRECCIÓN SCPAP005: Materializar con count() (NO usar .cache() en Serverless)
        antes = df_spark.count()
        df_spark = df_spark.dropDuplicates(subset=real_cols)
        despues = df_spark.count()
        duplicados = antes - despues
        print(f"  Registros: {antes} → {despues} (−{duplicados} duplicados)")

        # CORRECCIÓN SCPAP003: Usar nombre único para temp view (evitar sobrescritura silenciosa)
        temp_view_name = f"langgraph_etl_temp_{state['file_name'].replace('.','_').replace('-','_')}"
        df_spark.createOrReplaceTempView(temp_view_name)

        ts_fin_t = datetime.now()
        tiempo_t = (ts_fin_t - ts_ini_t).total_seconds()

        # ── Métricas de Transformación ──────────────────────────────────
        n_cols   = max(len(state["df_columnas"]),1)
        total    = max(state["total_filas"],1)
        oport_t  = total * n_cols
        defect_t = (total - despues) + duplicados
        dpmo_t   = round(defect_t / oport_t * 1_000_000, 2)
        calidad_t= round(despues / total * 100, 2)

        schema_t = StructType([
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("tabla_destino",     StringType(),  True),
            StructField("filas_entrada",     IntegerType(), True),
            StructField("filas_salida",      IntegerType(), True),
            StructField("duplicados_elim",   IntegerType(), True),
            StructField("calidad_pct",       DoubleType(),  True),
            StructField("dpmo_transformacion",DoubleType(), True),
            StructField("tiempo_segundos",   DoubleType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
        ])
        spark.createDataFrame([(
            state["file_name"], state["fuente"], state.get("tabla_destino","?"),
            total, despues, duplicados, calidad_t, dpmo_t, tiempo_t, ts_ini_t, ts_fin_t
        )], schema_t).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TRANSFORM)

        print(f"  M4={calidad_t:.1f}%  DPMO_T={dpmo_t:.1f}  Tiempo={tiempo_t:.1f}s  → log guardado")
        return {"registros_validos":despues, "registros_duplicados":duplicados,
                "ts_fin_transformacion": ts_fin_t.isoformat(), "error_transform":None,
                "temp_view_name": temp_view_name}  # Pasar nombre de vista temporal al siguiente nodo

    except Exception as e:
        print(f"  ❌ {e}")
        return {"registros_validos":0,"registros_duplicados":0,
                "ts_fin_transformacion": datetime.now().isoformat(), "error_transform":str(e)}

# ── NODO 5: CARGA EN DELTA LAKE ───────────────────────────────────────────
def nodo_carga(state: ETLState) -> dict:
    """
    Persiste el DataFrame en Delta Lake con mergeSchema=true.
    
    CORRECCIÓN #20: Estrategia diferenciada por tipo de tabla:
    - MÉTRICAS/LOGS: Siempre APPEND (acumular histórico)
    - DATOS: Si framework diferente → OVERWRITE (reemplazar todo)
             Si mismo framework → APPEND (agregar solo nuevos)
    
    Registra las métricas de carga (tiempo de escritura, registros insertados)
    en la tabla LOG_CARGA para análisis independiente de esta fase.
    """
    tabla_completa = f"{DB_NAME}.{state['tabla_destino']}"
    print(f"[NODO 5 — CARGA] → {tabla_completa}")
    ts_ini_c = datetime.now()
    try:
        # Usar el nombre único de temp view del nodo anterior
        temp_view_name = state.get("temp_view_name", "langgraph_etl_temp")
        df_spark = spark.table(temp_view_name)
        
        # CORRECCIÓN SCPAP005: Ejecutar count() dentro del try para materializar
        registros_a_insertar = df_spark.count()

        # ⭐ CORRECCIÓN #20: Determinar modo de escritura según tipo de tabla
        es_tabla_metricas = any(keyword in tabla_completa.lower() for keyword in 
                                ['control_logs', 'metricas_', 'log_'])
        
        modo_escritura = "append"  # Por defecto
        
        if not es_tabla_metricas:
            # Es tabla de DATOS → verificar si hay framework diferente
            if spark.catalog.tableExists(tabla_completa):
                try:
                    # Verificar si existe la columna pipeline_framework
                    df_existente = spark.table(tabla_completa)
                    
                    if 'pipeline_framework' in df_existente.columns:
                        frameworks_existentes = df_existente.select('pipeline_framework').distinct().collect()
                        frameworks_set = {r['pipeline_framework'] for r in frameworks_existentes if r['pipeline_framework']}
                        
                        if frameworks_set and FRAMEWORK_NAME not in frameworks_set:
                            # Hay datos de otro framework → OVERWRITE
                            modo_escritura = "overwrite"
                            print(f"  🔄 Framework diferente detectado ({frameworks_set}) → OVERWRITE")
                        else:
                            # Mismo framework → APPEND
                            print(f"  ➕ Mismo framework ({FRAMEWORK_NAME}) → APPEND")
                    else:
                        # No tiene columna framework (tabla antigua) → APPEND por seguridad
                        print(f"  ℹ️  Tabla sin columna framework → APPEND")
                except Exception as e:
                    print(f"  ⚠️  No se pudo verificar framework: {e} → usando APPEND")
            else:
                print(f"  🆕 Tabla nueva → CREATE")
        else:
            print(f"  📊 Tabla de métricas → APPEND (acumular histórico)")

        # Si la tabla destino existe y es APPEND, castear columnas para coincidir con el schema
        if spark.catalog.tableExists(tabla_completa) and modo_escritura == "append":
            schema_dest = spark.table(tabla_completa).schema
            # CORRECCIÓN SCPAP001 + SCPAP004: Cachear columnas y schema, usar batch cast
            df_cols = df_spark.columns
            cast_exprs = {}
            for campo in schema_dest.fields:
                if campo.name in df_cols:
                    cast_exprs[campo.name] = col(campo.name).cast(campo.dataType)
            if cast_exprs:  # Solo aplicar si hay columnas que castear
                df_spark = df_spark.withColumns(cast_exprs)

        # CORRECCIÓN SCPAP005: write.saveAsTable() es una action, materializa inmediatamente
        df_spark.write.format("delta").mode(modo_escritura).option("mergeSchema","true").saveAsTable(tabla_completa)
        ts_fin_c  = datetime.now()
        tiempo_c  = (ts_fin_c - ts_ini_c).total_seconds()

        print(f"  ✅ {registros_a_insertar} registros en {tabla_completa} ({tiempo_c:.1f}s)")

        # ── Métricas de Carga ────────────────────────────────────────────
        schema_c = StructType([
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("tabla_destino",     StringType(),  True),
            StructField("registros_insertados",IntegerType(),True),
            StructField("tiempo_escritura_s",DoubleType(),  True),
            StructField("status",            StringType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
        ])
        spark.createDataFrame([(
            state["file_name"], state["fuente"], tabla_completa,
            registros_a_insertar, tiempo_c, "OK", ts_ini_c, ts_fin_c
        )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)

        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":None}

    except Exception as e:
        ts_fin_c = datetime.now()
        print(f"  ❌ {e}")
        schema_c = StructType([
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("tabla_destino",     StringType(),  True),
            StructField("registros_insertados",IntegerType(),True),
            StructField("tiempo_escritura_s",DoubleType(),  True),
            StructField("status",            StringType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
        ])
        spark.createDataFrame([(
            state["file_name"], state["fuente"], tabla_completa,
            0, 0.0, f"ERROR: {str(e)[:100]}", ts_ini_c, ts_fin_c
        )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)
        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":str(e)}

# ── NODO 6: MÉTRICAS GENERALES M1–M5 ────────────────────────────────────
def nodo_metricas(state: ETLState) -> dict:
    """
    Calcula las 6 métricas del experimento para el proceso ETL completo
    y las graba en la tabla de logs general (LOG_TABLE).

    M1 Tiempo de ejecución    → segundos inicio→fin completo
    M2 Intervención manual    → siempre 0 (LangGraph es 100% autónomo)
    M3 Recuperación errores   → 2=sin reintentos, 1=hubo reintentos, 0=falló
    M4 Calidad del dato       → % registros válidos / total leído
    M5 Llamadas API LLM       → número de peticiones al LLM
    M6 DPMO                   → defectos por millón de oportunidades ()
    """
    print(f"[NODO 6 — MÉTRICAS GENERALES]")
    fin    = datetime.now()
    inicio = datetime.fromisoformat(state["inicio_ts"])
    tiempo = (fin - inicio).total_seconds()

    total   = max(state["total_filas"],1)
    validos = state["registros_validos"]
    calidad = round(validos / total * 100, 2)

    n_cols   = max(len(state["df_columnas"]),1)
    oport    = total * n_cols
    defectos = (total - validos) + state["registros_duplicados"]
    dpmo     = round(defectos / oport * 1_000_000, 2)

    hay_error = any([state.get("error_lectura"),state.get("error_mapeo"),
                     state.get("error_transform"),state.get("error_carga")])
    status = "ERROR" if hay_error else "PROCESADO"

    # ⚡ M3 basado en recuperación REAL de descarga (no en llamadas LLM)
    reintentos_reales = state.get("reintentos_extraccion", 0)
    recuperacion_real = state.get("recuperacion_extraccion", 2)
    if hay_error and recuperacion_real == 0:
        m3 = 0
    elif reintentos_reales > 0 and not hay_error:
        m3 = 1
    else:
        m3 = 2
    # ⚡ intervencion_manual = 1 cuando NO se recuperó
    intervencion = 1 if recuperacion_real == 0 else 0

    print(f"  ┌──────────────────────────────────────────────")
    print(f"  │ M1 Tiempo total       : {tiempo:.1f}s")
    print(f"  │ M2 Intervención manu. : 0 (autónomo)")
    print(f"  │ M3 Recup. errores     : {m3}/2")
    print(f"  │ M4 Calidad del dato   : {calidad:.1f}%")
    print(f"  │ M5 Llamadas API LLM   : {state.get('llamadas_api',0)}")
    print(f"  │ M6 DPMO               : {dpmo:.1f}")
    print(f"  │ Status                : {status}")
    print(f"  └──────────────────────────────────────────────")

    error_msg = " | ".join(filter(None,[
        state.get("error_lectura"),state.get("error_mapeo"),
        state.get("error_transform"),state.get("error_carga"),
    ])) or None

    log_schema = StructType([
        StructField("execution_id",            StringType(),  True),
        StructField("file_name",               StringType(),  True),
        StructField("framework",               StringType(),  True),
        StructField("fuente",                  StringType(),  True),
        StructField("tabla_destino",           StringType(),  True),
        StructField("execution_timestamp",     TimestampType(),True),
        StructField("status",                  StringType(),  True),
        StructField("m1_tiempo_segundos",      DoubleType(),  True),
        StructField("m2_intervencion_manual",  IntegerType(), True),
        StructField("m3_recuperacion_errores", IntegerType(), True),
        StructField("m4_calidad_pct",          DoubleType(),  True),
        StructField("m5_llamadas_api",         IntegerType(), True),
        StructField("m6_dpmo",                 DoubleType(),  True),
        StructField("total_filas",             IntegerType(), True),
        StructField("registros_validos",       IntegerType(), True),
        StructField("registros_duplicados",    IntegerType(), True),
        StructField("llm_response",            StringType(),  True),
        StructField("error_message",           StringType(),  True),
        # ⚡ MÉTRICAS EXPERIMENTALES
        StructField("escenario",               StringType(),  True),
        StructField("reintentos_realizados",   IntegerType(), True),
        StructField("recuperacion",            IntegerType(), True),
        StructField("n_columnas",              IntegerType(), True),
    ])
    spark.createDataFrame([(
        EXECUTION_ID,
        state["file_name"],FRAMEWORK_NAME,state["fuente"],
        state.get("tabla_completa","N/A"),fin,status,
        tiempo,intervencion,m3,calidad,state.get("llamadas_api",0),dpmo,
        state["total_filas"],validos,state["registros_duplicados"],
        state.get("llm_response","{}"),error_msg,
        # ⚡ MÉTRICAS EXPERIMENTALES
        ESCENARIO, reintentos_reales, recuperacion_real, n_cols,
    )], log_schema).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TABLE)

    print(f"  ✅ Métricas generales registradas en {LOG_TABLE}")
    return {"tiempo_segundos":tiempo,"calidad_pct":calidad,"dpmo":dpmo,
            "status":status,"error_final":error_msg}

# ── NODO ERROR ────────────────────────────────────────────────────────────
def nodo_error(state: ETLState) -> dict:
    """
    Nodo de manejo centralizado de errores.
    LangGraph enruta aquí automáticamente cuando cualquier nodo falla.
    Esto es lo que mide M3 (Recuperación ante errores).
    """
    print(f"[NODO ERROR] — {state.get('file_name','?')}")
    errores = " | ".join(filter(None,[
        state.get("error_lectura"),state.get("error_mapeo"),
        state.get("error_transform"),state.get("error_carga"),
    ]))
    print(f"  Causa: {errores or 'desconocida'}")
    return {"status":"ERROR","error_final":errores or "Error desconocido"}

print("✅ Estado ETLState y 7 nodos del grafo definidos.")

In [ ]:
# ── CORRECCIÓN: MAPEO CON VALIDACIÓN DE EXTENSIONES ──────────────────────
def nodo_mapeo_corregido(state: ETLState) -> dict:
    """
    Mapeo híbrido corregido:
    - Escenario A/C: intenta LLM primero, fallback a reglas si falla.
      Para AEBE con nombre original siempre usa reglas.
    - Escenario B: si el archivo no calza con patrones conocidos (renombrado),
      FUERZA el LLM para medir M5. Si LLM falla, registra tabla_temporal
      como degradación (no crash).
    - Siempre elimina extensiones si el LLM las incluye.
    """
    columnas = state.get('df_columnas') or state.get('columnas_raw', [])
    print(f"[NODO 3 — MAPEO HÍBRIDO CORREGIDO] columnas: {columnas[:4]}... [Escenario {ESCENARIO}]")
    llamadas = state.get("llamadas_llm", 0) or state.get("llamadas_api", 0)
    fn_upper = state["file_name"].upper()

    # ⚡ ESCENARIO B: verificar si el nombre calza con patrones conocidos
    reglas_calzan = False
    if ESCENARIO != "B":
        reglas_calzan = True
    else:
        patrones_conocidos = [
            "TABULADOS_EXCEL","TABULADOS","SERIES_HISTORICAS","SERIES_HIST",
            "T13","T26","USO_DEL_SUELO","TEMPERATURA","FAOSTAT",
            "AEBE_BANANOTAS",  # patrón ORIGINAL — si fue renombrado no aparecerá
        ]
        reglas_calzan = any(p in fn_upper for p in patrones_conocidos)
        if not reglas_calzan:
            print(f"  🔬 [ESCENARIO B] Archivo no calza con patrones → LLM obligatorio")

    # En Escenario A/C, AEBE con nombre original sigue usando reglas
    if reglas_calzan and ("AEBE" in fn_upper or "BANANOTAS" in fn_upper):
        fallback = _mapeo_basado_en_reglas(state["file_name"], columnas)
        tabla_aebe = fallback["tabla"]
        print(f"  🛡️  Archivo AEBE — mapeo forzado por reglas: '{tabla_aebe}'")
        return {"tabla_destino":fallback["tabla"],
                "cols_double":fallback["double"],
                "cols_integer":fallback["int"],
                "llm_response":json.dumps({"metodo":"regla_forzada_aebe","confianza":fallback["conf"]},ensure_ascii=False),
                "llamadas_api":llamadas, "error_mapeo":None}

    try:
        print(f"  LLM — intento 1/1...")
        
        # Prompt mejorado con AEBE incluido
        prompt_mejorado = f"""Identifica la tabla destino para el archivo: {state['file_name']}
Columnas encontradas: {', '.join(columnas[:8])}

Tablas válidas (elige EXACTAMENTE una):
- espac_banano_platano_provincia (T13/T26, Tabulados, Series Históricas - banano/plátano por provincia UNIFICADO)
- espac_uso_del_suelo (uso del suelo)
- sipa_temperatura_precipitacion (clima)
- faostat_produccion_banano_platano (FAO bananas/plantains UNIFICADO)
- aebe_exportaciones_regiones (AEBE Bananotas - rankings de exportaciones por región/país)
- banano_precios_semanales (precios)

⚠️ SI el archivo tiene "Tabulados" o "Series" en el nombre → SIEMPRE usar espac_banano_platano_provincia
⚠️ SI el archivo tiene "AEBE" o "BANANOTAS" en el nombre → SIEMPRE usar aebe_exportaciones_regiones

⚠️ IMPORTANTE: La tabla_destino debe ser EXACTAMENTE uno de los nombres arriba.
⚠️ NO incluyas extensiones (.xlsx, .csv, .xls) en el nombre.

Responde SOLO con JSON válido (sin markdown):
{{"tabla_destino":"nombre_sin_extension","confianza":0.9,"columnas_double":[],"columnas_integer":[]}}"""
        
        resp = llm.invoke([HumanMessage(content=prompt_mejorado)])
        llamadas += 1
        
        if resp.content and resp.content.strip():
            texto = re.sub(r"^```json\s*|^```\s*|```$","",resp.content.strip(),flags=re.MULTILINE).strip()
            start = texto.find("{")
            end = texto.rfind("}")
            if start != -1 and end != -1:
                texto = texto[start:end+1]
                mapeo = json.loads(texto)
                tabla = mapeo.get("tabla_destino","").lower().replace(" ","_")
                
                # ✅ VALIDACIÓN: Eliminar extensiones si el LLM las incluyó
                for ext in [".xlsx", ".xls", ".csv", ".json"]:
                    if tabla.endswith(ext):
                        print(f"  ⚠️ LLM incluyó extensión '{ext}' — removiendo...")
                        tabla = tabla[:-len(ext)]
                
                if tabla and tabla != "tabla_temporal":
                    # 🛡️ VALIDACIÓN SEGURIDAD: Si es archivo AEBE, verificar que el LLM mapeó correctamente
                    if "AEBE" in state["file_name"].upper():
                        if tabla == "aebe_exportaciones_regiones":
                            print(f"  ✅ LLM correcto para AEBE: '{tabla}' (confianza: {mapeo.get('confianza',0)})")
                            return {"tabla_destino":tabla, "cols_double":mapeo.get("columnas_double",[]),
                                    "cols_integer":mapeo.get("columnas_integer",[]),
                                    "llm_response":json.dumps(mapeo,ensure_ascii=False),
                                    "llamadas_llm":llamadas, "llamadas_api":llamadas, "error_mapeo":None}
                        else:
                            print(f"  ⚠️ LLM INCORRECTO para AEBE: mapeó a '{tabla}' (debería ser 'aebe_exportaciones_regiones')")
                            print(f"  🔧 Usando reglas como fallback de seguridad...")
                            # Continuar al fallback de reglas (no return aquí)
                    else:
                        # Para archivos NO-AEBE, confiar en el LLM
                        print(f"  ✅ LLM: '{tabla}' (confianza: {mapeo.get('confianza',0)})")
                        return {"tabla_destino":tabla, "cols_double":mapeo.get("columnas_double",[]),
                                "cols_integer":mapeo.get("columnas_integer",[]),
                                "llm_response":json.dumps(mapeo,ensure_ascii=False),
                                "llamadas_llm":llamadas, "llamadas_api":llamadas, "error_mapeo":None}
    except Exception as e:
        print(f"  ⚠️ LLM falló: {str(e)[:50]}...")
    
    # Fallback a reglas
    print(f"  🔧 Usando mapeo basado en reglas...")
    fallback = _mapeo_basado_en_reglas(state["file_name"], columnas)
    print(f"  ✅ Regla: '{fallback['tabla']}' (confianza: {fallback['conf']})")
    
    # Distinguir entre fallback normal y fallback por validación AEBE
    es_fallback_aebe = "AEBE" in state["file_name"].upper() and llamadas > 0
    metodo_fallback = "validacion_aebe_fallback" if es_fallback_aebe else "fallback_reglas"
    
    return {"tabla_destino":fallback["tabla"], 
            "cols_double":fallback["double"],
            "cols_integer":fallback["int"],
            "llm_response":json.dumps({"metodo":metodo_fallback,"confianza":fallback["conf"],"llm_intento":llamadas>0},ensure_ascii=False),
            "llamadas_llm":llamadas, "llamadas_api":llamadas, 
            "error_mapeo":None}

# Reemplazar el nodo en la definición del grafo
nodo_mapeo = nodo_mapeo_corregido

print("✅ Nodo de mapeo corregido: ahora elimina extensiones automáticamente")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER: DECISIÓN KNN CON LLM
# ══════════════════════════════════════════════════════════════════════════

def decidir_aplicar_knn_con_llm(df_pandas: pd.DataFrame, nombre_archivo: str, fuente: str) -> dict:
    """
    Consulta al LLM para decidir si es apropiado aplicar imputación KNN.
    
    Returns:
        dict con:
        - aplicar_knn: bool (True/False)
        - razon: str (explicación del LLM)
        - columnas_excluir: list (columnas que NO deben imputarse)
    """
    # Analizar valores faltantes
    numeric_cols = df_pandas.select_dtypes(include=[np.number]).columns.tolist()
    missing_info = {}
    
    for col in numeric_cols[:10]:  # Máximo 10 columnas para no saturar el prompt
        missing_count = df_pandas[col].isnull().sum()
        if missing_count > 0:
            missing_pct = (missing_count / len(df_pandas)) * 100
            missing_info[col] = f"{missing_pct:.1f}%"
    
    if not missing_info:
        return {"aplicar_knn": False, "razon": "No hay valores faltantes", "columnas_excluir": []}
    
    # Prompt para el LLM
    prompt = f"""Analiza si es apropiado usar imputación KNN (vecinos cercanos) para este dataset:

Archivo: {nombre_archivo}
Fuente: {fuente}
Total registros: {len(df_pandas)}

Valores faltantes por columna:
{json.dumps(missing_info, indent=2, ensure_ascii=False)}

Contexto:
- KNN rellena valores numéricos faltantes usando los 5 vecinos más cercanos
- Es apropiado cuando los valores faltantes son ERRORES de captura o medición
- NO es apropiado cuando los valores NULL tienen significado estructural (ej: provincia NULL = agregación regional)

Decide:
1. ¿Aplicar KNN? (true/false)
2. ¿Por qué?
3. ¿Qué columnas EXCLUIR del KNN? (claves primarias, IDs, campos estructuralmente NULL)

Responde en JSON:
{{
  "aplicar_knn": true/false,
  "razon": "explicación breve",
  "columnas_excluir": ["provincia_id", "anio", ...]
}}"""
    
    try:
        print(f"    🤔 Consultando al LLM sobre uso de KNN...")
        resp = llm.invoke([HumanMessage(content=prompt)])
        
        # Extraer JSON de la respuesta
        texto = re.sub(r"^```json\s*|^```\s*|```$", "", resp.content.strip(), flags=re.MULTILINE).strip()
        start = texto.find("{")
        end = texto.rfind("}")
        
        if start != -1 and end != -1:
            texto = texto[start:end+1]
            decision = json.loads(texto)
            
            aplicar = decision.get("aplicar_knn", False)
            razon = decision.get("razon", "Sin razón")
            excluir = decision.get("columnas_excluir", [])
            
            # 🚫 FORZAR EXCLUSIÓN de columnas que NUNCA deben imputarse
            columnas_protegidas = ['provincia_id', 'region_id', 'anio', 'ano', 'year', 'mes', 'month', 'canton_id', 'pais_codigo', 'producto_codigo']
            excluir = list(set(excluir + columnas_protegidas))
            
            print(f"    🧠 LLM decisión: {'✅ Aplicar KNN' if aplicar else '❌ No aplicar KNN'}")
            print(f"    📝 Razón: {razon}")
            if excluir:
                print(f"    🚫 Excluir: {excluir}")
            
            return {"aplicar_knn": aplicar, "razon": razon, "columnas_excluir": excluir}
    
    except Exception as e:
        print(f"    ⚠️  Error consultando LLM: {str(e)[:50]}")
        print(f"    🔧 Fallback: No aplicar KNN por seguridad")
    
    # Fallback conservador: NO aplicar KNN si el LLM falla
    return {"aplicar_knn": False, "razon": "Error en LLM - decisión conservadora", "columnas_excluir": []}

print("✅ Función decidir_aplicar_knn_con_llm() cargada")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# NODO 6: MÉTRICAS CONSOLIDADAS (control_logs_etl)
# Recoge métricas de extracción + transformación + carga y calcula M1–M5
# ══════════════════════════════════════════════════════════════════════════

def nodo_metricas(state: ETLState) -> dict:
    """
    Nodo final que consolida todas las métricas en control_logs_etl.
    Calcula:
    - M1: Tiempo total (segundos)
    - M2: Intervención manual (0=automático, 1=manual)
    - M3: Recuperación de errores (0=no recup, 1=recup parcial, 2=recup completa)
    - M4: Calidad de datos (%)
    - M5: Llamadas LLM (#)
    - M6: DPMO (Defectos Por Millón de Oportunidades)
    """
    print(f"[NODO 6 — MÉTRICAS CONSOLIDADAS]")
    
    try:
        # Calcular tiempo total desde inicio hasta ahora
        ts_inicio = datetime.fromisoformat(state["inicio_ts"])
        ts_fin = datetime.now()
        m1_tiempo = (ts_fin - ts_inicio).total_seconds()
        
        # M2: Intervención manual (siempre 0 en este pipeline automatizado)
        m2_intervencion = 0
        
        # M3: Recuperación de errores
        # Recupera de métricas de extracción (pasadas como parte del estado)
        reintentos = state.get("reintentos_extraccion", 0)
        # CORRECCION P3: error_lectura solo cuenta si transformacion TAMBIEN fallo
        # (PDFs fallan en leer_archivo pero se procesan OK con pdfplumber)
        tiene_errores = (
            state.get("error_mapeo") or 
            state.get("error_transform") or 
            state.get("error_carga")
        )
        if state.get("error_lectura") and state.get("error_transform"):
            tiene_errores = True
        
        # M3: 2=sin errores, 1=reintentos exitosos, 0=fallo no recuperado
        if tiene_errores:
            m3_recuperacion = 0
        elif reintentos > 0:
            m3_recuperacion = 1
        else:
            m3_recuperacion = 2
        
        # M4: Calidad de datos (ya calculada en transformación)
        m4_calidad = state.get("calidad_pct", 0.0)
        
        # M5: Llamadas LLM (acumuladas en mapeo + otras consultas)
        m5_llamadas = state.get("llamadas_api", 0)
        
        # M6: DPMO (Defectos Por Millón de Oportunidades)
        # Calculado en base a: registros inválidos + duplicados + errores
        total_filas = max(state.get("total_filas", 1), 1)
        registros_validos = state.get("registros_validos", 0)
        registros_dup = state.get("registros_duplicados", 0)
        n_columnas = len(state.get("df_columnas", [])) or len(state.get("columnas_raw", []))
        n_columnas = max(n_columnas, 1)
        
        # Oportunidades = filas * columnas
        oportunidades = total_filas * n_columnas
        # Defectos = filas que no llegaron a Gold (ya incluye duplicados)
        defectos = total_filas - registros_validos
        m6_dpmo = round((defectos / oportunidades) * 1_000_000, 2) if oportunidades > 0 else 0.0
        
        # Determinar status final
        if tiene_errores:
            status_final = "ERROR"
        elif m4_calidad < 90 or m6_dpmo > 10000:
            status_final = "WARNING"
        else:
            status_final = "OK"
        
        # Reintentos ya extraidos arriba
        
        # Preparar mensaje de error si existe
        error_msg = None
        if tiene_errores:
            errores = [e for e in [
                state.get("error_lectura"),
                state.get("error_mapeo"),
                state.get("error_transform"),
                state.get("error_carga")
            ] if e]
            error_msg = " | ".join(errores)[:500] if errores else "Error desconocido"
        
        # Guardar en tabla consolidada
        schema_log = StructType([
            StructField("execution_id",          StringType(),    True),
            StructField("file_name",             StringType(),    True),
            StructField("framework",             StringType(),    True),
            StructField("fuente",                StringType(),    True),
            StructField("tabla_destino",         StringType(),    True),
            StructField("execution_timestamp",   TimestampType(), True),
            StructField("status",                StringType(),    True),
            StructField("m1_tiempo_segundos",    DoubleType(),    True),
            StructField("m2_intervencion_manual",IntegerType(),   True),
            StructField("m3_recuperacion_errores",IntegerType(),  True),
            StructField("m4_calidad_pct",        DoubleType(),    True),
            StructField("m5_llamadas_api",       IntegerType(),   True),
            StructField("m6_dpmo",               DoubleType(),    True),
            StructField("total_filas",           IntegerType(),   True),
            StructField("registros_validos",     IntegerType(),   True),
            StructField("registros_duplicados",  IntegerType(),   True),
            StructField("llm_response",          StringType(),    True),
            StructField("error_message",         StringType(),    True),
            StructField("escenario",             StringType(),    True),
            StructField("reintentos_realizados", IntegerType(),   True),
            StructField("recuperacion",          IntegerType(),   True),
            StructField("n_columnas",            IntegerType(),   True),
        ])
        
        df_log = spark.createDataFrame([(
            EXECUTION_ID,
            state["file_name"],
            FRAMEWORK_NAME,
            state["fuente"],
            state.get("tabla_destino", "?"),
            ts_fin,
            status_final,
            round(m1_tiempo, 2),
            m2_intervencion,
            m3_recuperacion,
            round(m4_calidad, 2),
            m5_llamadas,
            m6_dpmo,
            total_filas,
            registros_validos,
            registros_dup,
            state.get("llm_response", "")[:500],  # Truncar para no exceder límite
            error_msg,
            state.get("escenario", ESCENARIO),
            reintentos,
            state.get("recuperacion_extraccion", 2),
            n_columnas,
        )], schema_log)
        
        df_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(LOG_TABLE)
        
        print(f"  ✅ Métricas guardadas: M1={m1_tiempo:.1f}s | M4={m4_calidad:.1f}% | M5={m5_llamadas} | M6={m6_dpmo:.0f} | Status={status_final}")
        
        return {
            "tiempo_segundos": m1_tiempo,
            "dpmo": m6_dpmo,
            "calidad_pct": m4_calidad,
            "status": status_final,
            "error_final": error_msg
        }
        
    except Exception as e:
        print(f"  ❌ Error guardando métricas: {e}")
        import traceback
        traceback.print_exc()
        return {
            "tiempo_segundos": 0.0,
            "dpmo": 999999.0,
            "calidad_pct": 0.0,
            "status": "ERROR",
            "error_final": f"Error en nodo_metricas: {e}"
        }

print("✅ nodo_metricas() definido.")

# 🧪 EJECUCIÓN MODULAR POR ESCENARIO

⚠️ **IMPORTANTE**: Ya NO uses el orquestador completo. Ahora ejecuta **una celda por vez**:

---

## 📝 Secuencia de Ejecución

### 1️⃣ **Setup (ya ejecutado)**
✅ Configuración global y funciones base (Celdas 6-39)

Si alguna variable falta:
* Re-ejecuta **Celda 9** (Imports y Configuración Global)
* Re-ejecuta **Celda 10** (Funciones Escenarios)
* Re-ejecuta **Celda 11** (Infraestructura control)

---

### 2️⃣ **Ejecutar Escenarios (UNO A LA VEZ)**

#### 🅰️ Escenario A (Normal)
* Sin variaciones de esquema
* Sin fallos controlados
* Ejecuta: **Celda 🅰️ EJECUTOR: Escenario A**
* **5 corridas** (R01-R05)
* Toma ~10-20 minutos

#### 🅱️ Escenario B (Variaciones de Esquema)
* Renombra columnas, inserta filas vacías
* LLM debe compensar con más llamadas
* Ejecuta: **Celda 🅱️ EJECUTOR: Escenario B**
* **5 corridas** (R01-R05)
* Toma ~15-30 minutos (más llamadas LLM)

#### 🅲️ Escenario C (Fallos Controlados)
* Simula truncamiento de archivos
* Verifica lógica de reintentos
* Ejecuta: **Celda 🅲️ EJECUTOR: Escenario C**
* **5 corridas** (R01-R05)
* Toma ~15-35 minutos (incluye reintentos)

---

### 3️⃣ **Verificar Resultados**

✅ Ejecuta: **Celda 📋 Verificación SQL**

Compara métricas entre escenarios:
* **M3**: Reintentos (C > A, B)
* **M4**: Tiempo (B, C > A)
* **M5**: Llamadas LLM (B > A, C)
* **M6**: Calidad de datos
* **M8**: Recuperación (C = 1)

---

## ⚡ Ventajas de Este Enfoque

✅ **Sin errores de variables faltantes**: cada celda declara lo que necesita  
✅ **Fácil debugging**: si falla, solo re-ejecutas ESE escenario  
✅ **Experimento completo**: 5 corridas por escenario (15 total)  
✅ **Aislamiento total**: cada escenario resetea sus datos  
✅ **Comparación directa**: SQL unificado al final  

---

## 🔄 Si Algo Falla

1. Lee el error en la salida de la celda
2. Si dice `NameError: name 'DB_NAME'` → re-ejecuta Celda 9
3. Si dice `función no definida` → re-ejecuta Celdas 13-39
4. Si falla ETL → revisa logs en `bd_banano_ec.control_logs_etl`

---

🚀 **COMIENZA AHORA**: Ejecuta Celda 🅰️ primero

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EJECUTOR ESCENARIO A - NORMAL (1 corrida de prueba)
# Sin variaciones de esquema, sin fallos controlados
# ══════════════════════════════════════════════════════════════════════════

import asyncio
import nest_asyncio
from datetime import datetime
import time
nest_asyncio.apply()

print("="*70)
print("🅰️ ESCENARIO A — 10 CORRIDAS (flujo normal)")
print("="*70)

# Variables globales necesarias (por si acaso)
DB_NAME = "bd_banano_ec"
FRAMEWORK_NAME = "LlamaIndex"
LOG_TABLE = f"{DB_NAME}.control_logs_etl"
LOG_EXTRACCION = f"{DB_NAME}.metricas_extraccion"

RAW_PATH_ESPAC = "/Volumes/databricksbanano/default/bronce/espac"
RAW_PATH_SIPA = "/Volumes/databricksbanano/default/bronce/sipa"
RAW_PATH_FAOSTAT = "/Volumes/databricksbanano/default/bronce/faostat"
RAW_PATH_AEBE = "/Volumes/databricksbanano/default/bronce/aebe"

# Configurar escenario A
ESCENARIO = "A"
CORRIDAS = 10  # Número de corridas del experimento

print(f"\n📍 Escenario: {ESCENARIO} (normal, sin variaciones)")
print(f"   Corridas programadas: {CORRIDAS}")

# Reset de tablas de datos (preserva logs)
TABLAS_DATOS = [
    f"{DB_NAME}.espac_banano_platano_provincia",
    f"{DB_NAME}.faostat_produccion_banano_platano",
    f"{DB_NAME}.espac_uso_del_suelo",
    f"{DB_NAME}.sipa_temperatura_precipitacion",
    f"{DB_NAME}.aebe_exportaciones_regiones",
    f"{DB_NAME}.tabla_temporal",
]

print("\n🔄 Limpiando tablas de datos...")
for t in TABLAS_DATOS:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    except:
        pass

print("   Vaciando volúmenes...")
for v in [RAW_PATH_ESPAC, RAW_PATH_SIPA, RAW_PATH_FAOSTAT, RAW_PATH_AEBE]:
    try:
        for f in dbutils.fs.ls(v):
            dbutils.fs.rm(f.path)
    except:
        pass

# ⚡ CRÍTICO: Limpiar control de descargas para forzar re-descarga
print("   Limpiando control_descargas_fuentes...")
try:
    spark.sql(f"DELETE FROM {DB_NAME}.control_descargas_fuentes WHERE framework = '{FRAMEWORK_NAME}'")
except:
    pass

print("   ✅ Datos limpios, logs preservados, descargas resetadas")

# Función helper para ejecutar una corrida
async def ejecutar_escenario_a(corrida_num: int):
    global EXECUTION_ID, ESCENARIO
    ESCENARIO = "A"
    EXECUTION_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_EA_R{corrida_num:02d}"
    print(f"\n🔢 CORRIDA {corrida_num}/{CORRIDAS} - ID: {EXECUTION_ID}")
    print("\n" + "═"*70)
    print("FASE 1: EXTRACCIÓN")
    print("═"*70)
    
    fuentes = [
        {"fuente_nombre": "ESPAC",
         "fuente_url": "https://www.ecuadorencifras.gob.ec/encuesta-de-superficie-y-produccion-agropecuaria-continua-espac/",
         "volumen_destino": RAW_PATH_ESPAC,
         "keywords_relevantes": ["banano", "platano", "tabulados", "series", "T13", "T26"]},
        {"fuente_nombre": "SIPA",
         "fuente_url": "https://sipa.agricultura.gob.ec/estadisticas-agropecuarias",
         "volumen_destino": RAW_PATH_SIPA,
         "keywords_relevantes": ["temperatura", "precipitacion", "uso suelo"]},
        {"fuente_nombre": "FAOSTAT",
         "fuente_url": "https://faostatservices.fao.org/api/v1/en/data/QCL",
         "volumen_destino": RAW_PATH_FAOSTAT,
         "keywords_relevantes": ["bananas", "plantains", "production", "Ecuador"]},
        {"fuente_nombre": "AEBE_BANANOTAS",
         "fuente_url": "https://www.aebe.com.ec/bananotas",
         "volumen_destino": RAW_PATH_AEBE,
         "keywords_relevantes": ["exportaciones", "regiones", "estadisticas"]},
    ]
    
    for fuente_cfg in fuentes:
        try:
            wf = ExtractionWorkflow(timeout=600, verbose=False)
            await wf.run(**fuente_cfg)
            print(f"  ✅ {fuente_cfg['fuente_nombre']}")
        except Exception as e:
            print(f"  ❌ {fuente_cfg['fuente_nombre']}: {e}")
    
    print("\n" + "═"*70)
    print("FASE 2: ETL")
    print("═"*70)
    
    # Listar archivos descargados
    VOLUMENES = [
        (RAW_PATH_ESPAC, "ESPAC"),
        (RAW_PATH_SIPA, "SIPA"),
        (RAW_PATH_FAOSTAT, "FAOSTAT"),
        (RAW_PATH_AEBE, "AEBE"),
    ]
    
    archivos_pendientes = []
    for vol_path, vol_label in VOLUMENES:
        try:
            for nombre in sorted(os.listdir(vol_path)):
                if nombre.startswith('.'):
                    continue
                if not any(nombre.lower().endswith(ext) for ext in ['.xlsx', '.xls', '.csv', '.json', '.pdf']):
                    continue
                ruta = f"{vol_path}/{nombre}"
                archivos_pendientes.append({"name": nombre, "path": ruta, "vol": vol_label})
        except Exception as e:
            print(f"  ⚠️ Listado {vol_label}: {e}")
    
    print(f"\n📊 {len(archivos_pendientes)} archivos a procesar")
    
    # Mapeo de fuentes para recuperar métricas de extracción
    fuente_map = {"ESPAC": "ESPAC", "SIPA": "SIPA", "FAOSTAT": "FAOSTAT", "AEBE": "AEBE_BANANOTAS"}
    
    # Procesar cada archivo
    for idx, archivo in enumerate(archivos_pendientes, 1):
        print(f"\n[{idx}/{len(archivos_pendientes)}] {archivo['name']}")
        
        # Recuperar métricas de extracción
        reintentos_ext, recuperacion_ext = 0, 2
        if spark.catalog.tableExists(LOG_EXTRACCION):
            try:
                fn = fuente_map.get(archivo["vol"], archivo["vol"])
                df_e = spark.table(LOG_EXTRACCION).filter(
                    (col("execution_id") == EXECUTION_ID) &
                    (col("fuente") == fn) &
                    (col("status").isin(["OK", "PARCIAL", "ERROR"]))
                ).orderBy(col("timestamp_fin").desc()).limit(1)
                if df_e.count() > 0:
                    r = df_e.first()
                    reintentos_ext = r["reintentos_realizados"] if "reintentos_realizados" in r.__fields__ else 0
                    recuperacion_ext = r["recuperacion"] if "recuperacion" in r.__fields__ else 2
            except:
                pass
        
        # Estado ETL inicial
        estado_etl = {
            "file_name": archivo["name"],
            "local_path": archivo["path"],
            "fuente": "",
            "inicio_ts": datetime.now().isoformat(),
            "df_columnas": [],
            "total_filas": 0,
            "ts_fin_lectura": "",
            "error_lectura": None,
            "tabla_destino": "",
            "cols_double": [],
            "cols_integer": [],
            "llm_response": "{}",
            "llamadas_api": 0,
            "error_mapeo": None,
            "registros_validos": 0,
            "registros_duplicados": 0,
            "ts_fin_transformacion": "",
            "error_transform": None,
            "temp_view_name": "",
            "tabla_completa": "",
            "ts_fin_carga": "",
            "error_carga": None,
            "tiempo_segundos": 0.0,
            "dpmo": 0.0,
            "calidad_pct": 0.0,
            "status": "PENDIENTE",
            "error_final": None,
            "reintentos_extraccion": reintentos_ext,
            "recuperacion_extraccion": recuperacion_ext,
        }
        
        try:
            # Ejecutar nodos ETL secuencialmente
            estado = estado_etl.copy()
            estado = {**estado, **nodo_deteccion(estado)}
            estado = {**estado, **nodo_lectura(estado)}
            estado = {**estado, **nodo_mapeo_corregido(estado)}
            estado = {**estado, **nodo_transformacion_con_metricas(estado)}
            estado = {**estado, **nodo_carga(estado)}
            estado = {**estado, **nodo_metricas(estado)}
            print(f"  ✅ {estado['status']}")
        except Exception as e:
            print(f"  ❌ Error: {e}")

# Ejecutar 5 corridas en bucle
import time

for corrida in range(1, CORRIDAS + 1):
    print("\n" + "#"*70)
    print(f"# CORRIDA {corrida}/{CORRIDAS}")
    print("#"*70)
    
    try:
        await ejecutar_escenario_a(corrida)
        print(f"\n✅ Corrida {corrida} completada exitosamente")
    except Exception as e:
        print(f"\n❌ Corrida {corrida} falló: {e}")
        import traceback
        traceback.print_exc()
    
    # Pausa breve entre corridas
    if corrida < CORRIDAS:
        print(f"\n⏸️  Pausa 5s antes de corrida {corrida + 1}...")
        time.sleep(5)

print("\n" + "="*70)
print(f"✅ ESCENARIO A COMPLETADO: {CORRIDAS} corridas finalizadas")
print("="*70)
print("\nVerifica los resultados ejecutando la celda de verificación SQL")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EXPORTAR MÉTRICAS ESCENARIO A — LlamaIndex
# ══════════════════════════════════════════════════════════════════════════

import pandas as pd
from datetime import datetime

print("="*70)
print("📊 EXPORTACIÓN DE MÉTRICAS — Escenario A (LlamaIndex)")
print("="*70)

# 1. Consultar SOLO Escenario A
print("\n📥 Extrayendo métricas del Escenario A...")

df_escenario_a = spark.sql("""
SELECT
    execution_id,
    file_name,
    fuente,
    framework,
    escenario,
    tabla_destino,
    execution_timestamp,
    m1_tiempo_segundos,
    m2_intervencion_manual,
    m3_recuperacion_errores,
    m4_calidad_pct,
    m5_llamadas_api,
    m6_dpmo,
    total_filas,
    registros_validos,
    registros_duplicados,
    status,
    error_message
FROM bd_banano_ec.control_logs_etl
WHERE escenario = 'A'
  AND framework = 'LlamaIndex'
ORDER BY execution_timestamp
""").toPandas()

print(f"   ✅ {len(df_escenario_a)} registros del Escenario A")

# 2. Guardar como CSV (openpyxl no disponible en cluster terminado)
archivo_salida = "/Workspace/Users/respaldocorreoarch@gmail.com/Banano/metricas_escenarioA_LlamaIndex.xlsx"
df_escenario_a.to_excel(archivo_salida, index=False, engine='openpyxl')

print(f"\n💾 Archivo guardado: {archivo_salida}")

# 3. Estadísticas de resumen
print("\n📈 ESTADÍSTICAS ESCENARIO A:")
print(f"   Total registros procesados: {len(df_escenario_a)}")
print(f"   Tiempo promedio (M1): {df_escenario_a['m1_tiempo_segundos'].mean():.2f}s")
print(f"   Calidad promedio (M4): {df_escenario_a['m4_calidad_pct'].mean():.2f}%")
print(f"   DPMO promedio (M6): {df_escenario_a['m6_dpmo'].mean():.2f}")
print(f"   Llamadas LLM totales (M5): {df_escenario_a['m5_llamadas_api'].sum()}")
print(f"   Total filas procesadas: {df_escenario_a['total_filas'].sum()}")
print(f"   Registros válidos: {df_escenario_a['registros_validos'].sum()}")
print(f"   Status OK: {(df_escenario_a['status'] == 'OK').sum()}")
print(f"   Status WARNING: {(df_escenario_a['status'] == 'WARNING').sum()}")
print(f"   Status ERROR: {(df_escenario_a['status'] == 'ERROR').sum()}")

print("\n✅ Exportación del Escenario A completada.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EJECUTOR ESCENARIO B - VARIACIONES DE ESQUEMA (1 corrida de prueba)
# Aplica transformaciones que rompen el patrón esperado
# ══════════════════════════════════════════════════════════════════════════

import asyncio
import nest_asyncio
from datetime import datetime
import time
import os
from pyspark.sql.functions import col
nest_asyncio.apply()

print("="*70)
print("🅱️ ESCENARIO B — 10 CORRIDAS (variaciones de esquema)")
print("="*70)

# Variables globales
DB_NAME = "bd_banano_ec"
FRAMEWORK_NAME = "LlamaIndex"
LOG_TABLE = f"{DB_NAME}.control_logs_etl"
LOG_EXTRACCION = f"{DB_NAME}.metricas_extraccion"

RAW_PATH_ESPAC = "/Volumes/databricksbanano/default/bronce/espac"
RAW_PATH_SIPA = "/Volumes/databricksbanano/default/bronce/sipa"
RAW_PATH_FAOSTAT = "/Volumes/databricksbanano/default/bronce/faostat"
RAW_PATH_AEBE = "/Volumes/databricksbanano/default/bronce/aebe"

# Configurar escenario B
ESCENARIO = "B"  # ⚡ IMPORTANTE: esto activa aplicar_variacion_esquema()
CORRIDAS = 10

print(f"\n📍 Escenario: {ESCENARIO} (variaciones de esquema)")
print(f"   Corridas programadas: {CORRIDAS}")

# Reset
TABLAS_DATOS = [
    f"{DB_NAME}.espac_banano_platano_provincia",
    f"{DB_NAME}.faostat_produccion_banano_platano",
    f"{DB_NAME}.espac_uso_del_suelo",
    f"{DB_NAME}.sipa_temperatura_precipitacion",
    f"{DB_NAME}.aebe_exportaciones_regiones",
    f"{DB_NAME}.tabla_temporal",
]

print("\n🔄 Limpiando tablas de datos...")
for t in TABLAS_DATOS:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    except:
        pass

print("   Vaciando volúmenes...")
for v in [RAW_PATH_ESPAC, RAW_PATH_SIPA, RAW_PATH_FAOSTAT, RAW_PATH_AEBE]:
    try:
        for f in dbutils.fs.ls(v):
            dbutils.fs.rm(f.path)
    except:
        pass

# ⚡ CRÍTICO: Limpiar control de descargas para forzar re-descarga
print("   Limpiando control_descargas_fuentes...")
try:
    spark.sql(f"DELETE FROM {DB_NAME}.control_descargas_fuentes WHERE framework = '{FRAMEWORK_NAME}'")
except:
    pass

print("   ✅ Datos limpios, descargas resetadas")

async def ejecutar_escenario_b(corrida_num: int):
    global EXECUTION_ID, ESCENARIO
    ESCENARIO = "B"
    EXECUTION_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_EB_R{corrida_num:02d}"
    print(f"\n🔢 CORRIDA {corrida_num}/{CORRIDAS} - ID: {EXECUTION_ID}")
    print("\n" + "═"*70)
    print("FASE 1: EXTRACCIÓN")
    print("═"*70)
    
    fuentes = [
        {"fuente_nombre": "ESPAC",
         "fuente_url": "https://www.ecuadorencifras.gob.ec/encuesta-de-superficie-y-produccion-agropecuaria-continua-espac/",
         "volumen_destino": RAW_PATH_ESPAC,
         "keywords_relevantes": ["banano", "platano", "tabulados", "series", "T13", "T26"]},
        {"fuente_nombre": "SIPA",
         "fuente_url": "https://sipa.agricultura.gob.ec/estadisticas-agropecuarias",
         "volumen_destino": RAW_PATH_SIPA,
         "keywords_relevantes": ["temperatura", "precipitacion", "uso suelo"]},
        {"fuente_nombre": "FAOSTAT",
         "fuente_url": "https://faostatservices.fao.org/api/v1/en/data/QCL",
         "volumen_destino": RAW_PATH_FAOSTAT,
         "keywords_relevantes": ["bananas", "plantains", "production", "Ecuador"]},
        {"fuente_nombre": "AEBE_BANANOTAS",
         "fuente_url": "https://www.aebe.com.ec/bananotas",
         "volumen_destino": RAW_PATH_AEBE,
         "keywords_relevantes": ["exportaciones", "regiones", "estadisticas"]},
    ]
    
    for fuente_cfg in fuentes:
        try:
            wf = ExtractionWorkflow(timeout=600, verbose=False)
            await wf.run(**fuente_cfg)
            print(f"  ✅ {fuente_cfg['fuente_nombre']}")
        except Exception as e:
            print(f"  ❌ {fuente_cfg['fuente_nombre']}: {e}")
    
    print("\n" + "═"*70)
    print("FASE 2: ETL CON VARIACIONES DE ESQUEMA")
    print("═"*70)
    
    VOLUMENES = [
        (RAW_PATH_ESPAC, "ESPAC"),
        (RAW_PATH_SIPA, "SIPA"),
        (RAW_PATH_FAOSTAT, "FAOSTAT"),
        (RAW_PATH_AEBE, "AEBE"),
    ]
    
    # Limpiar artefactos de perturbaciones anteriores
    # Garantiza que archivos generados por aplicar_variacion_esquema()
    # en corridas previas no se interpreten como fuentes nuevas.
    for _vp, _vl in VOLUMENES:
        try:
            for _nombre in os.listdir(_vp):
                if "_VAR" in _nombre.upper() or "BANANOTAS_REPORTE_" in _nombre:
                    os.remove(os.path.join(_vp, _nombre))
        except:
            pass
    
    archivos_pendientes = []
    for vol_path, vol_label in VOLUMENES:
        try:
            for nombre in sorted(os.listdir(vol_path)):
                if nombre.startswith('.'):
                    continue
                if not any(nombre.lower().endswith(ext) for ext in ['.xlsx', '.xls', '.csv', '.json', '.pdf']):
                    continue
                if "_TRANSFORMADO" in nombre.upper() or "_VAR" in nombre.upper():
                    continue
                ruta = f"{vol_path}/{nombre}"
                archivos_pendientes.append({"name": nombre, "path": ruta, "vol": vol_label})
        except Exception as e:
            print(f"  ⚠️ Listado {vol_label}: {e}")
    
    print(f"\n📊 {len(archivos_pendientes)} archivos a procesar")
    
    fuente_map = {"ESPAC": "ESPAC", "SIPA": "SIPA", "FAOSTAT": "FAOSTAT", "AEBE": "AEBE_BANANOTAS"}
    
    for idx, archivo in enumerate(archivos_pendientes, 1):
        print(f"\n[{idx}/{len(archivos_pendientes)}] {archivo['name']}")
        
        # ⚡ APLICAR VARIACIÓN DE ESQUEMA (esto es lo que hace único al Escenario B)
        path_procesado = aplicar_variacion_esquema(archivo["path"], archivo["name"], archivo["vol"])
        nombre_procesado = os.path.basename(path_procesado)
        
        # Recuperar métricas de extracción
        reintentos_ext, recuperacion_ext = 0, 2
        if spark.catalog.tableExists(LOG_EXTRACCION):
            try:
                fn = fuente_map.get(archivo["vol"], archivo["vol"])
                df_e = spark.table(LOG_EXTRACCION).filter(
                    (col("execution_id") == EXECUTION_ID) &
                    (col("fuente") == fn) &
                    (col("status").isin(["OK", "PARCIAL", "ERROR"]))
                ).orderBy(col("timestamp_fin").desc()).limit(1)
                if df_e.count() > 0:
                    r = df_e.first()
                    reintentos_ext = r["reintentos_realizados"] if "reintentos_realizados" in r.__fields__ else 0
                    recuperacion_ext = r["recuperacion"] if "recuperacion" in r.__fields__ else 2
            except:
                pass
        
        estado_etl = {
            "file_name": nombre_procesado,
            "local_path": path_procesado,
            "fuente": "",
            "inicio_ts": datetime.now().isoformat(),
            "df_columnas": [],
            "total_filas": 0,
            "ts_fin_lectura": "",
            "error_lectura": None,
            "tabla_destino": "",
            "cols_double": [],
            "cols_integer": [],
            "llm_response": "{}",
            "llamadas_api": 0,
            "error_mapeo": None,
            "registros_validos": 0,
            "registros_duplicados": 0,
            "ts_fin_transformacion": "",
            "error_transform": None,
            "temp_view_name": "",
            "tabla_completa": "",
            "ts_fin_carga": "",
            "error_carga": None,
            "tiempo_segundos": 0.0,
            "dpmo": 0.0,
            "calidad_pct": 0.0,
            "status": "PENDIENTE",
            "error_final": None,
            "reintentos_extraccion": reintentos_ext,
            "recuperacion_extraccion": recuperacion_ext,
        }
        
        try:
            estado = estado_etl.copy()
            estado = {**estado, **nodo_deteccion(estado)}
            estado = {**estado, **nodo_lectura(estado)}
            estado = {**estado, **nodo_mapeo_corregido(estado)}
            estado = {**estado, **nodo_transformacion_con_metricas(estado)}
            estado = {**estado, **nodo_carga(estado)}
            estado = {**estado, **nodo_metricas(estado)}
            print(f"  ✅ {estado['status']}")
        except Exception as e:
            print(f"  ❌ Error: {e}")

# Ejecutar 5 corridas en bucle
import time

for corrida in range(1, CORRIDAS + 1):
    print("\n" + "#"*70)
    print(f"# CORRIDA {corrida}/{CORRIDAS}")
    print("#"*70)
    
    try:
        await ejecutar_escenario_b(corrida)
        print(f"\n✅ Corrida {corrida} completada exitosamente")
    except Exception as e:
        print(f"\n❌ Corrida {corrida} falló: {e}")
        import traceback
        traceback.print_exc()
    
    if corrida < CORRIDAS:
        print(f"\n⏸️  Pausa 5s antes de corrida {corrida + 1}...")
        time.sleep(5)

print("\n" + "="*70)
print(f"✅ ESCENARIO B COMPLETADO: {CORRIDAS} corridas finalizadas")
print("="*70)
print("\nRevisa las métricas M5 (llamadas API) - deberían ser mayores en Escenario B")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EJECUTOR ESCENARIO C — COMPARABLE v3
# Protocolo experimental equiparable a AgenteCustom y LangGraph:
# - Sin preflight FAOSTAT (solo retry post-GET, como los demás)
# - Sin pausas artificiales entre corridas
# - Limpieza completa entre corridas (independencia experimental)
# ══════════════════════════════════════════════════════════════════════════

import asyncio
import nest_asyncio
from datetime import datetime
import time
import os
import requests
nest_asyncio.apply()

# ─── CONFIGURACIÓN ────────────────────────────────────────────────────────
DB_NAME = "bd_banano_ec"
FRAMEWORK_NAME = "LlamaIndex"
LOG_TABLE = f"{DB_NAME}.control_logs_etl"
LOG_EXTRACCION = f"{DB_NAME}.metricas_extraccion"

RAW_PATH_ESPAC = "/Volumes/databricksbanano/default/bronce/espac"
RAW_PATH_SIPA = "/Volumes/databricksbanano/default/bronce/sipa"
RAW_PATH_FAOSTAT = "/Volumes/databricksbanano/default/bronce/faostat"
RAW_PATH_AEBE = "/Volumes/databricksbanano/default/bronce/aebe"

VOLUMENES_PATHS = [RAW_PATH_ESPAC, RAW_PATH_SIPA, RAW_PATH_FAOSTAT, RAW_PATH_AEBE]

TABLAS_DATOS = [
    f"{DB_NAME}.espac_banano_platano_provincia",
    f"{DB_NAME}.faostat_produccion_banano_platano",
    f"{DB_NAME}.espac_uso_del_suelo",
    f"{DB_NAME}.sipa_temperatura_precipitacion",
    f"{DB_NAME}.aebe_exportaciones_regiones",
    f"{DB_NAME}.tabla_temporal",
]

# ─── MODO: TEST (1 corrida) o DEFINITIVO (10 corridas) ──────────────────
MODO_TEST = False  # DEFINITIVO: 10 corridas comparables
CORRIDAS = 10
SUFIJO = ""
ESCENARIO = "C"

print("="*70)
print(f"🅲️ ESCENARIO C — {'TEST' if MODO_TEST else 'DEFINITIVO'} ({CORRIDAS} corridas)")
print("="*70)
print(f"\n📍 Escenario: {ESCENARIO}")
print(f"   Corridas: {CORRIDAS}")
print(f"   Sufijo: '{SUFIJO}' {'(modo prueba)' if MODO_TEST else '(sin sufijo = definitivo)'}")
print(f"   Protocolo: COMPARABLE (sin preflight, sin pausas)")

# ─── FUNCIÓN DE LIMPIEZA (se ejecuta ANTES de cada corrida) ─────────────
def limpiar_estado_para_corrida(corrida_num):
    """Limpia TODOS los artefactos para garantizar independencia experimental.
    NO toca tablas de métricas (control_logs_etl, metricas_extraccion, etc.)"""
    print(f"\n  🧹 Limpiando estado para corrida {corrida_num}...")
    
    # 1. Vaciar volúmenes (eliminar archivos descargados)
    for v in VOLUMENES_PATHS:
        try:
            for f in dbutils.fs.ls(v):
                dbutils.fs.rm(f.path)
        except:
            pass
    print(f"     ✅ Volúmenes vaciados (4 paths)")
    
    # 2. Limpiar control_descargas_fuentes (forzar re-descarga completa)
    try:
        spark.sql(f"DELETE FROM {DB_NAME}.control_descargas_fuentes WHERE framework = '{FRAMEWORK_NAME}'")
    except:
        pass
    print(f"     ✅ control_descargas_fuentes limpio")
    
    # 3. Drop tablas de datos (ETL las recrea desde cero)
    for t in TABLAS_DATOS:
        try:
            spark.sql(f"DROP TABLE IF EXISTS {t}")
        except:
            pass
    print(f"     ✅ Tablas de datos eliminadas ({len(TABLAS_DATOS)})")
    
    # 4. Verificar que volúmenes están realmente vacíos
    archivos_residuales = 0
    for v in VOLUMENES_PATHS:
        try:
            archivos_residuales += len(dbutils.fs.ls(v))
        except:
            pass
    
    if archivos_residuales > 0:
        print(f"     ⚠️  ALERTA: {archivos_residuales} archivos residuales detectados")
    else:
        print(f"     ✅ Verificación: 0 archivos residuales")
    
    return archivos_residuales


async def ejecutar_escenario_c(corrida_num: int, sufijo=""):
    global EXECUTION_ID, ESCENARIO
    ESCENARIO = "C"
    EXECUTION_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_EC_R{corrida_num:02d}{sufijo}"
    print(f"\n🔢 CORRIDA {corrida_num}/{CORRIDAS} - ID: {EXECUTION_ID}")
    print("\n" + "═"*70)
    print("FASE 1: EXTRACCIÓN CON SIMULACIÓN DE FALLOS")
    print("═"*70)
    
    fuentes = [
        {"fuente_nombre": "ESPAC",
         "fuente_url": "https://www.ecuadorencifras.gob.ec/encuesta-de-superficie-y-produccion-agropecuaria-continua-espac/",
         "volumen_destino": RAW_PATH_ESPAC,
         "keywords_relevantes": ["banano", "platano", "tabulados", "series", "T13", "T26"]},
        {"fuente_nombre": "SIPA",
         "fuente_url": "https://sipa.agricultura.gob.ec/estadisticas-agropecuarias",
         "volumen_destino": RAW_PATH_SIPA,
         "keywords_relevantes": ["temperatura", "precipitacion", "uso suelo"]},
        {"fuente_nombre": "FAOSTAT",
         "fuente_url": "https://faostatservices.fao.org/api/v1/en/data/QCL",
         "volumen_destino": RAW_PATH_FAOSTAT,
         "keywords_relevantes": ["bananas", "plantains", "production", "Ecuador"]},
        {"fuente_nombre": "AEBE_BANANOTAS",
         "fuente_url": "https://www.aebe.com.ec/bananotas",
         "volumen_destino": RAW_PATH_AEBE,
         "keywords_relevantes": ["exportaciones", "regiones", "estadisticas"]},
    ]
    
    for fuente_cfg in fuentes:
        try:
            wf = ExtractionWorkflow(timeout=600, verbose=False)
            await wf.run(**fuente_cfg)
            print(f"  ✅ {fuente_cfg['fuente_nombre']}")
        except Exception as e:
            print(f"  ❌ {fuente_cfg['fuente_nombre']}: {e}")
    
    # Verificar archivos descargados post-extracción
    archivos_en_volumen = 0
    for v in VOLUMENES_PATHS:
        try:
            archivos_en_volumen += len([f for f in os.listdir(v) if not f.startswith('.')])
        except:
            pass
    print(f"\n  📦 Archivos en volúmenes post-extracción: {archivos_en_volumen}")
    
    print("\n" + "═"*70)
    print("FASE 2: ETL")
    print("═"*70)
    
    VOLUMENES = [
        (RAW_PATH_ESPAC, "ESPAC"),
        (RAW_PATH_SIPA, "SIPA"),
        (RAW_PATH_FAOSTAT, "FAOSTAT"),
        (RAW_PATH_AEBE, "AEBE"),
    ]
    
    archivos_pendientes = []
    for vol_path, vol_label in VOLUMENES:
        try:
            for nombre in sorted(os.listdir(vol_path)):
                if nombre.startswith('.'):
                    continue
                if not any(nombre.lower().endswith(ext) for ext in ['.xlsx', '.xls', '.csv', '.json', '.pdf']):
                    continue
                ruta = f"{vol_path}/{nombre}"
                archivos_pendientes.append({"name": nombre, "path": ruta, "vol": vol_label})
        except Exception as e:
            print(f"  ⚠️ Listado {vol_label}: {e}")
    
    print(f"\n📊 {len(archivos_pendientes)} archivos a procesar")
    
    fuente_map = {"ESPAC": "ESPAC", "SIPA": "SIPA", "FAOSTAT": "FAOSTAT", "AEBE": "AEBE_BANANOTAS"}
    
    for idx, archivo in enumerate(archivos_pendientes, 1):
        print(f"\n[{idx}/{len(archivos_pendientes)}] {archivo['name']}")
        
        reintentos_ext, recuperacion_ext = 0, 2
        if spark.catalog.tableExists(LOG_EXTRACCION):
            try:
                fn = fuente_map.get(archivo["vol"], archivo["vol"])
                df_e = spark.table(LOG_EXTRACCION).filter(
                    (col("execution_id") == EXECUTION_ID) &
                    (col("fuente") == fn) &
                    (col("status").isin(["OK", "PARCIAL", "ERROR"]))
                ).orderBy(col("timestamp_fin").desc()).limit(1)
                if df_e.count() > 0:
                    r = df_e.first()
                    reintentos_ext = r["reintentos_realizados"] if "reintentos_realizados" in r.__fields__ else 0
                    recuperacion_ext = r["recuperacion"] if "recuperacion" in r.__fields__ else 2
                    if reintentos_ext > 0:
                        print(f"  🔁 Recuperado tras {reintentos_ext} reintentos (extracción)")
            except:
                pass
        
        estado_etl = {
            "file_name": archivo["name"],
            "local_path": archivo["path"],
            "fuente": "",
            "inicio_ts": datetime.now().isoformat(),
            "df_columnas": [],
            "total_filas": 0,
            "ts_fin_lectura": "",
            "error_lectura": None,
            "tabla_destino": "",
            "cols_double": [],
            "cols_integer": [],
            "llm_response": "{}",
            "llamadas_api": 0,
            "error_mapeo": None,
            "registros_validos": 0,
            "registros_duplicados": 0,
            "ts_fin_transformacion": "",
            "error_transform": None,
            "temp_view_name": "",
            "tabla_completa": "",
            "ts_fin_carga": "",
            "error_carga": None,
            "tiempo_segundos": 0.0,
            "dpmo": 0.0,
            "calidad_pct": 0.0,
            "status": "PENDIENTE",
            "error_final": None,
            "reintentos_extraccion": reintentos_ext,
            "recuperacion_extraccion": recuperacion_ext,
        }
        
        try:
            estado = estado_etl.copy()
            estado = {**estado, **nodo_deteccion(estado)}
            estado = {**estado, **nodo_lectura(estado)}
            estado = {**estado, **nodo_mapeo_corregido(estado)}
            estado = {**estado, **nodo_transformacion_con_metricas(estado)}
            estado = {**estado, **nodo_carga(estado)}
            estado = {**estado, **nodo_metricas(estado)}
            print(f"  ✅ {estado['status']}")
        except Exception as e:
            print(f"  ❌ Error: {e}")
    
    return len(archivos_pendientes)


# ─── BUCLE PRINCIPAL: LIMPIEZA + EJECUCIÓN (sin preflight, sin pausas) ───
resultados = []

for corrida in range(1, CORRIDAS + 1):
    print("\n" + "#"*70)
    print(f"# CORRIDA {corrida}/{CORRIDAS} {'(TEST)' if MODO_TEST else '(DEFINITIVA)'}")
    print("#"*70)
    
    # Limpieza ANTES de cada corrida (independencia experimental)
    residuales = limpiar_estado_para_corrida(corrida)
    
    try:
        n_archivos = await ejecutar_escenario_c(corrida, sufijo=SUFIJO)
        resultados.append({"corrida": corrida, "archivos_procesados": n_archivos, "status": "OK"})
        print(f"\n✅ Corrida {corrida} completada: {n_archivos} archivos procesados")
    except Exception as e:
        resultados.append({"corrida": corrida, "archivos_procesados": 0, "status": f"ERROR: {e}"})
        print(f"\n❌ Corrida {corrida} falló: {e}")
        import traceback
        traceback.print_exc()

# ─── RESUMEN ──────────────────────────────────────────────────────────────
print("\n" + "="*70)
print(f"📊 RESUMEN {'TEST' if MODO_TEST else 'DEFINITIVO'}")
print("="*70)
for r in resultados:
    print(f"  Corrida {r['corrida']}: {r['archivos_procesados']} archivos - {r['status']}")

# Verificar en BD — EXCLUYENDO explícitamente cualquier corrida histórica marcada
like_pattern = f'%_EC_R%{SUFIJO}' if SUFIJO else '%_EC_R__'

df_verify = spark.sql(f"""
    SELECT execution_id, COUNT(*) as registros,
           SUM(CASE WHEN status IN ('PROCESADO','OK','WARNING') THEN 1 ELSE 0 END) as exitosos,
           SUM(reintentos_realizados) as total_reintentos,
           SUM(CASE WHEN reintentos_realizados > 0 THEN 1 ELSE 0 END) as archivos_con_reintento
    FROM {LOG_TABLE}
    WHERE framework = 'LlamaIndex' AND escenario = 'C'
      AND execution_id LIKE '{like_pattern}'
      AND execution_id NOT LIKE '%TEST%'
      AND COALESCE(observacion, '') = ''
    GROUP BY execution_id ORDER BY execution_id
""")
print("\n📋 Verificación en BD (solo corridas definitivas NUEVAS y sin marca histórica):")
df_verify.show(20, truncate=False)

total_registros = df_verify.agg({"registros": "sum"}).collect()[0][0] or 0
expected = CORRIDAS * 12
print(f"\n{'✅ PASSED' if total_registros == expected else '❌ FAILED'}: {total_registros}/{expected} registros esperados")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EXPORTAR RESULTADOS DEL EXPERIMENTO A EXCEL
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
tmp_file = f"/tmp/experimento_llamaindex_{timestamp}.xlsx"

print("="*70)
print("📥 EXPORTACIÓN DE RESULTADOS — Experimento LlamaIndex")
print("="*70)

with pd.ExcelWriter(tmp_file, engine='openpyxl') as writer:

    # 1. Logs generales (M1–M5 por archivo)
    if spark.catalog.tableExists(LOG_TABLE):
        df_logs = obtener_dataframe_deduplicado(LOG_TABLE).filter(
            F.col("framework") == "LlamaIndex"
        ).toPandas()
        df_logs.to_excel(writer, sheet_name='Logs_Generales', index=False)
        print(f"  ✅ Logs_Generales: {len(df_logs)} registros")

    # 2. Métricas de extracción
    if spark.catalog.tableExists(LOG_EXTRACCION):
        df_ext = obtener_dataframe_deduplicado(LOG_EXTRACCION).filter(
            F.col("framework") == "LlamaIndex"
        ).toPandas()
        df_ext.to_excel(writer, sheet_name='Metricas_Extraccion', index=False)
        print(f"  ✅ Metricas_Extraccion: {len(df_ext)} registros")

    # 3. Métricas de transformación
    if spark.catalog.tableExists(LOG_TRANSFORM):
        df_trans = obtener_dataframe_deduplicado(LOG_TRANSFORM).filter(
            F.col("framework") == "LlamaIndex"
        ).toPandas()
        df_trans.to_excel(writer, sheet_name='Metricas_Transformacion', index=False)
        print(f"  ✅ Metricas_Transformacion: {len(df_trans)} registros")

    # 4. Métricas de carga
    if spark.catalog.tableExists(LOG_CARGA):
        df_carga = obtener_dataframe_deduplicado(LOG_CARGA).filter(
            F.col("framework") == "LlamaIndex"
        ).toPandas()
        df_carga.to_excel(writer, sheet_name='Metricas_Carga', index=False)
        print(f"  ✅ Metricas_Carga: {len(df_carga)} registros")

    # 5. Resumen por escenario (promedios M1–M5)
    if spark.catalog.tableExists(LOG_TABLE):
        df_resumen = spark.sql(f"""
            SELECT
                escenario,
                COUNT(*) AS total_archivos,
                COUNT(DISTINCT execution_id) AS corridas,
                ROUND(AVG(m1_tiempo_segundos), 2) AS m1_tiempo_prom_s,
                ROUND(AVG(m2_intervencion_manual), 3) AS m2_intervencion_prom,
                ROUND(AVG(m3_recuperacion_errores), 3) AS m3_recuperacion_prom,
                ROUND(AVG(m4_calidad_pct), 2) AS m4_calidad_prom_pct,
                ROUND(AVG(m5_llamadas_api), 2) AS m5_llamadas_llm_prom,
                ROUND(AVG(m6_dpmo), 2) AS m6_dpmo_prom,
                SUM(CASE WHEN status = 'PROCESADO' THEN 1 ELSE 0 END) AS exitosos,
                SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) AS errores
            FROM {LOG_TABLE}
            WHERE framework = 'LlamaIndex' AND execution_id LIKE '%_E%_R%'
            GROUP BY escenario
            ORDER BY escenario
        """).toPandas()
        df_resumen.to_excel(writer, sheet_name='Resumen_Escenarios', index=False)
        print(f"  ✅ Resumen_Escenarios: {len(df_resumen)} filas")

    # 6. Detalle por fuente y escenario
    if spark.catalog.tableExists(LOG_TABLE):
        df_fuente = spark.sql(f"""
            SELECT
                escenario, fuente,
                COUNT(*) AS archivos,
                ROUND(AVG(m4_calidad_pct), 2) AS m4_calidad_prom,
                ROUND(AVG(m6_dpmo), 2) AS m6_dpmo_prom,
                ROUND(AVG(m1_tiempo_segundos), 2) AS tiempo_prom_s,
                SUM(CASE WHEN status = 'PROCESADO' THEN 1 ELSE 0 END) AS exitosos,
                SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) AS errores
            FROM {LOG_TABLE}
            WHERE framework = 'LlamaIndex' AND execution_id LIKE '%_E%_R%'
            GROUP BY escenario, fuente
            ORDER BY escenario, fuente
        """).toPandas()
        df_fuente.to_excel(writer, sheet_name='Detalle_Fuente_Escenario', index=False)
        print(f"  ✅ Detalle_Fuente_Escenario: {len(df_fuente)} filas")

# Copiar al workspace
ws_dest = f"/Workspace/Users/jfranco9@utmachala.edu.ec/experimento_llamaindex_{timestamp}.xlsx"
try:
    shutil.copy(tmp_file, ws_dest)
    print(f"\n{'='*50}")
    print(f"✅ Excel exportado exitosamente:")
    print(f"   {ws_dest}")
    print(f"{'='*50}")
except Exception as e:
    # Fallback: intentar con volumen
    try:
        vol_dest = f"/Volumes/databricksbanano/default/silver/experimento_llamaindex_{timestamp}.xlsx"
        dbutils.fs.cp(f"file:{tmp_file}", f"dbfs:{vol_dest}")
        print(f"\n✅ Excel en volumen: {vol_dest}")
    except Exception as e2:
        print(f"\n✅ Excel disponible en: {tmp_file}")
        print(f"   Error workspace: {e}")
        print(f"   Error volumen: {e2}")

In [ ]:
%sql
-- ══════════════════════════════════════════════════════════════════════════
-- VERIFICACIÓN DE MÉTRICAS: Comparación de Escenarios A, B, C
-- ══════════════════════════════════════════════════════════════════════════

-- Asegúrate de haber ejecutado las 3 celdas de escenarios antes de correr esto

SELECT
  escenario,
  fuente,
  COUNT(*) AS total_registros,
  SUM(CASE WHEN status = 'PROCESADO' THEN 1 ELSE 0 END) AS exitosos,
  SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) AS errores,
  
  -- M1: Tiempo de procesamiento
  ROUND(AVG(m1_tiempo_segundos), 2) AS avg_tiempo_seg,
  ROUND(MAX(m1_tiempo_segundos), 2) AS max_tiempo_seg,
  
  -- M3: Reintentos de extracción
  ROUND(AVG(reintentos_realizados), 2) AS avg_reintentos,
  MAX(reintentos_realizados) AS max_reintentos,
  
  -- M4: Calidad de datos
  ROUND(AVG(m4_calidad_pct), 2) AS avg_calidad_pct,
  
  -- M5: Llamadas al LLM
  SUM(m5_llamadas_api) AS total_llamadas_llm,
  ROUND(AVG(m5_llamadas_api), 2) AS avg_llamadas_llm,
  
  -- M6: DPMO
  ROUND(AVG(m6_dpmo), 2) AS avg_dpmo,
  
  -- M8: Recuperación
  ROUND(AVG(recuperacion), 2) AS avg_recuperacion
  
FROM bd_banano_ec.control_logs_etl
WHERE 
  escenario IN ('A', 'B', 'C')
  AND execution_id LIKE '20260624%'
GROUP BY escenario, fuente
ORDER BY fuente, escenario;

-- ══════════════════════════════════════════════════════════════════════════
-- MÉTRICAS ESPERADAS (promedio de 5 corridas por escenario):
-- ══════════════════════════════════════════════════════════════════════════
-- ESCENARIO A (normal):
--   • reintentos_realizados = 0 (sin fallos)
--   • llamadas_llm = mínimas (solo mapeo inicial)
--   • recuperacion = 2 (N/A - sin fallos)
--
-- ESCENARIO B (variaciones de esquema):
--   • reintentos_realizados = 0 (sin fallos de descarga)
--   • llamadas_llm > A (más consultas por schema ambiguo)
--   • tiempo_segundos > A (más procesamiento)
--
-- ESCENARIO C (fallos controlados):
--   • reintentos_realizados > 0 (al menos 1 reintento por fallo)
--   • recuperacion = 1 (recuperación exitosa)
--   • tiempo_segundos > A (por reintentos)
-- ══════════════════════════════════════════════════════════════════════════

-- NOTA: Cada escenario ejecuta 5 corridas (R01, R02, R03, R04, R05)
--       Esta consulta agrega las 5 corridas para obtener promedios

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ORQUESTADOR DEL EXPERIMENTO COMPLETO — LlamaIndex
# 3 escenarios (A, B, C) × 5 corridas = 15 ejecuciones totales
# ══════════════════════════════════════════════════════════════════════════

import asyncio, time
import nest_asyncio
nest_asyncio.apply()

print("="*70)
print("🧪 EXPERIMENTO LLAMAINDEX — 3 ESCENARIOS × 10 CORRIDAS")
print("="*70)

# ── RESET LOGS: Empezar experimento limpio (igual que LangGraph) ─────────
print("\n🗑️  Limpiando tablas de logs del experimento anterior...")
for _log_t in [LOG_TABLE, LOG_EXTRACCION, LOG_TRANSFORM, LOG_CARGA]:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {_log_t}")
        print(f"  ✅ Dropped: {_log_t}")
    except:
        pass

print("\n📋 Recreando tablas de métricas...")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    execution_id STRING, file_name STRING, framework STRING, fuente STRING,
    tabla_destino STRING, execution_timestamp TIMESTAMP, status STRING,
    m1_tiempo_segundos DOUBLE, m2_intervencion_manual INT,
    m3_recuperacion_errores INT, m4_calidad_pct DOUBLE,
    m5_llamadas_api INT, m6_dpmo DOUBLE, total_filas INT,
    registros_validos INT, registros_duplicados INT,
    llm_response STRING, error_message STRING,
    escenario STRING, reintentos_realizados INT, recuperacion INT,
    n_columnas INT
) USING DELTA
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_EXTRACCION} (
    execution_id STRING, fuente STRING, framework STRING,
    archivos_nuevos INT, archivos_omitidos INT, archivos_error INT,
    kb_descargados DOUBLE, tiempo_segundos DOUBLE,
    timestamp_inicio STRING, timestamp_fin STRING,
    llamadas_llm INT, analisis_agente STRING, status STRING,
    reintentos_realizados INT, recuperacion INT,
    recuperados INT, escenario STRING
) USING DELTA
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TRANSFORM} (
    execution_id STRING, file_name STRING, fuente STRING, framework STRING,
    tabla_destino STRING, filas_entrada INT, filas_salida INT,
    duplicados_elim INT, calidad_pct DOUBLE, dpmo_transformacion DOUBLE,
    tiempo_segundos DOUBLE, timestamp_inicio TIMESTAMP, timestamp_fin TIMESTAMP,
    analisis_agente STRING, llamadas_llm INT, status STRING
) USING DELTA
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_CARGA} (
    execution_id STRING, file_name STRING, fuente STRING, framework STRING,
    tabla_destino STRING, registros_insertados INT, tiempo_escritura_s DOUBLE,
    timestamp_inicio TIMESTAMP, timestamp_fin TIMESTAMP,
    analisis_agente STRING, llamadas_llm INT, status STRING
) USING DELTA
""")
print("✅ 4 tablas de métricas listas (vacías)")

ESCENARIOS             = ["A", "B", "C"]
CORRIDAS_POR_ESCENARIO = 10  # ✅ EXPERIMENTO COMPLETO: 5 corridas por escenario
TOTAL_CORRIDAS         = len(ESCENARIOS) * CORRIDAS_POR_ESCENARIO

# Tablas de DATOS que se borran entre corridas (NUNCA los logs)
TABLAS_DATOS_RESET = [
    f"{DB_NAME}.espac_banano_platano_provincia", f"{DB_NAME}.faostat_produccion_banano_platano",
    f"{DB_NAME}.espac_uso_del_suelo", f"{DB_NAME}.sipa_temperatura_precipitacion",
    f"{DB_NAME}.aebe_exportaciones_regiones", f"{DB_NAME}.espac_banano_provincia",
    f"{DB_NAME}.espac_platano_provincia", f"{DB_NAME}.faostat_produccion_bananos",
    f"{DB_NAME}.faostat_produccion_platanos", f"{DB_NAME}.espac_cultivos_permanentes",
    f"{DB_NAME}.espac_series_historicas", f"{DB_NAME}.sipa_uso_del_suelo", f"{DB_NAME}.tabla_temporal",
]
VOLUMENES_RESET = [RAW_PATH_ESPAC, RAW_PATH_SIPA, RAW_PATH_FAOSTAT, RAW_PATH_AEBE]

def reset_entre_corridas(esc, run):
    """Resetea tablas de datos y volúmenes. NUNCA toca los logs."""
    print(f"\n{'─'*50}\n🔄 RESET — Escenario {esc}, corrida {run}/{CORRIDAS_POR_ESCENARIO}")
    for t in TABLAS_DATOS_RESET:
        try: spark.sql(f"DROP TABLE IF EXISTS {t}")
        except: pass
    for v in VOLUMENES_RESET:
        try:
            for f in dbutils.fs.ls(v): dbutils.fs.rm(f.path)
        except: pass
    print("  ✅ Reset completo — Logs preservados intactos")

token_creation_time = datetime.now()
TOKEN_LIFETIME_MINUTES = 50

def renovar_token_drive_si_necesario():
    global token_creation_time, drive_service
    mins = (datetime.now() - token_creation_time).total_seconds() / 60
    if mins >= TOKEN_LIFETIME_MINUTES:
        print(f"\n🔄 Renovando token Drive ({mins:.1f} min)...")
        try:
            creds = service_account.Credentials.from_service_account_info(
                SERVICE_ACCOUNT_INFO, scopes=["https://www.googleapis.com/auth/drive"])
            drive_service = build("drive", "v3", credentials=creds)
            token_creation_time = datetime.now()
            print("  ✅ Token renovado")
        except Exception as e:
            print(f"  ⚠️ Error renovando token: {e}")

corrida_global = 0
inicio_total   = datetime.now()
fuente_map     = {"ESPAC":"ESPAC","SIPA":"SIPA","FAOSTAT":"FAOSTAT","AEBE":"AEBE_BANANOTAS"}

async def _ejecutar_corrida(escenario, numero_corrida):
    """Ejecuta una corrida completa (extracción + ETL)."""
    global ESCENARIO, EXECUTION_ID

    ESCENARIO    = escenario
    EXECUTION_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_E{escenario}_R{numero_corrida:02d}"
    print(f"   EXECUTION_ID: {EXECUTION_ID}")

    # ── EXTRACCIÓN ────────────────────────────────────────────────────────
    fuentes = [
        {"fuente_nombre":"ESPAC",
         "fuente_url":"https://www.ecuadorencifras.gob.ec/encuesta-de-superficie-y-produccion-agropecuaria-continua-espac/",
         "volumen_destino":RAW_PATH_ESPAC,
         "keywords_relevantes":["banano","platano","tabulados","series","T13","T26"]},
        {"fuente_nombre":"SIPA",
         "fuente_url":"https://sipa.agricultura.gob.ec/estadisticas-agropecuarias",
         "volumen_destino":RAW_PATH_SIPA,
         "keywords_relevantes":["temperatura","precipitacion","uso suelo","climatico"]},
        {"fuente_nombre":"FAOSTAT",
         "fuente_url":"https://faostatservices.fao.org/api/v1/en/data/QCL",
         "volumen_destino":RAW_PATH_FAOSTAT,
         "keywords_relevantes":["bananas","plantains","production","area harvested","Ecuador"]},
        {"fuente_nombre":"AEBE_BANANOTAS",
         "fuente_url":"https://www.aebe.com.ec/bananotas",
         "volumen_destino":RAW_PATH_AEBE,
         "keywords_relevantes":["exportaciones","regiones","estadisticas","cajas"]},
    ]

    for fuente_cfg in fuentes:
        try:
            wf = ExtractionWorkflow(timeout=600, verbose=False)
            await wf.run(**fuente_cfg)
        except Exception as e:
            print(f"  ❌ Extracción {fuente_cfg['fuente_nombre']}: {e}")

    # ── ETL por archivo ───────────────────────────────────────────────────
    VOLUMENES = [
        (RAW_PATH_ESPAC,"ESPAC"),(RAW_PATH_SIPA,"SIPA"),
        (RAW_PATH_FAOSTAT,"FAOSTAT"),(RAW_PATH_AEBE,"AEBE"),
    ]
    pendientes = []
    for vol_path, vol_label in VOLUMENES:
        try:
            for nombre in sorted(os.listdir(vol_path)):
                if nombre.startswith('.'): continue
                if not any(nombre.lower().endswith(ext) for ext in ['.xlsx','.xls','.csv','.json','.pdf']): continue
                if "_TRANSFORMADO" in nombre.upper() or "_VAR" in nombre.upper(): continue
                ruta = f"{vol_path}/{nombre}"
                pendientes.append({"name":nombre,"path":ruta,"vol":vol_label})
        except Exception as e:
            print(f"  ⚠️ Listado {vol_label}: {e}")

    for archivo in pendientes:
        # Aplicar variación de esquema (Escenario B)
        path_proc  = aplicar_variacion_esquema(archivo["path"], archivo["name"], archivo["vol"])
        nombre_proc = os.path.basename(path_proc)

        # Recuperar métricas extracción de este archivo (filtro por fuente)
        reintentos_ext, recuperacion_ext = 0, 2
        if spark.catalog.tableExists(LOG_EXTRACCION):
            try:
                fn = fuente_map.get(archivo["vol"], archivo["vol"])
                df_e = spark.table(LOG_EXTRACCION).filter(
                    (col("execution_id") == EXECUTION_ID) &
                    (col("fuente") == fn) &
                    (col("status").isin(["OK","PARCIAL","ERROR"]))
                ).orderBy(col("timestamp_fin").desc()).limit(1)
                if df_e.count() > 0:
                    r = df_e.first()
                    reintentos_ext   = r["reintentos_realizados"] if "reintentos_realizados" in r.__fields__ else 0
                    recuperacion_ext = r["recuperacion"] if "recuperacion" in r.__fields__ else 2
            except: pass

        estado_etl = {
            "file_name":nombre_proc, "local_path":path_proc,
            "fuente":"", "inicio_ts":datetime.now().isoformat(),
            "df_columnas":[], "total_filas":0, "ts_fin_lectura":"", "error_lectura":None,
            "tabla_destino":"", "cols_double":[], "cols_integer":[],
            "llm_response":"{}", "llamadas_api":0, "error_mapeo":None,
            "registros_validos":0, "registros_duplicados":0,
            "ts_fin_transformacion":"", "error_transform":None, "temp_view_name":"",
            "tabla_completa":"", "ts_fin_carga":"", "error_carga":None,
            "tiempo_segundos":0.0, "dpmo":0.0, "calidad_pct":0.0,
            "status":"PENDIENTE", "error_final":None,
            "reintentos_extraccion":reintentos_ext,
            "recuperacion_extraccion":recuperacion_ext,
            "escenario": escenario,
        }
        try:
            wf = ETLWorkflow(timeout=900, verbose=False)
            await wf.run(
                nombre_archivo=nombre_proc,
                ruta_archivo=path_proc,
                fuente=archivo["vol"],
                etl_state_override=estado_etl,
            )
        except Exception as e:
            # Si ETLWorkflow no acepta override, ejecutar nodos directamente
            try:
                estado_resultado = estado_etl.copy()
                estado_resultado = {**estado_resultado, **nodo_deteccion(estado_etl)}
                estado_resultado = {**estado_resultado, **nodo_lectura(estado_resultado)}
                estado_resultado = {**estado_resultado, **nodo_mapeo_corregido(estado_resultado)}
                estado_resultado = {**estado_resultado, **nodo_transformacion_con_metricas(estado_resultado)}
                estado_resultado = {**estado_resultado, **nodo_carga(estado_resultado)}
                estado_resultado = {**estado_resultado, **nodo_metricas(estado_resultado)}
            except Exception as e2:
                print(f"  ❌ ETL {archivo['name']}: {e2}")
                # Registrar fila de error para no dejar hueco silencioso
                try:
                    err_s = StructType([
                        StructField("execution_id",StringType(),True),StructField("file_name",StringType(),True),
                        StructField("framework",StringType(),True),StructField("fuente",StringType(),True),
                        StructField("tabla_destino",StringType(),True),StructField("execution_timestamp",TimestampType(),True),
                        StructField("status",StringType(),True),StructField("m1_tiempo_segundos",DoubleType(),True),
                        StructField("m2_intervencion_manual",IntegerType(),True),StructField("m3_recuperacion_errores",IntegerType(),True),
                        StructField("m4_calidad_pct",DoubleType(),True),StructField("m5_llamadas_api",IntegerType(),True),
                        StructField("m6_dpmo",DoubleType(),True),StructField("total_filas",IntegerType(),True),
                        StructField("registros_validos",IntegerType(),True),StructField("registros_duplicados",IntegerType(),True),
                        StructField("llm_response",StringType(),True),StructField("error_message",StringType(),True),
                        StructField("escenario",StringType(),True),StructField("reintentos_realizados",IntegerType(),True),
                        StructField("recuperacion",IntegerType(),True),
                    ])
                    spark.createDataFrame([(
                        EXECUTION_ID,archivo["name"],FRAMEWORK_NAME,archivo["vol"],"N/A",
                        datetime.now(),"ERROR",0.0,1,0,0.0,0,0.0,0,0,0,"{}",str(e2)[:200],
                        ESCENARIO,0,0,
                    )],err_s).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TABLE)
                except: pass

for escenario in ESCENARIOS:
    print(f"\n{'='*70}\n📋 ESCENARIO {escenario}\n{'='*70}")

    for numero_corrida in range(1, CORRIDAS_POR_ESCENARIO + 1):
        corrida_global += 1
        print(f"\n{'#'*70}")
        print(f"🏃 CORRIDA {corrida_global}/{TOTAL_CORRIDAS} — Escenario {escenario}, Run {numero_corrida}")
        print(f"{'#'*70}")

        renovar_token_drive_si_necesario()
        reset_entre_corridas(escenario, numero_corrida)

        asyncio.run(_ejecutar_corrida(escenario, numero_corrida))
        time.sleep(3)

# ── RESUMEN FINAL ─────────────────────────────────────────────────────────
print("\n" + "="*70)
print("📊 TABLA RESUMEN — PROMEDIOS M1–M5 POR ESCENARIO")
print("="*70)
spark.sql(f"""
SELECT escenario, COUNT(*) AS corridas,
    ROUND(AVG(m1_tiempo_segundos),2)     AS m1_tiempo_prom_s,
    ROUND(AVG(m2_intervencion_manual),3) AS m2_intervencion_prom,
    ROUND(AVG(m3_recuperacion_errores),3)AS m3_recuperacion_prom,
    ROUND(AVG(m4_calidad_pct),2)         AS m4_calidad_prom_pct,
    ROUND(AVG(m5_llamadas_api),2)        AS m5_llamadas_llm_prom,
    ROUND(AVG(m6_dpmo),1)               AS m6_dpmo_prom,
    ROUND(AVG(reintentos_realizados),2)  AS reintentos_prom
FROM {LOG_TABLE} WHERE execution_id LIKE '%_E%_R%'
GROUP BY escenario ORDER BY escenario
""").display()

print(f"\n✅ {TOTAL_CORRIDAS} corridas completadas — Logs en: {LOG_TABLE}")

In [ ]:
# ==============================================================================
# HELPER DE DEDUPLICACIÓN INTELIGENTE DE REGISTROS (Evita duplicar datos viejos y nuevos)
# ==============================================================================
def obtener_dataframe_deduplicado(tabla_or_df):
    """
    Garantiza que no se exporten registros repetidos (años/corridas viejas y nuevas).
    Deduplica por claves de negocio/ejecución antes de convertir a Pandas/Excel.
    """
    df = spark.table(tabla_or_df) if isinstance(tabla_or_df, str) else tabla_or_df
    t_name = (tabla_or_df if isinstance(tabla_or_df, str) else "").lower()
    cols = [c.lower() for c in df.columns]
    
    if "espac_banano_platano" in t_name:
        subset = [c for c in ["anio", "provincia", "variable"] if c in cols]
        return df.dropDuplicates(subset=subset) if len(subset) >= 2 else df.dropDuplicates()
    elif "espac_uso" in t_name:
        subset = [c for c in ["anio", "provincia", "uso_suelo"] if c in cols]
        return df.dropDuplicates(subset=subset) if len(subset) >= 2 else df.dropDuplicates()
    elif "sipa_temperatura" in t_name:
        subset = [c for c in ["anio", "mes", "provincia"] if c in cols]
        return df.dropDuplicates(subset=subset) if len(subset) >= 2 else df.dropDuplicates()
    elif "faostat" in t_name:
        subset = [c for c in ["anio", "elemento", "item"] if c in cols]
        return df.dropDuplicates(subset=subset) if len(subset) >= 2 else df.dropDuplicates()
    elif "aebe" in t_name:
        subset = [c for c in ["anio", "mes", "region"] if c in cols]
        return df.dropDuplicates(subset=subset) if len(subset) >= 2 else df.dropDuplicates()
    elif "control_logs" in t_name or "metricas" in t_name:
        if "execution_id" in cols and "archivo" in cols:
            return df.dropDuplicates(subset=["execution_id", "archivo"])
        elif "execution_id" in cols and "fuente" in cols:
            return df.dropDuplicates(subset=["execution_id", "fuente"])
        else:
            return df.dropDuplicates()
    else:
        return df.dropDuplicates()

# ══════════════════════════════════════════════════════════════════════════
# EXPORTAR RESULTADOS DEL EXPERIMENTO A EXCEL
# 6 hojas: Logs, Extracción, Transformación, Carga, Resumen, Detalle
# ══════════════════════════════════════════════════════════════════════════

import pandas as pd
import shutil
from datetime import datetime

print("="*70)
print("📥 EXPORTACIÓN DE RESULTADOS — Experimento LlamaIndex")
print("="*70)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archivo_tmp = f"/tmp/experimento_llamaindex_{timestamp}.xlsx"
archivo_ws = f"/Workspace/Users/jfranco9@utmachala.edu.ec/experimento_llamaindex_{timestamp}.xlsx"

# 1. Logs Generales
df_logs = obtener_dataframe_deduplicado(LOG_TABLE).filter(
    "execution_id LIKE '%_E%_R%'"
).toPandas()
print(f"  ✅ Logs_Generales: {len(df_logs)} registros")

# 2. Métricas Extracción
df_ext = obtener_dataframe_deduplicado(LOG_EXTRACCION).toPandas() if spark.catalog.tableExists(LOG_EXTRACCION) else pd.DataFrame()
print(f"  ✅ Metricas_Extraccion: {len(df_ext)} registros")

# 3. Métricas Transformación
df_trans = obtener_dataframe_deduplicado(LOG_TRANSFORM).toPandas() if spark.catalog.tableExists(LOG_TRANSFORM) else pd.DataFrame()
print(f"  ✅ Metricas_Transformacion: {len(df_trans)} registros")

# 4. Métricas Carga
df_carga = obtener_dataframe_deduplicado(LOG_CARGA).toPandas() if spark.catalog.tableExists(LOG_CARGA) else pd.DataFrame()
print(f"  ✅ Metricas_Carga: {len(df_carga)} registros")

# 5. Resumen por Escenario
df_resumen = spark.sql(f"""
SELECT 
    escenario,
    COUNT(*) AS n_registros,
    ROUND(AVG(m1_tiempo_segundos), 2) AS m1_tiempo_prom,
    ROUND(AVG(m2_intervencion_manual), 3) AS m2_intervencion_prom,
    ROUND(AVG(m3_recuperacion_errores), 3) AS m3_recuperacion_prom,
    ROUND(AVG(m4_calidad_pct), 2) AS m4_calidad_prom,
    ROUND(AVG(m5_llamadas_api), 2) AS m5_llamadas_prom,
    ROUND(AVG(m6_dpmo), 2) AS m6_dpmo_prom,
    SUM(CASE WHEN status = 'OK' THEN 1 ELSE 0 END) AS n_ok,
    SUM(CASE WHEN status = 'WARNING' THEN 1 ELSE 0 END) AS n_warning,
    SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) AS n_error
FROM {LOG_TABLE}
WHERE execution_id LIKE '%_E%_R%'
GROUP BY escenario
ORDER BY escenario
""").toPandas()
print(f"  ✅ Resumen_Escenarios: {len(df_resumen)} filas")

# 6. Detalle por Fuente y Escenario
df_detalle = spark.sql(f"""
SELECT 
    escenario, fuente,
    COUNT(*) AS n_archivos,
    ROUND(AVG(m1_tiempo_segundos), 2) AS m1_prom,
    ROUND(AVG(m4_calidad_pct), 2) AS m4_prom,
    ROUND(AVG(m6_dpmo), 2) AS m6_prom,
    ROUND(AVG(m5_llamadas_api), 2) AS m5_prom,
    SUM(CASE WHEN status = 'ERROR' THEN 1 ELSE 0 END) AS errores
FROM {LOG_TABLE}
WHERE execution_id LIKE '%_E%_R%'
GROUP BY escenario, fuente
ORDER BY escenario, fuente
""").toPandas()
print(f"  ✅ Detalle_Fuente_Escenario: {len(df_detalle)} filas")

# Escribir Excel
with pd.ExcelWriter(archivo_tmp, engine='openpyxl') as writer:
    df_logs.to_excel(writer, sheet_name='Logs_Generales', index=False)
    if len(df_ext) > 0:
        df_ext.to_excel(writer, sheet_name='Metricas_Extraccion', index=False)
    if len(df_trans) > 0:
        df_trans.to_excel(writer, sheet_name='Metricas_Transformacion', index=False)
    if len(df_carga) > 0:
        df_carga.to_excel(writer, sheet_name='Metricas_Carga', index=False)
    df_resumen.to_excel(writer, sheet_name='Resumen_Escenarios', index=False)
    df_detalle.to_excel(writer, sheet_name='Detalle_Fuente_Escenario', index=False)

# Copiar al Workspace
try:
    shutil.copy(archivo_tmp, archivo_ws)
    print(f"\n{'='*50}")
    print(f"✅ Excel exportado exitosamente:")
    print(f"   {archivo_ws}")
    print(f"{'='*50}")
except Exception as e:
    print(f"\n⚠️  Error copiando al Workspace: {e}")
    print(f"   Archivo disponible en: {archivo_tmp}")

In [ ]:
%sql
-- ═════════════════════════════════════════════════════════════════════════
-- VERIFICACIÓN: Comprobar que TODAS las columnas se llenen correctamente
-- Ejecuta esta celda DESPUÉS de correr un escenario (B o C)
-- ═════════════════════════════════════════════════════════════════════════

-- 1) Ver los registros más recientes con TODAS las columnas
SELECT 
  execution_id,
  file_name,
  framework,
  fuente,
  tabla_destino,
  execution_timestamp,
  status,
  m1_tiempo_segundos AS tiempo_s,
  m2_intervencion_manual AS intervencion,
  m3_recuperacion_errores AS recuperacion_errores,
  m4_calidad_pct AS calidad_pct,
  m5_llamadas_api AS llm_calls,
  m6_dpmo AS dpmo,
  total_filas,
  registros_validos,
  registros_duplicados,
  -- 🔧 LAS 3 COLUMNAS ANTES FALTANTES:
  escenario,
  reintentos_realizados,
  recuperacion,
  error_message
FROM bd_banano_ec.control_logs_etl
WHERE framework = 'LlamaIndex'
ORDER BY execution_timestamp DESC
LIMIT 10;

-- 2) Verificar que NO haya NULL en las columnas críticas
SELECT 
  COUNT(*) AS total_registros,
  SUM(CASE WHEN escenario IS NULL THEN 1 ELSE 0 END) AS escenario_null,
  SUM(CASE WHEN reintentos_realizados IS NULL THEN 1 ELSE 0 END) AS reintentos_null,
  SUM(CASE WHEN recuperacion IS NULL THEN 1 ELSE 0 END) AS recuperacion_null,
  -- Valores únicos de escenario (debería mostrar A, B, C)
  COLLECT_SET(escenario) AS escenarios_presentes
FROM bd_banano_ec.control_logs_etl
WHERE framework = 'LlamaIndex'
  AND execution_timestamp >= CURRENT_DATE()  -- Solo hoy
;

-- 3) Distribución por escenario (para confirmar que B y C se ejecutaron)
SELECT 
  escenario,
  COUNT(*) AS registros,
  AVG(m1_tiempo_segundos) AS tiempo_promedio_s,
  AVG(m4_calidad_pct) AS calidad_promedio_pct,
  AVG(m6_dpmo) AS dpmo_promedio,
  SUM(m5_llamadas_api) AS total_llm_calls
FROM bd_banano_ec.control_logs_etl
WHERE framework = 'LlamaIndex'
  AND escenario IS NOT NULL
GROUP BY escenario
ORDER BY escenario;

## 📊 Bloque 7 — Métricas y Consultas

Consultas de los resultados del pipeline. Identico al LangGraph — mismas tablas Delta.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("📥 MÉTRICAS DE EXTRACCIÓN — por fuente:")
spark.table(LOG_EXTRACCION).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN EXTRACCIÓN:")
spark.sql(f"""
SELECT
    fuente,
    archivos_nuevos,
    archivos_omitidos,
    archivos_error,
    ROUND(kb_descargados/1024,2)  AS mb_descargados,
    ROUND(tiempo_segundos,1)      AS tiempo_s
FROM {LOG_EXTRACCION}
ORDER BY fuente
""").display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("🔧 MÉTRICAS DE TRANSFORMACIÓN — por archivo:")
spark.table(LOG_TRANSFORM).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN TRANSFORMACIÓN por fuente:")
spark.sql(f"""
SELECT
    fuente,
    COUNT(*)                            AS archivos,
    SUM(filas_entrada)                  AS total_filas_entrada,
    SUM(filas_salida)                   AS total_filas_validas,
    SUM(duplicados_elim)                AS total_duplicados,
    ROUND(AVG(calidad_pct),2)           AS calidad_promedio_pct,
    ROUND(AVG(dpmo_transformacion),1)   AS dpmo_promedio,
    ROUND(SUM(tiempo_segundos),1)       AS tiempo_total_s
FROM {LOG_TRANSFORM}
GROUP BY fuente
ORDER BY fuente
""").display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("💾 MÉTRICAS DE CARGA — por archivo:")
spark.table(LOG_CARGA).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN CARGA por fuente:")
spark.sql(f"""
SELECT
    fuente,
    COUNT(*)                          AS archivos,
    SUM(registros_insertados)         AS total_insertados,
    ROUND(AVG(tiempo_escritura_s),2)  AS tiempo_escritura_promedio_s,
    SUM(CASE WHEN status='OK' THEN 1 ELSE 0 END) AS exitosos,
    SUM(CASE WHEN status!='OK' THEN 1 ELSE 0 END) AS con_error
FROM {LOG_CARGA}
GROUP BY fuente
ORDER BY fuente
""").display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("📊 MÉTRICAS M1–M5 — RESULTADOS COMPLETOS:")
spark.table(LOG_TABLE).orderBy("execution_timestamp", ascending=False).display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("\n📈 RESUMEN POR FUENTE:")
spark.sql(f"""
SELECT
    fuente,
    COUNT(*)                              AS archivos,
    ROUND(AVG(m1_tiempo_segundos),1)      AS M1_tiempo_prom_s,
    MAX(m2_intervencion_manual)           AS M2_intervencion,
    ROUND(AVG(m3_recuperacion_errores),2) AS M3_recup_errores,
    ROUND(AVG(m4_calidad_pct),2)          AS M4_calidad_pct,
    ROUND(AVG(m5_llamadas_api),2)         AS M5_llamadas_api,
    ROUND(AVG(m6_dpmo),1)                 AS M6_dpmo,
    SUM(CASE WHEN status='PROCESADO' THEN 1 ELSE 0 END) AS exitosos,
    SUM(CASE WHEN status='ERROR'     THEN 1 ELSE 0 END) AS errores
FROM {LOG_TABLE}
GROUP BY fuente
ORDER BY fuente
""").display()

In [ ]:
print(f"\n📋 RESUMEN EJECUTIVO {FRAMEWORK_NAME.upper()}:")
spark.sql(f"""
SELECT
    '{FRAMEWORK_NAME}'                       AS framework,
    COUNT(*)                                 AS archivos_procesados,
    ROUND(AVG(m1_tiempo_segundos),2)         AS avg_M1_tiempo_s,
    0                                        AS M2_intervencion_manual,
    ROUND(AVG(m3_recuperacion_errores),2)    AS avg_M3_recuperacion,
    ROUND(AVG(m4_calidad_pct),2)             AS avg_M4_calidad_pct,
    ROUND(AVG(m5_llamadas_api),2)            AS avg_M5_llamadas,
    ROUND(AVG(m6_dpmo),1)                    AS avg_M6_dpmo,
    SUM(CASE WHEN status='PROCESADO' THEN 1 ELSE 0 END) AS exitosos,
    ROUND(SUM(CASE WHEN status='PROCESADO' THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS tasa_exito_pct
FROM {LOG_TABLE}
""").display()

## 📤 Bloque 8 — Exportación a Google Drive (Agente 3)

Replica el Agente de Exportación de LangGraph:

| Paso | LangGraph (nodo) | LlamaIndex (@step) |
|------|------------------|--------------------|
| 1 | `nodo_deteccion_export` | `detectar_tabla` |
| 2 | `nodo_seleccion_export` (LLM) | `seleccionar_con_llm` |
| 3 | `nodo_conversion_export` | `convertir_a_excel` |
| 4 | `nodo_carga_export` | `subir_a_drive` |
| 5 | `nodo_validacion_export` | `validar_subida` |
| 6 | `nodo_metricas_export` | `registrar_metricas` |

**Diferencia clave:** exporta **.xlsx** (igual que LangGraph), NO csv.


In [ ]:
from typing import TypedDict, List, Optional
from datetime import datetime
import json

class ExportState(TypedDict):
    """Estado del agente de exportación a Google Drive."""
    
    # Identificación
    tabla_nombre: str           # Nombre completo de la tabla (ej: BD_BANANO_EC.espac_banano_provincia)
    archivo_csv: str            # Nombre del archivo CSV destino
    tipo: str                   # 'DATOS' o 'METRICAS'
    
    # Detección
    existe_tabla: bool          # Si la tabla existe en el catálogo
    num_registros: int          # Número de registros en la tabla
    ultima_modificacion: str    # Timestamp de última modificación
    ultima_exportacion: str     # Timestamp de última exportación
    requiere_exportacion: bool  # Si el LLM decide exportar
    
    # Selección (LLM)
    razonamiento_llm: str       # Por qué el LLM decidió exportar o no
    prioridad: int              # 1=ALTA, 2=MEDIA, 3=BAJA
    llamadas_llm: int           # Número de llamadas al LLM
    
    # Conversión
    csv_content: str            # Contenido CSV generado
    csv_size_bytes: int         # Tamaño del CSV en bytes
    hash_md5: str               # Hash MD5 del contenido
    ts_fin_conversion: str      # Timestamp fin conversión
    error_conversion: Optional[str]
    
    # Carga a Drive
    file_id_drive: str          # ID del archivo en Drive
    drive_status: str           # ACTUALIZADO, CREADO, ERROR
    intentos_carga: int         # Número de reintentos
    ts_fin_carga: str           # Timestamp fin carga
    error_carga: Optional[str]
    
    # Validación
    validacion_ok: bool         # Si la validación pasó
    registros_drive: int        # Registros leídos desde Drive (verificación)
    precision_pct: float        # % de precisión (registros_drive/num_registros)
    ts_fin_validacion: str      # Timestamp fin validación
    error_validacion: Optional[str]
    
    # Métricas
    inicio_ts: str              # Timestamp inicio proceso
    tiempo_total_s: float       # E1: Tiempo total
    bytes_transferidos: int     # E5: Bytes transferidos
    
    # Estado final
    status: str                 # EXPORTADO, ERROR, OMITIDO
    error_final: Optional[str]  # Error final si hubo

print("✅ ExportState definido.")

In [ ]:
def nodo_deteccion_export(estado: ExportState) -> ExportState:
    """
    Detecta si la tabla existe, cuántos registros tiene y cuándo fue modificada.
    """
    print(f"\n🔍 DETECCIÓN: {estado['tabla_nombre']}")
    
    try:
        # Verificar si la tabla existe
        if not spark.catalog.tableExists(estado["tabla_nombre"]):
            estado["existe_tabla"] = False
            estado["num_registros"] = 0
            estado["requiere_exportacion"] = False
            estado["status"] = "OMITIDO"
            estado["error_final"] = "Tabla no existe"
            print(f"  ⚠️  Tabla no existe: {estado['tabla_nombre']}")
            return estado
        
        estado["existe_tabla"] = True
        
        # Contar registros
        df = spark.table(estado["tabla_nombre"])
        count = df.count()
        estado["num_registros"] = count
        
        if count == 0:
            estado["requiere_exportacion"] = False
            estado["status"] = "OMITIDO"
            estado["error_final"] = "Tabla vacía (0 registros)"
            print(f"  ⚠️  Tabla vacía: 0 registros")
            return estado
        
        print(f"  📈 Tabla encontrada: {count:,} registros")
        
        # Inicializar campos requeridos si no existen
        if "archivo_csv" not in estado:
            tabla_sin_prefijo = estado["tabla_nombre"].split(".")[-1]
            estado["archivo_csv"] = f"{tabla_sin_prefijo}.xlsx"
        
        if "tipo" not in estado:
            # Determinar tipo basado en el nombre
            if "metricas_" in estado["tabla_nombre"] or "control_logs" in estado["tabla_nombre"]:
                estado["tipo"] = "METRICAS"
            else:
                estado["tipo"] = "DATOS"
        
        # Obtener timestamp de última modificación (si está disponible)
        try:
            table_details = spark.sql(f"DESCRIBE DETAIL {estado['tabla_nombre']}").collect()[0]
            estado["ultima_modificacion"] = str(table_details["lastModified"])
            print(f"  📅 Última modificación: {estado['ultima_modificacion']}")
        except:
            estado["ultima_modificacion"] = datetime.now().isoformat()
        
        # Verificar si ya fue exportada antes (si existe tabla de control)
        # TODO: implementar tabla de control de exportaciones
        estado["ultima_exportacion"] = ""  # Por ahora siempre exportar
        
        # Por defecto, marcar como que requiere exportación (el LLM decidirá)
        estado["requiere_exportacion"] = True
        
        print(f"  ✅ Detección completada")
        
    except Exception as e:
        estado["existe_tabla"] = False
        estado["num_registros"] = 0
        estado["requiere_exportacion"] = False
        estado["status"] = "ERROR"
        estado["error_final"] = f"Error en detección: {str(e)}"
        print(f"  ❌ Error: {e}")
    
    return estado

print("✅ nodo_deteccion_export() definido.")

In [ ]:
def nodo_seleccion_export(estado: ExportState) -> ExportState:
    """
    Usa el LLM para decidir si exportar la tabla y con qué prioridad.
    """
    print(f"\n🤖 SELECCIÓN LLM: {estado['tabla_nombre']}")
    
    try:
        # Preparar contexto para el LLM
        contexto = f"""
Tabla: {estado['tabla_nombre']}
Archivo CSV destino: {estado['archivo_csv']}
Tipo: {estado['tipo']}
Número de registros: {estado['num_registros']:,}
Última modificación: {estado['ultima_modificacion']}
Última exportación: {estado.get('ultima_exportacion', 'Nunca')}

Decide si esta tabla debe ser exportada a Google Drive en este momento.

Criterios de decisión:
1. Tablas de DATOS son de alta prioridad si tienen >100 registros
2. Tablas de MÉTRICAS son de alta prioridad si tienen datos nuevos
3. Tablas vacías o con muy pocos registros son de baja prioridad
4. Si fue exportada recientemente (<1 hora) y no cambió, es baja prioridad

Responde en formato JSON:
{{
  "debe_exportar": true/false,
  "prioridad": 1-3 (1=ALTA, 2=MEDIA, 3=BAJA),
  "razonamiento": "Explicación breve de la decisión"
}}
"""
        
        # Llamar al LLM (usando ChatDatabricks)
        response = llm.invoke([
            SystemMessage(content="Eres un asistente experto en gestión de datos. Respondes siempre en formato JSON válido."),
            HumanMessage(content=contexto)
        ])
        
        estado["llamadas_llm"] = estado.get("llamadas_llm", 0) + 1
        
        # Parsear respuesta
        respuesta = response.content.strip()
        
        # Limpiar markdown si existe
        if respuesta.startswith("```json"):
            respuesta = respuesta.replace("```json", "").replace("```", "").strip()
        
        decision = json.loads(respuesta)
        
        estado["requiere_exportacion"] = decision.get("debe_exportar", True)
        estado["procesar"] = decision.get("debe_exportar", True)  # Agregar para compatibilidad con workflow
        estado["prioridad"] = decision.get("prioridad", 2)
        estado["razonamiento_llm"] = decision.get("razonamiento", "Sin razonamiento")
        estado["llm_response"] = estado["razonamiento_llm"]  # Agregar alias
        
        print(f"  🎯 Decisión: {'EXPORTAR' if estado['requiere_exportacion'] else 'OMITIR'}")
        print(f"  📈 Prioridad: {estado['prioridad']} ({'ALTA' if estado['prioridad']==1 else 'MEDIA' if estado['prioridad']==2 else 'BAJA'})")
        print(f"  💬 Razonamiento: {estado['razonamiento_llm']}")
        
        if not estado["requiere_exportacion"]:
            estado["status"] = "OMITIDO"
            print(f"  ⚠️  Exportación omitida por decisión del LLM")
        
    except Exception as e:
        # Si el LLM falla, usar regla por defecto: exportar si tiene >0 registros
        print(f"  ⚠️  Error en LLM: {e}")
        print(f"  🔄 Aplicando regla por defecto...")
        
        estado["requiere_exportacion"] = estado["num_registros"] > 0
        estado["procesar"] = estado["num_registros"] > 0  # Agregar para compatibilidad con workflow
        estado["prioridad"] = 2  # MEDIA por defecto
        estado["razonamiento_llm"] = f"Fallback: Exportar porque tiene {estado['num_registros']} registros"
        estado["llm_response"] = estado["razonamiento_llm"]  # Agregar alias
        estado["llamadas_llm"] = estado.get("llamadas_llm", 0) + 1
        
        if not estado["requiere_exportacion"]:
            estado["status"] = "OMITIDO"
    
    return estado

print("✅ nodo_seleccion_export() definido.")

In [ ]:
import hashlib

def nodo_conversion_export(estado: ExportState) -> ExportState:
    """
    Convierte la tabla Spark a Excel y calcula hash MD5.
    """
    print(f"\n🔄 CONVERSIÓN: {estado['tabla_nombre']} → Excel")
    
    try:
        # Leer tabla
        df = obtener_dataframe_deduplicado(estado["tabla_nombre"])
        
        # Convertir a Pandas
        print(f"  🐌 Convirtiendo a Pandas...")
        pdf = df.toPandas()
        
        # Generar Excel en memoria
        print(f"  📝 Generando Excel...")
        excel_buffer = io.BytesIO()
        pdf.to_excel(excel_buffer, index=False, engine='openpyxl')
        excel_content = excel_buffer.getvalue()
        
        estado["csv_content"] = excel_content  # Mantenemos el nombre por compatibilidad
        estado["csv_size_bytes"] = len(excel_content)
        estado["excel_path"] = estado.get("archivo_csv", "temp.xlsx")  # Agregar excel_path
        
        # Calcular hash MD5
        hash_md5 = hashlib.md5(excel_content).hexdigest()
        estado["hash_md5"] = hash_md5
        
        estado["ts_fin_conversion"] = datetime.now().isoformat()
        estado["error_conversion"] = None
        
        print(f"  ✅ Excel generado: {estado['csv_size_bytes']:,} bytes")
        print(f"  🔑 Hash MD5: {hash_md5[:16]}...")
        
    except Exception as e:
        estado["error_conversion"] = str(e)
        estado["status"] = "ERROR"
        estado["error_final"] = f"Error en conversión: {str(e)}"
        print(f"  ❌ Error: {e}")
    
    return estado

print("✅ nodo_conversion_export() definido.")

In [ ]:
def nodo_carga_export(estado: ExportState) -> ExportState:
    """
    Sube el CSV a Google Drive con reintentos automáticos.
    """
    print(f"\n📤 CARGA A DRIVE: {estado['archivo_csv']}")
    
    max_intentos = 3
    estado["intentos_carga"] = 0
    
    for intento in range(1, max_intentos + 1):
        try:
            estado["intentos_carga"] = intento
            print(f"  🔄 Intento {intento}/{max_intentos}...")
            
            # Usar la función actualizar_archivo_drive que ya existe
            resultado = actualizar_archivo_drive(
                nombre_archivo=estado["archivo_csv"],
                contenido_bytes=estado["csv_content"],  # Usar contenido_bytes, no contenido_csv
                folder_id=FOLDER_OUTPUT_ID
            )
            
            estado["drive_status"] = resultado["status"]
            estado["file_id_drive"] = resultado.get("fileId", "")
            
            if resultado["status"] in ["ACTUALIZADO", "CREADO"]:
                print(f"  {resultado['mensaje']}")
                print(f"  🆔 File ID: {estado['file_id_drive']}")
                
                estado["error_carga"] = None
                estado["ts_fin_carga"] = datetime.now().isoformat()
                estado["bytes_transferidos"] = estado["csv_size_bytes"]
                break  # Éxito, salir del loop
            else:
                # Error, pero podemos reintentar
                print(f"  ⚠️  {resultado['mensaje']}")
                if intento < max_intentos:
                    print(f"  🔄 Reintentando en 2 segundos...")
                    import time
                    time.sleep(2)
                else:
                    # Último intento fallido
                    estado["error_carga"] = resultado.get("mensaje", "Error desconocido")
                    estado["status"] = "ERROR"
                    estado["error_final"] = f"Error en carga después de {max_intentos} intentos"
                    print(f"  ❌ Carga fallida después de {max_intentos} intentos")
        
        except Exception as e:
            print(f"  ❌ Error en intento {intento}: {e}")
            if intento < max_intentos:
                print(f"  🔄 Reintentando en 2 segundos...")
                import time
                time.sleep(2)
            else:
                estado["error_carga"] = str(e)
                estado["status"] = "ERROR"
                estado["error_final"] = f"Error en carga: {str(e)}"
    
    return estado

print("✅ nodo_carga_export() definido.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════

def nodo_validacion_export(estado: ExportState) -> ExportState:
    """
    Valida que el archivo en Drive tiene el número correcto de registros.
    """
    print(f"\n✅ VALIDACIÓN: {estado['archivo_csv']}")
    
    try:
        # Por simplicidad, asumimos que si se subió sin error, está OK
        # En producción, podrías descargar el archivo y verificar registros
        
        if estado["drive_status"] in ["ACTUALIZADO", "CREADO"]:
            # Para archivos Excel, simplemente asumir que si se subió OK, la cantidad es correcta
            # (no podemos contar líneas en un Excel binario fácilmente)
            estado["registros_drive"] = estado.get("num_registros", 0)  # ✅ Obtener de estado
            
            # Calcular precisión
            esperados = estado["num_registros"]
            obtenidos = estado["registros_drive"]
            
            if esperados > 0:
                estado["precision_pct"] = (obtenidos / esperados) * 100
            else:
                estado["precision_pct"] = 0.0
            
            # Validación pasa si precision >= 99%
            estado["validacion_ok"] = estado["precision_pct"] >= 99.0
            
            if estado["validacion_ok"]:
                print(f"  ✅ Validación exitosa: {obtenidos:,}/{esperados:,} registros ({estado['precision_pct']:.2f}%)")
                estado["status"] = "EXPORTADO"
            else:
                print(f"  ⚠️  Validación parcial: {obtenidos:,}/{esperados:,} registros ({estado['precision_pct']:.2f}%)")
                estado["status"] = "EXPORTADO"
                estado["error_validacion"] = f"Precisión baja: {estado['precision_pct']:.2f}%"
        else:
            estado["validacion_ok"] = False
            estado["registros_drive"] = 0
            estado["precision_pct"] = 0.0
            estado["status"] = "ERROR"
            print(f"  ❌ Validación fallida: carga no exitosa")
        
        estado["ts_fin_validacion"] = datetime.now().isoformat()
        
    except Exception as e:
        estado["error_validacion"] = str(e)
        estado["validacion_ok"] = False
        estado["status"] = "ERROR"
        print(f"  ❌ Error en validación: {e}")
    
    return estado

print("✅ nodo_validacion_export() definido.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════

def nodo_metricas_export(estado: ExportState) -> ExportState:
    """
    Calcula métricas finales de exportación (E1-E5).
    """
    print(f"\n📊 MÉTRICAS: {estado['archivo_csv']}")
    
    try:
        # E1: Tiempo total
        inicio = datetime.fromisoformat(estado["inicio_ts"])
        fin = datetime.now()
        estado["tiempo_total_s"] = (fin - inicio).total_seconds()
        
        # E2: Tablas exportadas (siempre 1 en este contexto)
        # E3: Precisión ya calculada en validación
        # E4: Reintentos ya registrado en estado["intentos_carga"]
        # E5: Bytes transferidos ya en estado["bytes_transferidos"]
        
        print(f"  E1 Tiempo total:     {estado['tiempo_total_s']:.2f}s")
        print(f"  E2 Tablas exportadas: 1")
        print(f"  E3 Precisión:        {estado.get('precision_pct', 0):.2f}%")
        print(f"  E4 Reintentos:        {estado.get('intentos_carga', 0)}")
        print(f"  E5 Bytes transferidos: {estado.get('bytes_transferidos', 0):,} bytes")
        
        # Guardar métricas en tabla Delta (opcional)
        # TODO: crear tabla de métricas de exportación
        
        print(f"  ✅ Métricas calculadas")
        
    except Exception as e:
        print(f"  ⚠️  Error calculando métricas: {e}")
    
    return estado


def nodo_error_export(estado: ExportState) -> ExportState:
    """
    Maneja errores del proceso de exportación.
    """
    print(f"\n❌ ERROR: {estado['archivo_csv']}")
    print(f"  Razón: {estado.get('error_final', 'Error desconocido')}")
    
    estado["status"] = "ERROR"
    
    # Calcular tiempo hasta el error
    if estado.get("inicio_ts"):
        inicio = datetime.fromisoformat(estado["inicio_ts"])
        fin = datetime.now()
        estado["tiempo_total_s"] = (fin - inicio).total_seconds()
    
    return estado

print("✅ nodo_metricas_export() y nodo_error_export() definidos.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# AGENTE 3: EXPORT WORKFLOW (LlamaIndex)
# Mismo patron bridge: cada @step construye ExportState, llama al nodo LangGraph,
# extrae campos y emite el Event de salida.
# ==============================================================================

class InicioExportEvent(Event):
    tabla_nombre:   str
    timestamp_inicio: str

class TablaDetectadaEvent(Event):
    tabla_nombre:   str
    existe:         bool
    n_registros:    int
    timestamp_inicio: str

class TablaSeleccionadaEvent(Event):
    tabla_nombre:   str
    procesar:       bool
    razon_llm:      str
    llamadas_llm:   int
    timestamp_inicio: str

class ExcelConvertidoEvent(Event):
    tabla_nombre:   str
    excel_path:     str
    n_registros:    int
    llamadas_llm:   int
    timestamp_inicio: str
    excel_content_b64: str  # Contenido Excel codificado en base64

class SubidaCompletaEvent(Event):
    tabla_nombre:   str
    file_id:        str
    web_view_link:  str
    n_registros:    int
    llamadas_llm:   int
    timestamp_inicio: str

class ExportErrorEvent(Event):
    tabla_nombre:   str
    causa:          str
    llamadas_llm:   int
    timestamp_inicio: str


class ExportWorkflow(Workflow):

    @step
    async def iniciar(self, ctx: Context, ev: StartEvent) -> InicioExportEvent:
        return InicioExportEvent(
            tabla_nombre     = ev.get("tabla_nombre"),
            timestamp_inicio = datetime.now().isoformat(),
        )

    @step
    async def detectar_tabla(self, ctx: Context, ev: InicioExportEvent) -> TablaDetectadaEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 1 - DETECTAR] {ev.tabla_nombre}")
        estado = {"tabla_nombre": ev.tabla_nombre, "error_export": None, "llamadas_llm": 0}
        estado = nodo_deteccion_export(estado)
        if estado.get("error_export"):
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_export"],
                llamadas_llm=0, timestamp_inicio=ev.timestamp_inicio)
        return TablaDetectadaEvent(
            tabla_nombre=ev.tabla_nombre, existe=estado.get("existe_tabla", False),  # Usar existe_tabla
            n_registros=estado.get("num_registros", 0), timestamp_inicio=ev.timestamp_inicio)  # Usar num_registros

    @step
    async def seleccionar_con_llm(self, ctx: Context, ev: TablaDetectadaEvent) -> TablaSeleccionadaEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 2 - LLM SELECCION] {ev.tabla_nombre}")
        if not ev.existe:
            return TablaSeleccionadaEvent(tabla_nombre=ev.tabla_nombre, procesar=False,
                razon_llm="Tabla no existe", llamadas_llm=0, timestamp_inicio=ev.timestamp_inicio)
        # Preparar estado con TODOS los campos necesarios
        # ✅ Usar DRIVE_FILE_MAP para obtener el nombre correcto del archivo
        nombre_archivo = DRIVE_FILE_MAP.get(ev.tabla_nombre)
        if not nombre_archivo:
            # Fallback: generar desde el nombre de la tabla
            tabla_sin_prefijo = ev.tabla_nombre.split(".")[-1]
            nombre_archivo = f"{tabla_sin_prefijo}.xlsx"
        
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "num_registros": ev.n_registros,
            "existe_tabla": ev.existe,
            "archivo_csv": nombre_archivo,  # ✅ Usar nombre del mapeo
            "tipo": "METRICAS" if "metricas_" in ev.tabla_nombre or "control_logs" in ev.tabla_nombre else "DATOS",
            "ultima_modificacion": datetime.now().isoformat(),
            "error_export": None,
            "llamadas_llm": 0
        }
        estado = nodo_seleccion_export(estado)
        if estado.get("error_export"):
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_export"],
                llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)
        return TablaSeleccionadaEvent(
            tabla_nombre=ev.tabla_nombre, procesar=estado.get("procesar", True),
            razon_llm=estado.get("llm_response","")[:200],
            llamadas_llm=estado.get("llamadas_llm", 1), timestamp_inicio=ev.timestamp_inicio)

    @step
    async def convertir_a_excel(self, ctx: Context, ev: TablaSeleccionadaEvent) -> ExcelConvertidoEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 3 - CONVERTIR] {ev.tabla_nombre}")
        if not ev.procesar:
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre,
                causa=f"Omitido por LLM: {ev.razon_llm}",
                llamadas_llm=ev.llamadas_llm, timestamp_inicio=ev.timestamp_inicio)
        
        # Obtener num_registros directamente de la tabla
        df = obtener_dataframe_deduplicado(ev.tabla_nombre)
        num_registros = df.count()
        
        # ✅ Usar DRIVE_FILE_MAP para obtener el nombre correcto del archivo
        nombre_archivo = DRIVE_FILE_MAP.get(ev.tabla_nombre)
        if not nombre_archivo:
            tabla_sin_prefijo = ev.tabla_nombre.split(".")[-1]
            nombre_archivo = f"{tabla_sin_prefijo}.xlsx"
        
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "procesar": True,
            "archivo_csv": nombre_archivo,  # ✅ Usar nombre del mapeo
            "num_registros": num_registros,  # ✅ Agregar num_registros
            "llamadas_llm": ev.llamadas_llm,
            "error_export": None
        }
        estado = nodo_conversion_export(estado)
        if estado.get("error_export"):
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_export"],
                llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)
        # Codificar contenido en base64 para pasar en el evento
        import base64
        excel_b64 = base64.b64encode(estado.get("csv_content", b"")).decode('utf-8')
        return ExcelConvertidoEvent(
            tabla_nombre=ev.tabla_nombre,
            excel_path=estado.get("excel_path",""),
            n_registros=estado.get("num_registros", 0),  # ✅ Propagado desde estado
            llamadas_llm=estado.get("llamadas_llm",0),
            timestamp_inicio=ev.timestamp_inicio,
            excel_content_b64=excel_b64)

    @step
    async def subir_a_drive(self, ctx: Context, ev: ExcelConvertidoEvent) -> SubidaCompletaEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 4 - SUBIR DRIVE] {ev.tabla_nombre}")
        # Decodificar contenido desde base64
        import base64
        excel_content = base64.b64decode(ev.excel_content_b64)
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "archivo_csv": ev.excel_path,
            "csv_content": excel_content,
            "csv_size_bytes": len(excel_content),
            "num_registros": ev.n_registros,
            "llamadas_llm": ev.llamadas_llm,
            "error_export": None
        }
        estado = nodo_carga_export(estado)
        if estado.get("error_carga"):  # ✅ Corregir: error_carga, no error_export
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_carga"],
                llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)
        estado = nodo_validacion_export(estado)
        return SubidaCompletaEvent(
            tabla_nombre=ev.tabla_nombre,
            file_id=estado.get("file_id_drive",""),  # ✅ Corregir nombre campo
            web_view_link=estado.get("drive_link",""),
            n_registros=ev.n_registros,  # ✅ Ya viene del evento
            llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)

    @step
    async def registrar_metricas(self, ctx: Context, ev: SubidaCompletaEvent) -> StopEvent:
        print(f"\n[EXPORT PASO 5 - METRICAS] {ev.tabla_nombre}")
        # ✅ Usar DRIVE_FILE_MAP para obtener el nombre correcto del archivo
        nombre_archivo = DRIVE_FILE_MAP.get(ev.tabla_nombre)
        if not nombre_archivo:
            tabla_sin_prefijo = ev.tabla_nombre.split(".")[-1]
            nombre_archivo = f"{tabla_sin_prefijo}.xlsx"
        
        # ✅ Calcular bytes transferidos desde el archivo Excel
        df = obtener_dataframe_deduplicado(ev.tabla_nombre)
        pdf = df.toPandas()
        excel_buffer = io.BytesIO()
        pdf.to_excel(excel_buffer, index=False, engine='openpyxl')
        bytes_transferidos = len(excel_buffer.getvalue())
        
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "archivo_csv": nombre_archivo,  # ✅ Usar nombre del mapeo
            "drive_file_id": ev.file_id,
            "drive_link": ev.web_view_link,
            "num_registros": ev.n_registros,  # ✅ Cambiar a num_registros
            "bytes_transferidos": bytes_transferidos,  # ✅ Agregar bytes transferidos
            "llamadas_llm": ev.llamadas_llm,
            "inicio_ts": ev.timestamp_inicio
        }
        estado = nodo_metricas_export(estado)
        return StopEvent(result={
            "status": "EXPORTADO", "tabla": ev.tabla_nombre,
            "file_id": ev.file_id, "link": ev.web_view_link, "registros": ev.n_registros,
        })

    @step
    async def manejar_error(self, ctx: Context, ev: ExportErrorEvent) -> StopEvent:
        print(f"\nERROR EXPORT {ev.tabla_nombre}: {ev.causa}")
        return StopEvent(result={"status":"ERROR","tabla":ev.tabla_nombre,"causa":ev.causa})

print("OK ExportWorkflow (AGENTE 3) definido.")
print("   Pasos: detectar -> seleccionar_llm -> convertir_excel -> subir_drive -> metricas -> END")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ORQUESTADOR AGENTE 3: EXPORTAR TABLAS A GOOGLE DRIVE
# ══════════════════════════════════════════════════════════════════════════

import asyncio

async def ejecutar_agente3_exportacion():
    """Exporta cada tabla Delta al Drive usando ExportWorkflow."""
    # Tablas de datos (Gold)
    tablas_gold = [
        f"{DB_NAME}.espac_banano_platano_provincia",
        f"{DB_NAME}.espac_uso_del_suelo",
        f"{DB_NAME}.sipa_temperatura_precipitacion",
        f"{DB_NAME}.faostat_produccion_banano_platano",
        f"{DB_NAME}.aebe_exportaciones_regiones",
    ]
    
    # Tablas de métricas
    tablas_metricas = [
        f"{DB_NAME}.metricas_extraccion",
        f"{DB_NAME}.metricas_transformacion",
        f"{DB_NAME}.metricas_carga",
        f"{DB_NAME}.control_logs_etl",
    ]
    
    # Combinar todas las tablas a exportar
    todas_las_tablas = tablas_gold + tablas_metricas

    resultados_export = []
    ts_global         = datetime.now()

    for tabla in todas_las_tablas:
        print(f"\n{'='*70}")
        print(f"AGENTE 3 EXPORT: {tabla}")
        print(f"{'='*70}")
        try:
            wf = ExportWorkflow(timeout=300, verbose=False)
            resultado = await wf.run(tabla_nombre=tabla)
            resultados_export.append(resultado)
            st = resultado.get('status','?') if resultado else 'ERROR'
            print(f"  => {st}")
        except Exception as e:
            print(f"  ERROR en {tabla}: {e}")
            resultados_export.append({"status":"ERROR","tabla":tabla,"causa":str(e)})

    tiempo_total = (datetime.now() - ts_global).total_seconds()
    ok  = sum(1 for r in resultados_export if r and r.get('status') == 'EXPORTADO')
    err = sum(1 for r in resultados_export if r and r.get('status') == 'ERROR')
    print(f"\n{'='*70}")
    print(f"EXPORTACION COMPLETADA en {tiempo_total:.1f}s")
    print(f"  Tablas exportadas: {ok}/{len(todas_las_tablas)} | Errores: {err}")
    print(f"{'='*70}")
    return resultados_export

resultados_export = await ejecutar_agente3_exportacion()



---

## 📝 Notas de Implementación

### Diferencias Arquitectónicas LangGraph vs LlamaIndex Workflows

| Aspecto | LangGraph | LlamaIndex Workflows |
|---------|-----------|----------------------|
| **Orquestación** | `StateGraph` con nodos y edges | `Workflow` con `@step` y eventos |
| **Estado** | `TypedDict` mutable compartido | Eventos inmutables tipados |
| **Ejecución** | `graph.invoke(state)` síncrono | `await workflow.run(...)` asíncrono |
| **Condicionales** | `add_conditional_edges` + router | Tipo de retorno del `@step` |
| **Lógica ETL** | Directa en cada nodo | Patrón bridge: `@step` → llama nodo LG |
| **Transparencia** | Fácil de visualizar con `display(graph)` | Requiere definir flujo manualmente |

### Logica compartida (sin duplicacion)
Todas las funciones de transformacion (`leer_archivo`, `castear_columnas`,
`transformar_espac_t13_t26_mejorado_v3`, etc.) son identicas en ambos frameworks.
LlamaIndex las llama a traves del patron bridge, garantizando resultados identicos.

### Metricas M1–M5 ()
Ambos frameworks registran en las mismas tablas Delta con el campo `framework`
diferenciando los registros, permitiendo comparativas directas en Bloque 10.
